# TE-VMamba reproduction on Kaggle: SSM-Aware Token-Efficient VMamba via Adaptive Patch Pruning and Merging for Person Re-ID (CVPR 2026)

This notebook runs the **official TE-VMamba code** (github.com/Kylian07/TE-VMamba) in a single Kaggle notebook,
with **nothing cloned from GitHub**:
* The authors' backbone files (`vmamba.py`, `LayerTokenReducer.py`, `AdaptivePairMergeGate.py`, `csms6s.py`,
  `csm_triton.py`) are **embedded verbatim** in the cells below. Only three small compatibility patches are
  applied, each listed where it happens.
* The authors' **CUDA selective-scan kernel** (`kernels/selective_scan`, about 85 KB of source) is embedded and
  **compiled on Kaggle's GPU** (5–10 min). If the build fails, a slower pure-PyTorch scan is used and the log says
  so loudly.
* Data loading, sampler, losses, optimiser, LR schedule, training loop and evaluation are ported 1:1 from
  `datasets/`, `loss/`, `solver/`, `processor/` and `utils/metrics.py`.

**Protocol (the repo's `v2v_tiny` configs):** VMamba-Tiny (dims 96, depths 2-2-8-2) with ImageNet initialisation,
224×224 input, PK sampler (16 identities × 4 images), cross-entropy plus triplet loss (margin 0.3), AdamW
(lr 3e-4, wd 0.05), cosine schedule with 5 warm-up epochs, AMP, 300 epochs (600 for Occluded-Duke). Token pruning
runs at blocks 5–7 and pairwise token merging at blocks 9–11.

**Datasets:** Market-1501, DukeMTMC-reID and Occluded-Duke from Kaggle, plus Occluded-REID from its authors'
official zip. The next cell lists them.

**How to run**
1. *Settings → Accelerator → GPU T4 x2* (one GPU is used) and *Internet → On* (needed for ImageNet weights, the
   Occluded-REID zip and the Occluded-Duke split lists).
2. *Add Input* the Kaggle datasets in the next cell.
3. Smoke test with `RUN_MODE = "quick"`, then `RUN_MODE = "full"` and **Save Version → Save & Run All**.
4. The full protocol is long (see the time estimate printed after the first epoch). The notebook saves a resumable
   checkpoint every few epochs and stops cleanly before Kaggle's 12 h limit. Attach this notebook's output as an
   input and run again to continue.

## Datasets

| Dataset | Source | Notes |
|---|---|---|
| Market-1501 | Kaggle [`pengcw1/market-1501`](https://www.kaggle.com/datasets/pengcw1/market-1501) | standard layout (used successfully in the PAD runs) |
| DukeMTMC-reID | Kaggle [`whurobin/dukemtmcreid`](https://www.kaggle.com/datasets/whurobin/dukemtmcreid) | standard layout (used successfully in the PAD runs) |
| Occluded-Duke | **built in the notebook** from `whurobin/dukemtmcreid` and the official Occluded-DukeMTMC name lists (Miao et al., ICCV 2019; `train/query/gallery.list`, downloaded from github.com/lightas/Occluded-DukeMTMC-Dataset) | 15,618 train, 2,210 query, 17,661 gallery |
| Occluded-REID | **downloaded** from its authors' release (Zhuo et al., ICME 2018; `Occluded_REID.zip`, 42 MB, sha256-checked) and converted | 200 IDs × (5 occluded + 5 full-body). Standard supervised split: a random half of the IDs (seed 0) for training, the rest for testing (occluded images = query, full-body = gallery) |

MSMT17 is not used: its images are not on Kaggle, and the repo's MSMT17 tiny config even points to Market. The
repo has no CUHK03 loader or config.

In [ ]:
# ---- first code cell: pin ONE GPU (no multi-GPU code path in the repo except DDP) ----
import os, time
T_NOTEBOOK_START = time.time()     # used to stop cleanly before Kaggle's session limit
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("TEV_GPU", "0")

import copy, csv, glob, hashlib, json, logging, math, random, re, shutil, subprocess, sys, time, urllib.request, warnings, zipfile
import os.path as osp
from collections import defaultdict
from datetime import timedelta

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import Sampler
import torchvision.transforms as T

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm
from timm.data.random_erasing import RandomErasing

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="timm")
ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}) | timm {timm.__version__} | device {DEVICE} "
      f"{torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''} | GPUs visible: {torch.cuda.device_count()}")

## Run configuration (edit this cell)

In [ ]:
RUN_MODE = os.environ.get("TEV_RUN_MODE", "quick")   # "quick" = smoke test (minutes) | "full" = paper protocol

# Datasets to train+evaluate, in this order (each is an independent run, as in the paper):
DATASETS_TO_RUN = os.environ.get("TEV_DATASETS", "market1501,occ_duke,occ_reid,dukemtmc").split(",")

MODEL_SIZE = "tiny"         # "tiny" (default; official ImageNet weights exist) | "small" | "base" (repo's v2v_* configs)
PRETRAIN = os.environ.get("TEV_PRETRAIN", "imagenet")   # "imagenet": official VMamba ImageNet-1K weights when they match MODEL_SIZE | "none": as shipped (empty path)
EPOCH_SCALE = 1.0           # 1.0 = the repo's schedule (300 / 600 epochs). e.g. 0.5 halves every schedule (state it when reporting!)
EVAL_EVERY = 50             # extra mid-training evaluations every N epochs (the repo only evaluates at the end); 0 = off
CKPT_EVERY = 5              # save a resumable training state every N epochs
MAX_SESSION_HOURS = float(os.environ.get("TEV_MAX_HOURS", "11.3"))   # stop cleanly before Kaggle's 12 h limit
RESUME = True               # continue from the latest state found in OUT_DIR or in an attached previous output

# Paths (Kaggle defaults; env overrides for local testing)
INPUT_ROOT = os.environ.get("TEV_INPUT_ROOT", "/kaggle/input")
WORK_DIR = os.environ.get("TEV_WORK_DIR", "/kaggle/working")
_tmp = "/kaggle/temp" if osp.isdir("/kaggle/temp") else "/tmp"
DATA_ROOT = os.environ.get("TEV_DATA_ROOT", osp.join(_tmp, "tev_data"))
SRC_DIR = os.environ.get("TEV_SRC_DIR", osp.join(_tmp, "tev_src"))         # embedded source files are written here
OUT_DIR = osp.join(WORK_DIR, "TEVMamba_outputs")
CACHE_DIR = os.environ.get("TEV_CACHE", osp.join(_tmp, "tev_cache"))
for d in (DATA_ROOT, SRC_DIR, OUT_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)
DATA_OVERRIDES = {}         # e.g. {"market1501": "/kaggle/input/.../Market-1501-v15.09.15"}
NUM_WORKERS = min(4, os.cpu_count() or 2)

QUICK = dict(epochs=2, max_iters=10, eval_max_query=200, eval_max_gallery=2000)
TEST_IMG_SIZE = os.environ.get("TEV_TEST_SIZE")          # local CPU tests only: e.g. "64" to shrink inputs

if RUN_MODE != "full":
    print("!" * 100 + "\n!! RUN_MODE = \"quick\": SMOKE TEST ONLY. Numbers are NOT results. "
          "Set RUN_MODE = \"full\" for the real experiment.\n" + "!" * 100)
print(f"RUN_MODE={RUN_MODE} | MODEL_SIZE={MODEL_SIZE} | PRETRAIN={PRETRAIN} | EPOCH_SCALE={EPOCH_SCALE} | "
      f"datasets={DATASETS_TO_RUN}\nDATA_ROOT={DATA_ROOT} | OUT_DIR={OUT_DIR}")

## Configuration: port of `config/defaults.py` + `configs/<dataset>/v2v_<size>.yml`
Values below are copied from the repo's YAML files. Comments mark the keys that the code **never reads**: they are
kept for completeness but have no effect, exactly as in the original.

In [ ]:
class CN(dict):
    """Tiny attribute-dict so ported code can keep `cfg.SOLVER.BASE_LR` syntax."""
    def __getattr__(self, k):
        try:
            return self[k]
        except KeyError:
            raise AttributeError(k)

    def __setattr__(self, k, v):
        self[k] = v

    @staticmethod
    def wrap(d):
        return CN({k: CN.wrap(v) if isinstance(v, dict) else v for k, v in d.items()})


def _merge(a, b):
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = _merge(out[k], v) if (k in out and isinstance(out[k], dict) and isinstance(v, dict)) else copy.deepcopy(v)
    return out


BASE = {   # == configs/Market/v2v_base.yml merged over config/defaults.py
    "MODEL": {
        "NAME": "vssm", "PRETRAIN_CHOICE": "imagenet", "PRETRAIN_PATH": "", "DROP_PATH_RATE": 0.5,
        "NECK": "bnneck", "COS_LAYER": False, "ID_LOSS_TYPE": "softmax", "METRIC_LOSS_TYPE": "triplet",
        "IF_LABELSMOOTH": "off", "IF_WITH_CENTER": "no", "NO_MARGIN": False,
        "ID_LOSS_WEIGHT": 1.0, "TRIPLET_LOSS_WEIGHT": 1.0,
        "VSSM": {"IN_CHANS": 3, "PATCH_SIZE": 4, "EMBED_DIM": 128, "DEPTHS": [2, 2, 20, 2], "SSM_RATIO": 1.0,
                 "SSM_D_STATE": 1, "SSM_DT_RANK": "auto", "SSM_CONV": 3, "SSM_CONV_BIAS": False,
                 "SSM_FORWARDTYPE": "v05_noz", "MLP_RATIO": 4.0, "DOWNSAMPLE": "v3", "PATCHEMBED": "v2",
                 "NORM_LAYER": "ln2d"},
        "TOKEN_ADAPT": {
            "ENABLE": True, "TOKEN_PRUNE": True, "TOKEN_MERGE": True,
            "PRUNE_LAYERS": [5, 6, 7], "MERGE_LAYERS": [12, 13, 14, 15, 16], "MERGE_START": 8,
            "PRUNE_THRESHOLD": 0.7,     # NOT READ: LayerTokenPrune is built with alpha_init=0.7 hard-coded
            "PRUNE_SIM": "cos_delta",   # NOT READ
            "PRUNE_IMP": "l2",          # NOT READ
            "MERGE_TAU_GATE": 0.5,      # NOT READ (tau_init=0.5 hard-coded, same value)
            "MERGE_GATE_HIDDEN": 64,    # NOT READ (64 hard-coded, same value)
            "FINAL_THRESHOLD": 0.1,     # NOT READ by any module
        },
    },
    "INPUT": {"SIZE_TRAIN": [224, 224], "SIZE_TEST": [224, 224], "PROB": 0.5, "RE_PROB": 0.3, "PADDING": 10,
              "PIXEL_MEAN": [0.485, 0.456, 0.406], "PIXEL_STD": [0.229, 0.224, 0.225]},
    "AUG": {"COLOR_JITTER": 0.0, "AUTO_AUGMENT": "none"},
    "DATASETS": {"NAMES": "market1501", "ROOT_DIR": DATA_ROOT},
    "DATALOADER": {"SAMPLER": "softmax_triplet", "NUM_INSTANCE": 4, "NUM_WORKERS": NUM_WORKERS},
    "SOLVER": {"OPTIMIZER_NAME": "AdamW", "MAX_EPOCHS": 300, "BASE_LR": 3e-4, "IMS_PER_BATCH": 64,
               "WARMUP_EPOCHS": 5, "WARMUP_METHOD": "linear", "LARGE_FC_LR": False, "BIAS_LR_FACTOR": 1,
               "MOMENTUM": 0.9, "WEIGHT_DECAY": 0.05, "WEIGHT_DECAY_BIAS": 0.0, "SEED": 1234, "MARGIN": 0.3,
               "CENTER_LR": 0.5, "CENTER_LOSS_WEIGHT": 0.0005, "CHECKPOINT_PERIOD": 50, "LOG_PERIOD": 100,
               "EVAL_PERIOD": 300},
    "TEST": {"IMS_PER_BATCH": 256, "RE_RANKING": False, "NECK_FEAT": "before", "FEAT_NORM": "yes"},
}

SIZE_OVERRIDES = {   # differences of configs/*/v2v_{tiny,small}.yml from v2v_base.yml
    "base": {},
    "small": {"MODEL": {"DROP_PATH_RATE": 0.3, "VSSM": {"EMBED_DIM": 96}}},
    "tiny": {"MODEL": {"DROP_PATH_RATE": 0.2, "VSSM": {"EMBED_DIM": 96, "DEPTHS": [2, 2, 8, 2]},
                       "TOKEN_ADAPT": {"MERGE_LAYERS": [9, 10, 11]}}},
}

DATASET_OVERRIDES = {   # differences of configs/<dataset>/v2v_*.yml from the Market file
    "market1501": {},
    "dukemtmc": {"DATASETS": {"NAMES": "dukemtmc"}, "MODEL": {"TOKEN_ADAPT": {"PRUNE_THRESHOLD": 0.25}}},
    # NOTE: the shipped configs/OCC_Duke/v2v_tiny.yml keeps NAMES: "market1501" (a copy-paste slip); the base/small
    # Occluded-Duke configs use "occ_duke", which is clearly the intent, so it is used here.
    "occ_duke": {"DATASETS": {"NAMES": "occ_duke"}, "MODEL": {"TOKEN_ADAPT": {"PRUNE_THRESHOLD": 0.25}},
                 "SOLVER": {"MAX_EPOCHS": 600, "EVAL_PERIOD": 600}},
    "occ_reid": {"DATASETS": {"NAMES": "occ_reid"}, "MODEL": {"TOKEN_ADAPT": {"PRUNE_THRESHOLD": 0.25}}},
}

# Official VMamba ImageNet-1K checkpoints (MzeroMiko/VMamba releases). Only VMamba-T [s1l8] has exactly the same
# architecture as the repo's v2v configs; the released S/B checkpoints are the s2l15 design (depths 2-2-15-2,
# ssm_ratio 2.0), which does not match v2v_small/v2v_base, so those sizes fall back to random init.
IMAGENET_CKPTS = {
    "tiny": ("vssm1_tiny_0230s_ckpt_epoch_264.pth",
             "https://github.com/MzeroMiko/VMamba/releases/download/%23v2cls/vssm1_tiny_0230s_ckpt_epoch_264.pth"),
}


def build_cfg(dataset):
    raw = _merge(_merge(BASE, SIZE_OVERRIDES[MODEL_SIZE]), DATASET_OVERRIDES[dataset])
    cfg = CN.wrap(raw)
    cfg.SOLVER.MAX_EPOCHS = max(1, int(round(cfg.SOLVER.MAX_EPOCHS * EPOCH_SCALE)))
    if RUN_MODE == "quick":
        cfg.SOLVER.MAX_EPOCHS = QUICK["epochs"]
        cfg.SOLVER.WARMUP_EPOCHS = 1
        cfg.SOLVER.LOG_PERIOD = 5
    if TEST_IMG_SIZE:
        s = int(TEST_IMG_SIZE)
        cfg.INPUT.SIZE_TRAIN = cfg.INPUT.SIZE_TEST = [s, s]
        cfg.SOLVER.IMS_PER_BATCH = 16
    return cfg


print(json.dumps({k: build_cfg("occ_duke")[k] for k in ("MODEL", "SOLVER")}, indent=1)[:1500])

## Selective-scan CUDA kernel (embedded from `kernels/selective_scan`, compiled here)
VMamba's SS2D layer scans each feature map along 4 directions with Mamba's *selective scan*. The authors ship a
CUDA implementation (the `oflex` variant from VMamba, built on Mamba's kernels). The files below are copied
**verbatim** from the repo, written to disk, and compiled for the GPU in this session (Kaggle T4 = sm_75).

The compiled extension is cached in the notebook output. When that output is attached to a later session, the
build is skipped.

In [ ]:
KERNEL_DIR = osp.join(SRC_DIR, "selective_scan")
os.makedirs(osp.join(KERNEL_DIR, "csrc", "selective_scan", "cusoflex"), exist_ok=True)
print("kernel sources ->", KERNEL_DIR)

In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/selective_scan.h
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#pragma once

////////////////////////////////////////////////////////////////////////////////////////////////////

struct SSMScanParamsBase {
    using index_t = uint32_t;

    int batch, seqlen, n_chunks;
    index_t a_batch_stride;
    index_t b_batch_stride;
    index_t out_batch_stride;

    // Common data pointers.
    void *__restrict__ a_ptr;
    void *__restrict__ b_ptr;
    void *__restrict__ out_ptr;
    void *__restrict__ x_ptr;
};

////////////////////////////////////////////////////////////////////////////////////////////////////

struct SSMParamsBase {
    using index_t = uint32_t;

    int batch, dim, seqlen, dstate, n_groups, n_chunks;
    int dim_ngroups_ratio;

    bool delta_softplus;

    index_t A_d_stride;
    index_t A_dstate_stride;
    index_t B_batch_stride;
    index_t B_d_stride;
    index_t B_dstate_stride;
    index_t B_group_stride;
    index_t C_batch_stride;
    index_t C_d_stride;
    index_t C_dstate_stride;
    index_t C_group_stride;
    index_t u_batch_stride;
    index_t u_d_stride;
    index_t delta_batch_stride;
    index_t delta_d_stride;
    index_t out_batch_stride;
    index_t out_d_stride;

    // Common data pointers.
    void *__restrict__ A_ptr;
    void *__restrict__ B_ptr;
    void *__restrict__ C_ptr;
    void *__restrict__ D_ptr;
    void *__restrict__ u_ptr;
    void *__restrict__ delta_ptr;
    void *__restrict__ delta_bias_ptr;
    void *__restrict__ out_ptr;
    void *__restrict__ x_ptr;
};

struct SSMParamsBwd: public SSMParamsBase {
    index_t dout_batch_stride;
    index_t dout_d_stride;
    index_t dA_d_stride;
    index_t dA_dstate_stride;
    index_t dB_batch_stride;
    index_t dB_group_stride;
    index_t dB_d_stride;
    index_t dB_dstate_stride;
    index_t dC_batch_stride;
    index_t dC_group_stride;
    index_t dC_d_stride;
    index_t dC_dstate_stride;
    index_t du_batch_stride;
    index_t du_d_stride;
    index_t ddelta_batch_stride;
    index_t ddelta_d_stride;

    // Common data pointers.
    void *__restrict__ dout_ptr;
    void *__restrict__ dA_ptr;
    void *__restrict__ dB_ptr;
    void *__restrict__ dC_ptr;
    void *__restrict__ dD_ptr;
    void *__restrict__ du_ptr;
    void *__restrict__ ddelta_ptr;
    void *__restrict__ ddelta_bias_ptr;
};


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/selective_scan_common.h
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#pragma once

#include <cuda_bf16.h>
#include <cuda_fp16.h>
#include <c10/util/complex.h>  // For scalar_value_type

#define MAX_DSTATE 256

inline __device__ float2 operator+(const float2 & a, const float2 & b){
    return {a.x + b.x, a.y + b.y};
}

inline __device__ float3 operator+(const float3 &a, const float3 &b) {
  return {a.x + b.x, a.y + b.y, a.z + b.z};
}

inline __device__ float4 operator+(const float4 & a, const float4 & b){
    return {a.x + b.x, a.y + b.y, a.z + b.z, a.w + b.w};
}

////////////////////////////////////////////////////////////////////////////////////////////////////

template<int BYTES> struct BytesToType {};

template<> struct BytesToType<16> {
    using Type = uint4;
    static_assert(sizeof(Type) == 16);
};

template<> struct BytesToType<8> {
    using Type = uint64_t;
    static_assert(sizeof(Type) == 8);
};

template<> struct BytesToType<4> {
    using Type = uint32_t;
    static_assert(sizeof(Type) == 4);
};

template<> struct BytesToType<2> {
    using Type = uint16_t;
    static_assert(sizeof(Type) == 2);
};

template<> struct BytesToType<1> {
    using Type = uint8_t;
    static_assert(sizeof(Type) == 1);
};

////////////////////////////////////////////////////////////////////////////////////////////////////

template<typename scalar_t, int N>
struct Converter{
    static inline __device__ void to_float(const scalar_t (&src)[N], float (&dst)[N]) {
        #pragma unroll
        for (int i = 0; i < N; ++i) { dst[i] = src[i]; }
    }
};

template<int N>
struct Converter<at::Half, N>{
    static inline __device__ void to_float(const at::Half (&src)[N], float (&dst)[N]) {
        static_assert(N % 2 == 0);
        auto &src2 = reinterpret_cast<const half2 (&)[N / 2]>(src);
        auto &dst2 = reinterpret_cast<float2 (&)[N / 2]>(dst);
        #pragma unroll
        for (int i = 0; i < N / 2; ++i) { dst2[i] = __half22float2(src2[i]); }
    }
};

#if __CUDA_ARCH__ >= 800
template<int N>
struct Converter<at::BFloat16, N>{
    static inline __device__ void to_float(const at::BFloat16 (&src)[N], float (&dst)[N]) {
        static_assert(N % 2 == 0);
        auto &src2 = reinterpret_cast<const nv_bfloat162 (&)[N / 2]>(src);
        auto &dst2 = reinterpret_cast<float2 (&)[N / 2]>(dst);
        #pragma unroll
        for (int i = 0; i < N / 2; ++i) { dst2[i] = __bfloat1622float2(src2[i]); }
    }
};
#endif

////////////////////////////////////////////////////////////////////////////////////////////////////
template<typename scalar_t> struct SSMScanOp;

template<>
struct SSMScanOp<float> {
    __device__ __forceinline__ float2 operator()(const float2 &ab0, const float2 &ab1) const {
        return make_float2(ab1.x * ab0.x, ab1.x * ab0.y + ab1.y);
    }
};

// A stateful callback functor that maintains a running prefix to be applied
// during consecutive scan operations.
template <typename scalar_t> struct SSMScanPrefixCallbackOp {
    using scan_t = std::conditional_t<std::is_same_v<scalar_t, float>, float2, float4>;
    scan_t running_prefix;
    // Constructor
    __device__ SSMScanPrefixCallbackOp(scan_t running_prefix_) : running_prefix(running_prefix_) {}
    // Callback operator to be entered by the first warp of threads in the block.
    // Thread-0 is responsible for returning a value for seeding the block-wide scan.
    __device__ scan_t operator()(scan_t block_aggregate) {
        scan_t old_prefix = running_prefix;
        running_prefix = SSMScanOp<scalar_t>()(running_prefix, block_aggregate);
        return old_prefix;
    }
};

////////////////////////////////////////////////////////////////////////////////////////////////////

template<typename Ktraits>
inline __device__ void load_input(typename Ktraits::input_t *u,
                                  typename Ktraits::input_t (&u_vals)[Ktraits::kNItems],
                                  typename Ktraits::BlockLoadT::TempStorage &smem_load,
                                  int seqlen) {
    if constexpr (Ktraits::kIsEvenLen) {
        auto& smem_load_vec = reinterpret_cast<typename Ktraits::BlockLoadVecT::TempStorage&>(smem_load);
        using vec_t = typename Ktraits::vec_t;
        Ktraits::BlockLoadVecT(smem_load_vec).Load(
            reinterpret_cast<vec_t*>(u),
            reinterpret_cast<vec_t(&)[Ktraits::kNLoads]>(u_vals)
       );
    } else {
        Ktraits::BlockLoadT(smem_load).Load(u, u_vals, seqlen, 0.f);
    }
}

template<typename Ktraits>
inline __device__ void load_weight(typename Ktraits::input_t *Bvar,
                                   typename Ktraits::weight_t (&B_vals)[Ktraits::kNItems],
                                   typename Ktraits::BlockLoadWeightT::TempStorage &smem_load_weight,
                                   int seqlen) {
    constexpr int kNItems = Ktraits::kNItems;
    typename Ktraits::input_t B_vals_load[kNItems];
    if constexpr (Ktraits::kIsEvenLen) {
        auto& smem_load_weight_vec = reinterpret_cast<typename Ktraits::BlockLoadWeightVecT::TempStorage&>(smem_load_weight);
        using vec_t = typename Ktraits::vec_t;
        Ktraits::BlockLoadWeightVecT(smem_load_weight_vec).Load(
            reinterpret_cast<vec_t*>(Bvar),
            reinterpret_cast<vec_t(&)[Ktraits::kNLoads]>(B_vals_load)
      );
    } else {
        Ktraits::BlockLoadWeightT(smem_load_weight).Load(Bvar, B_vals_load, seqlen, 0.f);
    }
    // #pragma unroll
    // for (int i = 0; i < kNItems; ++i) { B_vals[i] = B_vals_load[i]; }
    Converter<typename Ktraits::input_t, kNItems>::to_float(B_vals_load, B_vals);
}

template<typename Ktraits>
inline __device__ void store_output(typename Ktraits::input_t *out,
                                    const float (&out_vals)[Ktraits::kNItems],
                                    typename Ktraits::BlockStoreT::TempStorage &smem_store,
                                    int seqlen) {
    typename Ktraits::input_t write_vals[Ktraits::kNItems];
    #pragma unroll
    for (int i = 0; i < Ktraits::kNItems; ++i) { write_vals[i] = out_vals[i]; }
    if constexpr (Ktraits::kIsEvenLen) {
        auto& smem_store_vec = reinterpret_cast<typename Ktraits::BlockStoreVecT::TempStorage&>(smem_store);
        using vec_t = typename Ktraits::vec_t;
        Ktraits::BlockStoreVecT(smem_store_vec).Store(
            reinterpret_cast<vec_t*>(out),
            reinterpret_cast<vec_t(&)[Ktraits::kNLoads]>(write_vals)
       );
    } else {
        Ktraits::BlockStoreT(smem_store).Store(out, write_vals, seqlen);
    }
}

template<typename Ktraits>
inline __device__ void store_output1(typename Ktraits::output_t *out,
                                    const float (&out_vals)[Ktraits::kNItems],
                                    typename Ktraits::BlockStoreOutputT::TempStorage &smem_store,
                                    int seqlen) {
    typename Ktraits::output_t write_vals[Ktraits::kNItems];
    #pragma unroll
    for (int i = 0; i < Ktraits::kNItems; ++i) { write_vals[i] = out_vals[i]; }
    if constexpr (Ktraits::kIsEvenLen) {
        auto& smem_store_vec = reinterpret_cast<typename Ktraits::BlockStoreOutputVecT::TempStorage&>(smem_store);
        using vec_t = typename Ktraits::vec_t;
        Ktraits::BlockStoreOutputVecT(smem_store_vec).Store(
            reinterpret_cast<vec_t*>(out),
            reinterpret_cast<vec_t(&)[Ktraits::kNLoadsOutput]>(write_vals)
       );
    } else {
        Ktraits::BlockStoreOutputT(smem_store).Store(out, write_vals, seqlen);
    }
}

template<typename Ktraits>
inline __device__ void load_output(typename Ktraits::output_t *u,
                                  typename Ktraits::output_t (&u_vals)[Ktraits::kNItems],
                                  typename Ktraits::BlockLoadOutputT::TempStorage &smem_load,
                                  int seqlen) {
    if constexpr (Ktraits::kIsEvenLen) {
        auto& smem_load_vec = reinterpret_cast<typename Ktraits::BlockLoadOutputVecT::TempStorage&>(smem_load);
        using vec_t = typename Ktraits::vec_t;
        Ktraits::BlockLoadOutputVecT(smem_load_vec).Load(
            reinterpret_cast<vec_t*>(u),
            reinterpret_cast<vec_t(&)[Ktraits::kNLoadsOutput]>(u_vals)
       );
    } else {
        Ktraits::BlockLoadOutputT(smem_load).Load(u, u_vals, seqlen, 0.f);
    }
}


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/static_switch.h
// Inspired by https://github.com/NVIDIA/DALI/blob/main/include/dali/core/static_switch.h
// and https://github.com/pytorch/pytorch/blob/master/aten/src/ATen/Dispatch.h

#pragma once

/// @param COND       - a boolean expression to switch by
/// @param CONST_NAME - a name given for the constexpr bool variable.
/// @param ...       - code to execute for true and false
///
/// Usage:
/// ```
/// BOOL_SWITCH(flag, BoolConst, [&] {
///     some_function<BoolConst>(...);
/// });
/// ```
#define BOOL_SWITCH(COND, CONST_NAME, ...)                                           \
    [&] {                                                                            \
        if (COND) {                                                                  \
            constexpr bool CONST_NAME = true;                                        \
            return __VA_ARGS__();                                                    \
        } else {                                                                     \
            constexpr bool CONST_NAME = false;                                       \
            return __VA_ARGS__();                                                    \
        }                                                                            \
    }()


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cub_extra.cuh
// WarpMask is copied from /usr/local/cuda-12.1/include/cub/util_ptx.cuh
// PowerOfTwo is copied from /usr/local/cuda-12.1/include/cub/util_type.cuh

#pragma once

#include <cub/util_type.cuh>
#include <cub/util_arch.cuh>
#include <cub/util_namespace.cuh>
#include <cub/util_debug.cuh>

/**
 * \brief Statically determine if N is a power-of-two
 */
 template <int N>
 struct PowerOfTwo
 {
     enum { VALUE = ((N & (N - 1)) == 0) };
 };
 

/**
 * @brief Returns the warp mask for a warp of @p LOGICAL_WARP_THREADS threads
 *
 * @par
 * If the number of threads assigned to the virtual warp is not a power of two,
 * it's assumed that only one virtual warp exists.
 *
 * @tparam LOGICAL_WARP_THREADS <b>[optional]</b> The number of threads per
 *                              "logical" warp (may be less than the number of
 *                              hardware warp threads).
 * @param warp_id Id of virtual warp within architectural warp
 */
 template <int LOGICAL_WARP_THREADS, int LEGACY_PTX_ARCH = 0>
 __host__ __device__ __forceinline__
 unsigned int WarpMask(unsigned int warp_id)
 {
   constexpr bool is_pow_of_two = PowerOfTwo<LOGICAL_WARP_THREADS>::VALUE;
   constexpr bool is_arch_warp  = LOGICAL_WARP_THREADS == CUB_WARP_THREADS(0);
 
   unsigned int member_mask = 0xFFFFFFFFu >>
                              (CUB_WARP_THREADS(0) - LOGICAL_WARP_THREADS);
 
   if (is_pow_of_two && !is_arch_warp)
   {
     member_mask <<= warp_id * LOGICAL_WARP_THREADS;
   }
 
   return member_mask;
 }
 


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/reverse_scan.cuh
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#pragma once

#include <cub/config.cuh>

#include <cub/util_ptx.cuh>
#include <cub/util_type.cuh>
#include <cub/block/block_raking_layout.cuh>
// #include <cub/detail/uninitialized_copy.cuh>
#include "uninitialized_copy.cuh"
#include "cub_extra.cuh"

/**
 * Perform a reverse sequential reduction over \p LENGTH elements of the \p input array.  The aggregate is returned.
 */
template <
    int         LENGTH,
    typename    T,
    typename    ReductionOp>
__device__ __forceinline__ T ThreadReverseReduce(const T (&input)[LENGTH], ReductionOp reduction_op) {
    static_assert(LENGTH > 0);
    T retval = input[LENGTH - 1];
    #pragma unroll
    for (int i = LENGTH - 2; i >= 0; --i) { retval = reduction_op(retval, input[i]); }
    return retval;
}

/**
 * Perform a sequential inclusive postfix reverse scan over the statically-sized \p input array, seeded with the specified \p postfix.  The aggregate is returned.
 */
template <
    int         LENGTH,
    typename    T,
    typename    ScanOp>
__device__ __forceinline__ T ThreadReverseScanInclusive(
    const T (&input)[LENGTH],
    T (&output)[LENGTH],
    ScanOp scan_op,
    const T postfix)
{
    T inclusive = postfix;
    #pragma unroll
    for (int i = LENGTH - 1; i >= 0; --i) {
        inclusive = scan_op(inclusive, input[i]);
        output[i] = inclusive;
    }
}

/**
 * Perform a sequential exclusive postfix reverse scan over the statically-sized \p input array, seeded with the specified \p postfix.  The aggregate is returned.
 */
template <
    int         LENGTH,
    typename    T,
    typename    ScanOp>
__device__ __forceinline__ T ThreadReverseScanExclusive(
    const T (&input)[LENGTH],
    T (&output)[LENGTH],
    ScanOp scan_op,
    const T postfix)
{
    // Careful, output maybe be aliased to input
    T exclusive = postfix;
    T inclusive;
    #pragma unroll
    for (int i = LENGTH - 1; i >= 0; --i) {
        inclusive = scan_op(exclusive, input[i]);
        output[i] = exclusive;
        exclusive = inclusive;
    }
    return inclusive;
}


/**
 * \brief WarpReverseScan provides SHFL-based variants of parallel postfix scan of items partitioned across a CUDA thread warp.
 *
 * LOGICAL_WARP_THREADS must be a power-of-two
 */
template <
    typename    T,                      ///< Data type being scanned
    int         LOGICAL_WARP_THREADS    ///< Number of threads per logical warp
    >
struct WarpReverseScan {
    //---------------------------------------------------------------------
    // Constants and type definitions
    //---------------------------------------------------------------------

    /// Whether the logical warp size and the PTX warp size coincide
    static constexpr bool IS_ARCH_WARP = (LOGICAL_WARP_THREADS == CUB_WARP_THREADS(0));
    /// The number of warp scan steps
    static constexpr int STEPS = cub::Log2<LOGICAL_WARP_THREADS>::VALUE;
    static_assert(LOGICAL_WARP_THREADS == 1 << STEPS);


    //---------------------------------------------------------------------
    // Thread fields
    //---------------------------------------------------------------------

    /// Lane index in logical warp
    unsigned int lane_id;

    /// Logical warp index in 32-thread physical warp
    unsigned int warp_id;

    /// 32-thread physical warp member mask of logical warp
    unsigned int member_mask;

    //---------------------------------------------------------------------
    // Construction
    //---------------------------------------------------------------------

    /// Constructor
    explicit __device__ __forceinline__
    WarpReverseScan()
        : lane_id(cub::LaneId())
        , warp_id(IS_ARCH_WARP ? 0 : (lane_id / LOGICAL_WARP_THREADS))
        // , member_mask(cub::WarpMask<LOGICAL_WARP_THREADS>(warp_id))
        , member_mask(WarpMask<LOGICAL_WARP_THREADS>(warp_id))
    {
        if (!IS_ARCH_WARP) {
            lane_id = lane_id % LOGICAL_WARP_THREADS;
        }
    }


    /// Broadcast
    __device__ __forceinline__ T Broadcast(
        T               input,              ///< [in] The value to broadcast
        int             src_lane)           ///< [in] Which warp lane is to do the broadcasting
    {
        return cub::ShuffleIndex<LOGICAL_WARP_THREADS>(input, src_lane, member_mask);
    }


    /// Inclusive scan
    template <typename ScanOpT>
    __device__ __forceinline__ void InclusiveReverseScan(
        T               input,              ///< [in] Calling thread's input item.
        T               &inclusive_output,  ///< [out] Calling thread's output item.  May be aliased with \p input.
        ScanOpT         scan_op)            ///< [in] Binary scan operator
    {
        inclusive_output = input;
        #pragma unroll
        for (int STEP = 0; STEP < STEPS; STEP++) {
            int offset = 1 << STEP;
            T temp = cub::ShuffleDown<LOGICAL_WARP_THREADS>(
                inclusive_output, offset, LOGICAL_WARP_THREADS - 1, member_mask
            );
            // Perform scan op if from a valid peer
            inclusive_output = static_cast<int>(lane_id) >= LOGICAL_WARP_THREADS - offset
                ? inclusive_output : scan_op(temp, inclusive_output);
        }
    }

    /// Exclusive scan
    // Get exclusive from inclusive
    template <typename ScanOpT>
    __device__ __forceinline__ void ExclusiveReverseScan(
        T              input,              ///< [in] Calling thread's input item.
        T              &exclusive_output,  ///< [out] Calling thread's output item.  May be aliased with \p input.
        ScanOpT        scan_op,            ///< [in] Binary scan operator
        T              &warp_aggregate)    ///< [out] Warp-wide aggregate reduction of input items.
    {
        T inclusive_output;
        InclusiveReverseScan(input, inclusive_output, scan_op);
        warp_aggregate = cub::ShuffleIndex<LOGICAL_WARP_THREADS>(inclusive_output, 0, member_mask);
        // initial value unknown
        exclusive_output = cub::ShuffleDown<LOGICAL_WARP_THREADS>(
            inclusive_output, 1, LOGICAL_WARP_THREADS - 1, member_mask
        );
    }

    /**
     * \brief Computes both inclusive and exclusive reverse scans using the specified binary scan functor across the calling warp.  Because no initial value is supplied, the \p exclusive_output computed for the last <em>warp-lane</em> is undefined.
     */
    template <typename ScanOpT>
    __device__ __forceinline__ void ReverseScan(
        T               input,              ///< [in] Calling thread's input item.
        T               &inclusive_output,  ///< [out] Calling thread's inclusive-scan output item.
        T               &exclusive_output,  ///< [out] Calling thread's exclusive-scan output item.
        ScanOpT         scan_op)            ///< [in] Binary scan operator
    {
        InclusiveReverseScan(input, inclusive_output, scan_op);
        // initial value unknown
        exclusive_output = cub::ShuffleDown<LOGICAL_WARP_THREADS>(
            inclusive_output, 1, LOGICAL_WARP_THREADS - 1, member_mask
        );
    }

};

/**
 * \brief BlockReverseScan provides variants of raking-based parallel postfix scan across a CUDA thread block.
 */
template <
    typename    T,              ///< Data type being scanned
    int         BLOCK_DIM_X,    ///< The thread block length in threads along the X dimension
    bool        MEMOIZE=false   ///< Whether or not to buffer outer raking scan partials to incur fewer shared memory reads at the expense of higher register pressure
    >
struct BlockReverseScan {
    //---------------------------------------------------------------------
    // Types and constants
    //---------------------------------------------------------------------

    /// Constants
    /// The thread block size in threads
    static constexpr int BLOCK_THREADS = BLOCK_DIM_X;

    /// Layout type for padded thread block raking grid
    using BlockRakingLayout = cub::BlockRakingLayout<T, BLOCK_THREADS>;
    // The number of reduction elements is not a multiple of the number of raking threads for now
    static_assert(BlockRakingLayout::UNGUARDED);

    /// Number of raking threads
    static constexpr int RAKING_THREADS = BlockRakingLayout::RAKING_THREADS;
    /// Number of raking elements per warp synchronous raking thread
    static constexpr int SEGMENT_LENGTH = BlockRakingLayout::SEGMENT_LENGTH;
    /// Cooperative work can be entirely warp synchronous
    static constexpr bool WARP_SYNCHRONOUS = (int(BLOCK_THREADS) == int(RAKING_THREADS));

    ///  WarpReverseScan utility type
    using WarpReverseScan = WarpReverseScan<T, RAKING_THREADS>;

    /// Shared memory storage layout type
    struct _TempStorage {
        typename BlockRakingLayout::TempStorage raking_grid;     ///< Padded thread block raking grid
    };


    /// Alias wrapper allowing storage to be unioned
    struct TempStorage : cub::Uninitialized<_TempStorage> {};


    //---------------------------------------------------------------------
    // Per-thread fields
    //---------------------------------------------------------------------

    // Thread fields
    _TempStorage    &temp_storage;
    unsigned int    linear_tid;
    T               cached_segment[SEGMENT_LENGTH];


    //---------------------------------------------------------------------
    // Utility methods
    //---------------------------------------------------------------------

    /// Performs upsweep raking reduction, returning the aggregate
    template <typename ScanOp>
    __device__ __forceinline__ T Upsweep(ScanOp scan_op) {
        T *smem_raking_ptr = BlockRakingLayout::RakingPtr(temp_storage.raking_grid, linear_tid);
        // Read data into registers
        #pragma unroll
        for (int i = 0; i < SEGMENT_LENGTH; ++i) { cached_segment[i] = smem_raking_ptr[i]; }
        T raking_partial = cached_segment[SEGMENT_LENGTH - 1];
        #pragma unroll
        for (int i = SEGMENT_LENGTH - 2; i >= 0; --i) {
            raking_partial = scan_op(raking_partial, cached_segment[i]);
        }
        return raking_partial;
    }


    /// Performs exclusive downsweep raking scan
    template <typename ScanOp>
    __device__ __forceinline__ void ExclusiveDownsweep(
        ScanOp          scan_op,
        T               raking_partial)
    {
        T *smem_raking_ptr = BlockRakingLayout::RakingPtr(temp_storage.raking_grid, linear_tid);
        // Read data back into registers
        if (!MEMOIZE) {
            #pragma unroll
            for (int i = 0; i < SEGMENT_LENGTH; ++i) { cached_segment[i] = smem_raking_ptr[i]; }
        }
        ThreadReverseScanExclusive(cached_segment, cached_segment, scan_op, raking_partial);
        // Write data back to smem
        #pragma unroll
        for (int i = 0; i < SEGMENT_LENGTH; ++i) { smem_raking_ptr[i] = cached_segment[i]; }
    }


    //---------------------------------------------------------------------
    // Constructors
    //---------------------------------------------------------------------

    /// Constructor
    __device__ __forceinline__ BlockReverseScan(
        TempStorage &temp_storage)
    :
        temp_storage(temp_storage.Alias()),
        linear_tid(cub::RowMajorTid(BLOCK_DIM_X, 1, 1))
    {}


    /// Computes an exclusive thread block-wide postfix scan using the specified binary \p scan_op functor.  Each thread contributes one input element.  the call-back functor \p block_postfix_callback_op is invoked by the first warp in the block, and the value returned by <em>lane</em><sub>0</sub> in that warp is used as the "seed" value that logically postfixes the thread block's scan inputs.  Also provides every thread with the block-wide \p block_aggregate of all inputs.
    template <
        typename ScanOp,
        typename BlockPostfixCallbackOp>
    __device__ __forceinline__ void ExclusiveReverseScan(
        T                       input,                          ///< [in] Calling thread's input item
        T                       &exclusive_output,              ///< [out] Calling thread's output item (may be aliased to \p input)
        ScanOp                  scan_op,                        ///< [in] Binary scan operator
        BlockPostfixCallbackOp  &block_postfix_callback_op)     ///< [in-out] <b>[<em>warp</em><sub>0</sub> only]</b> Call-back functor for specifying a thread block-wide postfix to be applied to all inputs.
    {
        if (WARP_SYNCHRONOUS) {
            // Short-circuit directly to warp-synchronous scan
            T block_aggregate;
            WarpReverseScan warp_scan;
            warp_scan.ExclusiveReverseScan(input, exclusive_output, scan_op, block_aggregate);
            // Obtain warp-wide postfix in lane0, then broadcast to other lanes
            T block_postfix = block_postfix_callback_op(block_aggregate);
            block_postfix = warp_scan.Broadcast(block_postfix, 0);
            exclusive_output = linear_tid == BLOCK_THREADS - 1 ? block_postfix : scan_op(block_postfix, exclusive_output);
        } else {
            // Place thread partial into shared memory raking grid
            T *placement_ptr = BlockRakingLayout::PlacementPtr(temp_storage.raking_grid, linear_tid);
            detail::uninitialized_copy(placement_ptr, input);
            cub::CTA_SYNC();
            // Reduce parallelism down to just raking threads
            if (linear_tid < RAKING_THREADS) {
                WarpReverseScan warp_scan;
                // Raking upsweep reduction across shared partials
                T upsweep_partial = Upsweep(scan_op);
                // Warp-synchronous scan
                T exclusive_partial, block_aggregate;
                warp_scan.ExclusiveReverseScan(upsweep_partial, exclusive_partial, scan_op, block_aggregate);
                // Obtain block-wide postfix in lane0, then broadcast to other lanes
                T block_postfix = block_postfix_callback_op(block_aggregate);
                block_postfix = warp_scan.Broadcast(block_postfix, 0);
                // Update postfix with warpscan exclusive partial
                T downsweep_postfix = linear_tid == RAKING_THREADS - 1
                    ? block_postfix : scan_op(block_postfix, exclusive_partial);
                // Exclusive raking downsweep scan
                ExclusiveDownsweep(scan_op, downsweep_postfix);
            }
            cub::CTA_SYNC();
            // Grab thread postfix from shared memory
            exclusive_output = *placement_ptr;

            // // Compute warp scan in each warp.
            // // The exclusive output from the last lane in each warp is invalid.
            // T inclusive_output;
            // WarpReverseScan warp_scan;
            // warp_scan.ReverseScan(input, inclusive_output, exclusive_output, scan_op);

            // // Compute the warp-wide postfix and block-wide aggregate for each warp.  Warp postfix for the last warp is invalid.
            // T block_aggregate;
            // T warp_postfix = ComputeWarpPostfix(scan_op, inclusive_output, block_aggregate);

            // // Apply warp postfix to our lane's partial
            // if (warp_id != 0) {
            //     exclusive_output = scan_op(warp_postfix, exclusive_output);
            //     if (lane_id == 0) { exclusive_output = warp_postfix; }
            // }

            // // Use the first warp to determine the thread block postfix, returning the result in lane0
            // if (warp_id == 0) {
            //     T block_postfix = block_postfix_callback_op(block_aggregate);
            //     if (lane_id == 0) {
            //         // Share the postfix with all threads
            //         detail::uninitialized_copy(&temp_storage.block_postfix,
            //                                   block_postfix);

            //         exclusive_output = block_postfix; // The block postfix is the exclusive output for tid0
            //     }
            // }

            // cub::CTA_SYNC();

            // // Incorporate thread block postfix into outputs
            // T block_postfix = temp_storage.block_postfix;
            // if (linear_tid > 0) { exclusive_output = scan_op(block_postfix, exclusive_output); }
        }
    }


    /**
     * \brief Computes an inclusive block-wide postfix scan using the specified binary \p scan_op functor.  Each thread contributes an array of consecutive input elements.  the call-back functor \p block_postfix_callback_op is invoked by the first warp in the block, and the value returned by <em>lane</em><sub>0</sub> in that warp is used as the "seed" value that logically postfixes the thread block's scan inputs.  Also provides every thread with the block-wide \p block_aggregate of all inputs.
     */
    template <
        int             ITEMS_PER_THREAD,
        typename        ScanOp,
        typename        BlockPostfixCallbackOp>
    __device__ __forceinline__ void InclusiveReverseScan(
        T                       (&input)[ITEMS_PER_THREAD],     ///< [in] Calling thread's input items
        T                       (&output)[ITEMS_PER_THREAD],    ///< [out] Calling thread's output items (may be aliased to \p input)
        ScanOp                  scan_op,                        ///< [in] Binary scan functor
        BlockPostfixCallbackOp   &block_postfix_callback_op)    ///< [in-out] <b>[<em>warp</em><sub>0</sub> only]</b> Call-back functor for specifying a block-wide postfix to be applied to the logical input sequence.
    {
        // Reduce consecutive thread items in registers
        T thread_postfix = ThreadReverseReduce(input, scan_op);
        // Exclusive thread block-scan
        ExclusiveReverseScan(thread_postfix, thread_postfix, scan_op, block_postfix_callback_op);
        // Inclusive scan in registers with postfix as seed
        ThreadReverseScanInclusive(input, output, scan_op, thread_postfix);
    }

};


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/uninitialized_copy.cuh
/******************************************************************************
 * Copyright (c) 2011-2022, NVIDIA CORPORATION.  All rights reserved.
 *
 * Redistribution and use in source and binary forms, with or without
 * modification, are permitted provided that the following conditions are met:
 *     * Redistributions of source code must retain the above copyright
 *       notice, this list of conditions and the following disclaimer.
 *     * Redistributions in binary form must reproduce the above copyright
 *       notice, this list of conditions and the following disclaimer in the
 *       documentation and/or other materials provided with the distribution.
 *     * Neither the name of the NVIDIA CORPORATION nor the
 *       names of its contributors may be used to endorse or promote products
 *       derived from this software without specific prior written permission.
 *
 * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
 * AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
 * IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE
 * ARE DISCLAIMED. IN NO EVENT SHALL NVIDIA CORPORATION BE LIABLE FOR ANY
 * DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES
 * (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;
 * LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND
 * ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT
 * (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS
 * SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
 *
 ******************************************************************************/

#pragma once

#include <cub/config.cuh>

#include <cuda/std/type_traits>


namespace detail
{

#if defined(_NVHPC_CUDA)
template <typename T, typename U>
__host__ __device__ void uninitialized_copy(T *ptr, U &&val)
{
  // NVBug 3384810
  new (ptr) T(::cuda::std::forward<U>(val));
}
#else
template <typename T,
          typename U,
          typename ::cuda::std::enable_if<
            ::cuda::std::is_trivially_copyable<T>::value,
            int
          >::type = 0>
__host__ __device__ void uninitialized_copy(T *ptr, U &&val)
{
  *ptr = ::cuda::std::forward<U>(val);
}

template <typename T,
         typename U,
         typename ::cuda::std::enable_if<
           !::cuda::std::is_trivially_copyable<T>::value,
           int
         >::type = 0>
__host__ __device__ void uninitialized_copy(T *ptr, U &&val)
{
  new (ptr) T(::cuda::std::forward<U>(val));
}
#endif

} // namespace detail


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cusoflex/selective_scan_oflex.h
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#pragma once

////////////////////////////////////////////////////////////////////////////////////////////////////

struct SSMScanParamsBase {
    using index_t = uint32_t;

    int batch, seqlen, n_chunks;
    index_t a_batch_stride;
    index_t b_batch_stride;
    index_t out_batch_stride;

    // Common data pointers.
    void *__restrict__ a_ptr;
    void *__restrict__ b_ptr;
    void *__restrict__ out_ptr;
    void *__restrict__ x_ptr;
};

////////////////////////////////////////////////////////////////////////////////////////////////////

struct SSMParamsBase {
    using index_t = uint32_t;

    int batch, dim, seqlen, dstate, n_groups, n_chunks;
    int dim_ngroups_ratio, dim_deltagroups_ratio;

    bool delta_softplus;

    index_t A_d_stride;
    index_t A_dstate_stride;
    index_t B_batch_stride;
    index_t B_d_stride;
    index_t B_dstate_stride;
    index_t B_group_stride;
    index_t C_batch_stride;
    index_t C_d_stride;
    index_t C_dstate_stride;
    index_t C_group_stride;
    index_t u_batch_stride;
    index_t u_d_stride;
    index_t delta_batch_stride;
    index_t delta_d_stride;
    index_t out_batch_stride;
    index_t out_d_stride;

    // Common data pointers.
    void *__restrict__ A_ptr;
    void *__restrict__ B_ptr;
    void *__restrict__ C_ptr;
    void *__restrict__ D_ptr;
    void *__restrict__ u_ptr;
    void *__restrict__ delta_ptr;
    void *__restrict__ delta_bias_ptr;
    void *__restrict__ out_ptr;
    void *__restrict__ x_ptr;
};

struct SSMParamsBwd: public SSMParamsBase {
    index_t dout_batch_stride;
    index_t dout_d_stride;
    index_t dA_d_stride;
    index_t dA_dstate_stride;
    index_t dB_batch_stride;
    index_t dB_group_stride;
    index_t dB_d_stride;
    index_t dB_dstate_stride;
    index_t dC_batch_stride;
    index_t dC_group_stride;
    index_t dC_d_stride;
    index_t dC_dstate_stride;
    index_t du_batch_stride;
    index_t du_d_stride;
    index_t ddelta_batch_stride;
    index_t ddelta_d_stride;

    // Common data pointers.
    void *__restrict__ dout_ptr;
    void *__restrict__ dA_ptr;
    void *__restrict__ dB_ptr;
    void *__restrict__ dC_ptr;
    void *__restrict__ dD_ptr;
    void *__restrict__ du_ptr;
    void *__restrict__ ddelta_ptr;
    void *__restrict__ ddelta_bias_ptr;
};


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cusoflex/selective_scan_oflex.cpp
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <torch/extension.h>
#include <vector>

#include "selective_scan_oflex.h"
#define MAX_DSTATE 256

#define CHECK_SHAPE(x, ...) TORCH_CHECK(x.sizes() == torch::IntArrayRef({__VA_ARGS__}), #x " must have shape (" #__VA_ARGS__ ")")
using weight_t = float;

#define DISPATCH_ITYPE_FLOAT_AND_HALF_AND_BF16(ITYPE, NAME, ...)                    \
    if (ITYPE == at::ScalarType::Half) {                                            \
        using input_t = at::Half;                                                   \
        __VA_ARGS__();                                                              \
    } else if (ITYPE == at::ScalarType::BFloat16) {                                 \
        using input_t = at::BFloat16;                                               \
        __VA_ARGS__();                                                              \
    } else if (ITYPE == at::ScalarType::Float)  {                                   \
        using input_t = float;                                                      \
        __VA_ARGS__();                                                              \
    } else {                                                                        \
        AT_ERROR(#NAME, " not implemented for input type '", toString(ITYPE), "'"); \
    }

template<int knrows, typename input_t, typename weight_t, typename output_t>
void selective_scan_fwd_cuda(SSMParamsBase &params, cudaStream_t stream);

template <int knrows, typename input_t, typename weight_t, typename output_t>
void selective_scan_bwd_cuda(SSMParamsBwd &params, cudaStream_t stream);

void set_ssm_params_fwd(SSMParamsBase &params,
                        const size_t n_chunks,
                        const at::Tensor &u,
                        const at::Tensor &delta,
                        const at::Tensor &A,
                        const at::Tensor &B,
                        const at::Tensor &C,
                        const at::Tensor &out,
                        void* D_ptr,
                        void* delta_bias_ptr,
                        void* x_ptr,
                        bool delta_softplus) {

    // Reset the parameters
    memset(&params, 0, sizeof(params));

    params.batch = u.size(0);
    params.dim = u.size(1);
    params.seqlen = u.size(2);
    params.dstate = B.size(2);
    params.n_groups = B.size(1);
    params.n_chunks = n_chunks;
    params.dim_ngroups_ratio = params.dim / params.n_groups;
    params.dim_deltagroups_ratio = params.dim / delta.size(1);

    params.delta_softplus = delta_softplus;

    // Set the pointers and strides.
    params.u_ptr = u.data_ptr();
    params.delta_ptr = delta.data_ptr();
    params.A_ptr = A.data_ptr();
    params.B_ptr = B.data_ptr();
    params.C_ptr = C.data_ptr();
    params.D_ptr = D_ptr;
    params.delta_bias_ptr = delta_bias_ptr;
    params.out_ptr = out.data_ptr();
    params.x_ptr = x_ptr;

    // All stride are in elements, not bytes.
    params.A_d_stride = A.stride(0);
    params.A_dstate_stride = A.stride(1);
    params.B_batch_stride = B.stride(0);
    params.B_group_stride = B.stride(1);
    params.B_dstate_stride = B.stride(2);
    params.C_batch_stride = C.stride(0);
    params.C_group_stride = C.stride(1);
    params.C_dstate_stride = C.stride(2);
    params.u_batch_stride = u.stride(0);
    params.u_d_stride = u.stride(1);
    params.delta_batch_stride = delta.stride(0);
    params.delta_d_stride = delta.stride(1);

    params.out_batch_stride = out.stride(0);
    params.out_d_stride = out.stride(1);
}

void set_ssm_params_bwd(SSMParamsBwd &params,
                        const size_t n_chunks,
                        const at::Tensor &u,
                        const at::Tensor &delta,
                        const at::Tensor &A,
                        const at::Tensor &B,
                        const at::Tensor &C,
                        const at::Tensor &out,
                        void* D_ptr,
                        void* delta_bias_ptr,
                        void* x_ptr,
                        const at::Tensor &dout,
                        const at::Tensor &du,
                        const at::Tensor &ddelta,
                        const at::Tensor &dA,
                        const at::Tensor &dB,
                        const at::Tensor &dC,
                        void* dD_ptr,
                        void* ddelta_bias_ptr,
                        bool delta_softplus) {
    set_ssm_params_fwd(params, n_chunks,
                       u, delta, A, B, C, dout,
                       D_ptr, delta_bias_ptr, x_ptr, delta_softplus);

    // Set the pointers and strides.
    params.dout_ptr = dout.data_ptr();
    params.du_ptr = du.data_ptr();
    params.dA_ptr = dA.data_ptr();
    params.dB_ptr = dB.data_ptr();
    params.dC_ptr = dC.data_ptr();
    params.dD_ptr = dD_ptr;
    params.ddelta_ptr = ddelta.data_ptr();
    params.ddelta_bias_ptr = ddelta_bias_ptr;
    // All stride are in elements, not bytes.
    params.dout_batch_stride = dout.stride(0);
    params.dout_d_stride = dout.stride(1);
    params.dA_d_stride = dA.stride(0);
    params.dA_dstate_stride = dA.stride(1);
    params.dB_batch_stride = dB.stride(0);
    params.dB_group_stride = dB.stride(1);
    params.dB_dstate_stride = dB.stride(2);
    params.dC_batch_stride = dC.stride(0);
    params.dC_group_stride = dC.stride(1);
    params.dC_dstate_stride = dC.stride(2);
    params.du_batch_stride = du.stride(0);
    params.du_d_stride = du.stride(1);
    params.ddelta_batch_stride = ddelta.stride(0);
    params.ddelta_d_stride = ddelta.stride(1);

}

std::vector<at::Tensor>
selective_scan_fwd(const at::Tensor &u, const at::Tensor &delta,
                  const at::Tensor &A, const at::Tensor &B, const at::Tensor &C,
                  const c10::optional<at::Tensor> &D_,
                  const c10::optional<at::Tensor> &delta_bias_,
                  bool delta_softplus,
                  int nrows,
                  bool out_float
                  ) {
    auto input_type = u.scalar_type();
    auto weight_type = A.scalar_type();
    TORCH_CHECK(input_type == at::ScalarType::Float || input_type == at::ScalarType::Half || input_type == at::ScalarType::BFloat16);
    TORCH_CHECK(weight_type == at::ScalarType::Float);

    TORCH_CHECK(delta.scalar_type() == input_type);
    TORCH_CHECK(B.scalar_type() == input_type);
    TORCH_CHECK(C.scalar_type() == input_type);

    TORCH_CHECK(u.is_cuda());
    TORCH_CHECK(delta.is_cuda());
    TORCH_CHECK(A.is_cuda());
    TORCH_CHECK(B.is_cuda());
    TORCH_CHECK(C.is_cuda());

    TORCH_CHECK(u.stride(-1) == 1 || u.size(-1) == 1);
    TORCH_CHECK(delta.stride(-1) == 1 || delta.size(-1) == 1);

    const auto sizes = u.sizes();
    const int batch_size = sizes[0];
    const int dim = sizes[1];
    const int seqlen = sizes[2];
    const int dstate = A.size(1);
    const int n_groups = B.size(1);
    const int delta_dim = delta.size(1);

    TORCH_CHECK(dim % n_groups == 0, "dims should be dividable by n_groups");
    TORCH_CHECK(dim % delta_dim == 0, "dims should be dividable by delta_dim");
    TORCH_CHECK(dstate <= MAX_DSTATE, "selective_scan only supports state dimension <= 256");

    CHECK_SHAPE(u, batch_size, dim, seqlen);
    CHECK_SHAPE(delta, batch_size, delta_dim, seqlen);
    CHECK_SHAPE(A, dim, dstate);
    CHECK_SHAPE(B, batch_size, n_groups, dstate, seqlen);
    TORCH_CHECK(B.stride(-1) == 1 || B.size(-1) == 1);
    CHECK_SHAPE(C, batch_size, n_groups, dstate, seqlen);
    TORCH_CHECK(C.stride(-1) == 1 || C.size(-1) == 1);

    if (D_.has_value()) {
        auto D = D_.value();
        TORCH_CHECK(D.scalar_type() == at::ScalarType::Float);
        TORCH_CHECK(D.is_cuda());
        TORCH_CHECK(D.stride(-1) == 1 || D.size(-1) == 1);
        CHECK_SHAPE(D, dim);
    }

    if (delta_bias_.has_value()) {
        auto delta_bias = delta_bias_.value();
        TORCH_CHECK(delta_bias.scalar_type() == at::ScalarType::Float);
        TORCH_CHECK(delta_bias.is_cuda());
        TORCH_CHECK(delta_bias.stride(-1) == 1 || delta_bias.size(-1) == 1);
        CHECK_SHAPE(delta_bias, delta_dim);
    }

    const int n_chunks = (seqlen + 2048 - 1) / 2048; // max is 128 * 16 = 2048 in fwd_kernel
    at::Tensor out = torch::empty({batch_size, dim, seqlen}, u.options().dtype(out_float? (at::ScalarType::Float): input_type));
    at::Tensor x = torch::empty({batch_size, dim, n_chunks, dstate * 2}, u.options().dtype(weight_type));

    SSMParamsBase params;
    set_ssm_params_fwd(params, n_chunks,
                       u, delta, A, B, C, out,
                       D_.has_value() ? D_.value().data_ptr() : nullptr,
                       delta_bias_.has_value() ? delta_bias_.value().data_ptr() : nullptr,
                       x.data_ptr(),
                       delta_softplus);

    // Otherwise the kernel will be launched from cuda:0 device
    // Cast to char to avoid compiler warning about narrowing
    at::cuda::CUDAGuard device_guard{(char)u.get_device()};
    auto stream = at::cuda::getCurrentCUDAStream().stream();
    DISPATCH_ITYPE_FLOAT_AND_HALF_AND_BF16(u.scalar_type(), "selective_scan_fwd", [&] {
        if (!out_float) {
            selective_scan_fwd_cuda<1, input_t, weight_t, input_t>(params, stream);
        } else {
            selective_scan_fwd_cuda<1, input_t, weight_t, float>(params, stream);
        }
    });
    std::vector<at::Tensor> result = {out, x};
    return result;
}

std::vector<at::Tensor>
selective_scan_bwd(const at::Tensor &u, const at::Tensor &delta,
                  const at::Tensor &A, const at::Tensor &B, const at::Tensor &C,
                  const c10::optional<at::Tensor> &D_,
                  const c10::optional<at::Tensor> &delta_bias_,
                  const at::Tensor &dout,
                  const c10::optional<at::Tensor> &x_,
                  bool delta_softplus,
                  int nrows
                  ) {
    auto input_type = u.scalar_type();
    auto weight_type = A.scalar_type();
    auto output_type = dout.scalar_type();
    TORCH_CHECK(input_type == at::ScalarType::Float || input_type == at::ScalarType::Half || input_type == at::ScalarType::BFloat16);
    TORCH_CHECK(weight_type == at::ScalarType::Float);
    TORCH_CHECK(output_type == input_type || output_type == at::ScalarType::Float);

    TORCH_CHECK(delta.scalar_type() == input_type);
    TORCH_CHECK(B.scalar_type() == input_type);
    TORCH_CHECK(C.scalar_type() == input_type);

    TORCH_CHECK(u.is_cuda());
    TORCH_CHECK(delta.is_cuda());
    TORCH_CHECK(A.is_cuda());
    TORCH_CHECK(B.is_cuda());
    TORCH_CHECK(C.is_cuda());
    TORCH_CHECK(dout.is_cuda());

    TORCH_CHECK(u.stride(-1) == 1 || u.size(-1) == 1);
    TORCH_CHECK(delta.stride(-1) == 1 || delta.size(-1) == 1);
    TORCH_CHECK(dout.stride(-1) == 1 || dout.size(-1) == 1);

    const auto sizes = u.sizes();
    const int batch_size = sizes[0];
    const int dim = sizes[1];
    const int seqlen = sizes[2];
    const int dstate = A.size(1);
    const int n_groups = B.size(1);
    const int delta_dim = delta.size(1);
    const int delta_dim_repeat = dim / delta_dim;

    TORCH_CHECK(dim % n_groups == 0, "dims should be dividable by n_groups");
    TORCH_CHECK(dim % delta_dim == 0, "dims should be dividable by delta_dim");
    TORCH_CHECK(dstate <= MAX_DSTATE, "selective_scan only supports state dimension <= 256");
    
    CHECK_SHAPE(u, batch_size, dim, seqlen);
    CHECK_SHAPE(delta, batch_size, delta_dim, seqlen);
    CHECK_SHAPE(A, dim, dstate);
    CHECK_SHAPE(B, batch_size, n_groups, dstate, seqlen);
    TORCH_CHECK(B.stride(-1) == 1 || B.size(-1) == 1);
    CHECK_SHAPE(C, batch_size, n_groups, dstate, seqlen);
    TORCH_CHECK(C.stride(-1) == 1 || C.size(-1) == 1);
    CHECK_SHAPE(dout, batch_size, dim, seqlen);

    if (D_.has_value()) {
        auto D = D_.value();
        TORCH_CHECK(D.scalar_type() == at::ScalarType::Float);
        TORCH_CHECK(D.is_cuda());
        TORCH_CHECK(D.stride(-1) == 1 || D.size(-1) == 1);
        CHECK_SHAPE(D, dim);
    }

    if (delta_bias_.has_value()) {
        auto delta_bias = delta_bias_.value();
        TORCH_CHECK(delta_bias.scalar_type() == at::ScalarType::Float);
        TORCH_CHECK(delta_bias.is_cuda());
        TORCH_CHECK(delta_bias.stride(-1) == 1 || delta_bias.size(-1) == 1);
        CHECK_SHAPE(delta_bias, delta_dim);
    }

    at::Tensor out;
    const int n_chunks = (seqlen + 2048 - 1) / 2048; // 1024
    if (n_chunks > 1) { TORCH_CHECK(x_.has_value()); }
    if (x_.has_value()) {
        auto x = x_.value();
        TORCH_CHECK(x.scalar_type() == weight_type);
        TORCH_CHECK(x.is_cuda());
        TORCH_CHECK(x.is_contiguous());
        CHECK_SHAPE(x, batch_size, dim, n_chunks, 2 * dstate);
    }

    at::Tensor du = torch::empty_like(u);
    at::Tensor ddelta = torch::empty_like(u);
    at::Tensor dA = torch::zeros_like(A);
    at::Tensor dB = torch::zeros_like(B, B.options().dtype(torch::kFloat32));
    at::Tensor dC = torch::zeros_like(C, C.options().dtype(torch::kFloat32));
    at::Tensor dD;
    if (D_.has_value()) { dD = torch::zeros_like(D_.value()); }
    at::Tensor ddelta_bias;
    if (delta_bias_.has_value()) { ddelta_bias = torch::zeros({dim}, delta_bias_.value().options()); }

    SSMParamsBwd params;
    set_ssm_params_bwd(params, n_chunks,
                       u, delta, A, B, C, out,
                       D_.has_value() ? D_.value().data_ptr() : nullptr,
                       delta_bias_.has_value() ? delta_bias_.value().data_ptr() : nullptr,
                       x_.has_value() ? x_.value().data_ptr() : nullptr,
                       dout, du, ddelta, dA, dB, dC,
                       D_.has_value() ? dD.data_ptr() : nullptr,
                       delta_bias_.has_value() ? ddelta_bias.data_ptr() : nullptr,
                       delta_softplus);

    // Otherwise the kernel will be launched from cuda:0 device
    // Cast to char to avoid compiler warning about narrowing
    at::cuda::CUDAGuard device_guard{(char)u.get_device()};
    auto stream = at::cuda::getCurrentCUDAStream().stream();
    DISPATCH_ITYPE_FLOAT_AND_HALF_AND_BF16(u.scalar_type(), "selective_scan_bwd", [&] {
        if (output_type == input_type) {
            selective_scan_bwd_cuda<1, input_t, weight_t, input_t>(params, stream);
        } else {
            selective_scan_bwd_cuda<1, input_t, weight_t, float>(params, stream);
        }
    });

    std::vector<at::Tensor> result = {du, ddelta, dA, dB.to(B.dtype()), dC.to(C.dtype()), dD, ddelta_bias};
    if (dim != delta_dim) {
        result[1] = ddelta.view({batch_size, delta_dim, delta_dim_repeat, seqlen}).sum(2);
        if (delta_bias_.has_value()) {
            result[6] = ddelta_bias.view({delta_dim, delta_dim_repeat}).sum(1);
        }
    }
    return result;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("fwd", &selective_scan_fwd, "Selective scan forward");
    m.def("bwd", &selective_scan_bwd, "Selective scan backward");
}


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cusoflex/selective_scan_fwd_kernel_oflex.cuh
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#pragma once

#include <c10/util/BFloat16.h>
#include <c10/util/Half.h>
#include <c10/cuda/CUDAException.h>  // For C10_CUDA_CHECK and C10_CUDA_KERNEL_LAUNCH_CHECK

#include <cub/block/block_load.cuh>
#include <cub/block/block_store.cuh>
#include <cub/block/block_scan.cuh>

#include "selective_scan_oflex.h"
#include "selective_scan_common.h"
#include "static_switch.h"

template<int kNThreads_, int kNItems_, bool kIsEvenLen_, typename input_t_, typename weight_t_, typename output_t_>
struct Selective_Scan_fwd_kernel_traits {
    static_assert(kNItems_ % 4 == 0);
    using input_t = input_t_;
    using weight_t = weight_t_;
    using output_t = output_t_;
    static constexpr int kNThreads = kNThreads_;
    // Setting MinBlocksPerMP to be 3 (instead of 2) for 128 threads improves occupancy.
    static constexpr int kMinBlocks = kNThreads < 128 ? 5 : 3;
    static constexpr int kNItems = kNItems_;
    static constexpr int MaxDState = MAX_DSTATE;
    static constexpr int kNBytes = sizeof(input_t);
    static_assert(kNBytes == 2 || kNBytes == 4);
    static constexpr int kNElts = kNBytes == 4 ? 4 : std::min(8, kNItems);
    static_assert(kNItems % kNElts == 0);
    static constexpr int kNLoads = kNItems / kNElts;
    static constexpr bool kIsEvenLen = kIsEvenLen_;
    static constexpr bool kDirectIO = kIsEvenLen && kNLoads == 1;
    static constexpr int kNLoadsOutput = sizeof(output_t) * kNLoads / kNBytes;
    static constexpr bool kDirectIOOutput = kDirectIO && (kNLoadsOutput == 1);
    using vec_t = typename BytesToType<kNBytes * kNElts>::Type;
    using scan_t = float2;
    using BlockLoadT = cub::BlockLoad<input_t, kNThreads, kNItems, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadVecT = cub::BlockLoad<vec_t, kNThreads, kNLoads,
        !kDirectIO ? cub::BLOCK_LOAD_WARP_TRANSPOSE : cub::BLOCK_LOAD_DIRECT>;
    using BlockLoadWeightT = cub::BlockLoad<input_t, kNThreads, kNItems, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadWeightVecT = cub::BlockLoad<vec_t, kNThreads, kNLoads,
        !kDirectIO ? cub::BLOCK_LOAD_WARP_TRANSPOSE  : cub::BLOCK_LOAD_DIRECT>;
    using BlockStoreT = cub::BlockStore<input_t, kNThreads, kNItems, cub::BLOCK_STORE_WARP_TRANSPOSE>;
    using BlockStoreVecT = cub::BlockStore<vec_t, kNThreads, kNLoads,
        !kDirectIO ? cub::BLOCK_STORE_WARP_TRANSPOSE : cub::BLOCK_STORE_DIRECT>;
    using BlockStoreOutputT = cub::BlockStore<output_t, kNThreads, kNItems, cub::BLOCK_STORE_WARP_TRANSPOSE>;
    using BlockStoreOutputVecT = cub::BlockStore<vec_t, kNThreads, kNLoadsOutput,
        !kDirectIOOutput ? cub::BLOCK_STORE_WARP_TRANSPOSE  : cub::BLOCK_STORE_DIRECT>;
    // using BlockScanT = cub::BlockScan<scan_t, kNThreads, cub::BLOCK_SCAN_RAKING_MEMOIZE>;
    // using BlockScanT = cub::BlockScan<scan_t, kNThreads, cub::BLOCK_SCAN_RAKING>;
    using BlockScanT = cub::BlockScan<scan_t, kNThreads, cub::BLOCK_SCAN_WARP_SCANS>;
    static constexpr int kSmemIOSize = std::max({sizeof(typename BlockLoadT::TempStorage),
                                                 sizeof(typename BlockLoadVecT::TempStorage),
                                                 2 * sizeof(typename BlockLoadWeightT::TempStorage),
                                                 2 * sizeof(typename BlockLoadWeightVecT::TempStorage),
                                                 sizeof(typename BlockStoreT::TempStorage),
                                                 sizeof(typename BlockStoreVecT::TempStorage),
                                                 sizeof(typename BlockStoreOutputT::TempStorage),
                                                 sizeof(typename BlockStoreOutputVecT::TempStorage)});
    static constexpr int kSmemSize = kSmemIOSize + sizeof(typename BlockScanT::TempStorage);
};

template<typename Ktraits>
__global__ __launch_bounds__(Ktraits::kNThreads, Ktraits::kMinBlocks)
void selective_scan_fwd_kernel(SSMParamsBase params) {
    constexpr int kNThreads = Ktraits::kNThreads;
    constexpr int kNItems = Ktraits::kNItems;
    constexpr bool kDirectIO = Ktraits::kDirectIO;
    using input_t = typename Ktraits::input_t;
    using weight_t = typename Ktraits::weight_t;
    using output_t = typename Ktraits::output_t;
    using scan_t = typename Ktraits::scan_t;

    // Shared memory.
    extern __shared__ char smem_[];
    auto& smem_load = reinterpret_cast<typename Ktraits::BlockLoadT::TempStorage&>(smem_);
    auto& smem_load_weight = reinterpret_cast<typename Ktraits::BlockLoadWeightT::TempStorage&>(smem_);
    auto& smem_load_weight1 = *reinterpret_cast<typename Ktraits::BlockLoadWeightT::TempStorage*>(smem_ + sizeof(typename Ktraits::BlockLoadWeightT::TempStorage));
    auto& smem_store = reinterpret_cast<typename Ktraits::BlockStoreT::TempStorage&>(smem_);
    auto& smem_store1 = reinterpret_cast<typename Ktraits::BlockStoreOutputT::TempStorage&>(smem_);
    auto& smem_scan = *reinterpret_cast<typename Ktraits::BlockScanT::TempStorage*>(smem_ + Ktraits::kSmemIOSize);
    scan_t *smem_running_prefix = reinterpret_cast<scan_t *>(smem_ + Ktraits::kSmemSize);

    const int batch_id = blockIdx.x;
    const int dim_id = blockIdx.y;
    const int group_id = dim_id / (params.dim_ngroups_ratio);
    const int delta_group_id = dim_id / (params.dim_deltagroups_ratio);
    input_t *u = reinterpret_cast<input_t *>(params.u_ptr) + batch_id * params.u_batch_stride
        + dim_id * params.u_d_stride;
    input_t *delta = reinterpret_cast<input_t *>(params.delta_ptr) + batch_id * params.delta_batch_stride
        + delta_group_id * params.delta_d_stride;
    weight_t *A = reinterpret_cast<weight_t *>(params.A_ptr) + dim_id * params.A_d_stride;
    input_t *Bvar = reinterpret_cast<input_t *>(params.B_ptr) + batch_id * params.B_batch_stride + group_id * params.B_group_stride;
    input_t *Cvar = reinterpret_cast<input_t *>(params.C_ptr) + batch_id * params.C_batch_stride + group_id * params.C_group_stride;
    scan_t *x = reinterpret_cast<scan_t *>(params.x_ptr) + (batch_id * params.dim + dim_id) * params.n_chunks * params.dstate;

    float D_val = 0; // attention!
    if (params.D_ptr != nullptr) {
        D_val = reinterpret_cast<float *>(params.D_ptr)[dim_id];
    }
    float delta_bias = 0;
    if (params.delta_bias_ptr != nullptr) {
        delta_bias = reinterpret_cast<float *>(params.delta_bias_ptr)[delta_group_id];
    }

    constexpr int kChunkSize = kNThreads * kNItems;
    for (int chunk = 0; chunk < params.n_chunks; ++chunk) {
        input_t u_vals[kNItems], delta_vals_load[kNItems];
        __syncthreads();
        load_input<Ktraits>(u, u_vals, smem_load, params.seqlen - chunk * kChunkSize);
        if constexpr (!kDirectIO) { __syncthreads(); }
        load_input<Ktraits>(delta, delta_vals_load, smem_load, params.seqlen - chunk * kChunkSize);
        u += kChunkSize;
        delta += kChunkSize;

        float delta_vals[kNItems], delta_u_vals[kNItems], out_vals[kNItems];
        #pragma unroll
        for (int i = 0; i < kNItems; ++i) {
            float u_val = float(u_vals[i]);
            delta_vals[i] = float(delta_vals_load[i]) + delta_bias;
            if (params.delta_softplus) {
                delta_vals[i] = delta_vals[i] <= 20.f ? log1pf(expf(delta_vals[i])) : delta_vals[i];
            }
            delta_u_vals[i] = delta_vals[i] * u_val;
            out_vals[i] = D_val * u_val;
        }

        __syncthreads();
        for (int state_idx = 0; state_idx < params.dstate; ++state_idx) {
            constexpr float kLog2e = M_LOG2E;
            weight_t A_val = A[state_idx * params.A_dstate_stride];
            A_val *= kLog2e;
            weight_t B_vals[kNItems], C_vals[kNItems];
            load_weight<Ktraits>(Bvar + state_idx * params.B_dstate_stride, B_vals,
                    smem_load_weight, (params.seqlen - chunk * kChunkSize));
            load_weight<Ktraits>(Cvar + state_idx * params.C_dstate_stride, C_vals,
                    smem_load_weight1, (params.seqlen - chunk * kChunkSize));
            __syncthreads();
            scan_t thread_data[kNItems];
            #pragma unroll
            for (int i = 0; i < kNItems; ++i) {
                thread_data[i] = make_float2(exp2f(delta_vals[i] * A_val), B_vals[i] * delta_u_vals[i]);
                if constexpr (!Ktraits::kIsEvenLen) {  // So that the last state is correct
                    if (threadIdx.x * kNItems + i >= params.seqlen - chunk * kChunkSize) {
                        thread_data[i] = make_float2(1.f, 0.f);
                    }
                }
            }
            // Initialize running total
            scan_t running_prefix;
            // If we use WARP_SCAN then all lane 0 of all warps (not just thread 0) needs to read
            running_prefix = chunk > 0 && threadIdx.x % 32 == 0 ? smem_running_prefix[state_idx] : make_float2(1.f, 0.f);
            // running_prefix = chunk > 0 && threadIdx.x == 0 ? smem_running_prefix[state_idx] : make_float2(1.f, 0.f);
            SSMScanPrefixCallbackOp<weight_t> prefix_op(running_prefix);
            Ktraits::BlockScanT(smem_scan).InclusiveScan(
                thread_data, thread_data, SSMScanOp<weight_t>(), prefix_op
            );
            // There's a syncthreads in the scan op, so we don't need to sync here.
            // Unless there's only 1 warp, but then it's the same thread (0) reading and writing.
            if (threadIdx.x == 0) {
                smem_running_prefix[state_idx] = prefix_op.running_prefix;
                x[chunk * params.dstate + state_idx] = prefix_op.running_prefix;
            }
            #pragma unroll
            for (int i = 0; i < kNItems; ++i) {
                out_vals[i] += thread_data[i].y * C_vals[i];
            }
        }

        output_t *out = reinterpret_cast<output_t *>(params.out_ptr) + batch_id * params.out_batch_stride
            + dim_id * params.out_d_stride + chunk * kChunkSize;
        __syncthreads();
        store_output1<Ktraits>(out, out_vals, smem_store1, params.seqlen - chunk * kChunkSize);
        Bvar += kChunkSize;
        Cvar += kChunkSize;
    }
}

template<int kNThreads, int kNItems, typename input_t, typename weight_t, typename output_t>
void selective_scan_fwd_launch(SSMParamsBase &params, cudaStream_t stream) {
    BOOL_SWITCH(params.seqlen % (kNThreads * kNItems) == 0, kIsEvenLen, [&] {
        using Ktraits = Selective_Scan_fwd_kernel_traits<kNThreads, kNItems, kIsEvenLen, input_t, weight_t, output_t>;
        constexpr int kSmemSize = Ktraits::kSmemSize + Ktraits::MaxDState * sizeof(typename Ktraits::scan_t);
        // printf("smem_size = %d\n", kSmemSize);
        dim3 grid(params.batch, params.dim);
        auto kernel = &selective_scan_fwd_kernel<Ktraits>;
        if (kSmemSize >= 48 * 1024) {
            C10_CUDA_CHECK(cudaFuncSetAttribute(kernel, cudaFuncAttributeMaxDynamicSharedMemorySize, kSmemSize));
        }
        kernel<<<grid, Ktraits::kNThreads, kSmemSize, stream>>>(params);
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    });
}

template<int knrows, typename input_t, typename weight_t, typename output_t>
void selective_scan_fwd_cuda(SSMParamsBase &params, cudaStream_t stream) {
    if (params.seqlen <= 128) {
        selective_scan_fwd_launch<32, 4, input_t, weight_t, output_t>(params, stream);
    } else if (params.seqlen <= 256) {
        selective_scan_fwd_launch<32, 8, input_t, weight_t, output_t>(params, stream);
    } else if (params.seqlen <= 512) {
        selective_scan_fwd_launch<32, 16, input_t, weight_t, output_t>(params, stream);
    } else if (params.seqlen <= 1024) {
        selective_scan_fwd_launch<64, 16, input_t, weight_t, output_t>(params, stream);
    } else {
        selective_scan_fwd_launch<128, 16, input_t, weight_t, output_t>(params, stream);
    }
}


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cusoflex/selective_scan_bwd_kernel_oflex.cuh
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/

#pragma once

#include <c10/util/BFloat16.h>
#include <c10/util/Half.h>
#include <c10/cuda/CUDAException.h>  // For C10_CUDA_CHECK and C10_CUDA_KERNEL_LAUNCH_CHECK
#include <ATen/cuda/Atomic.cuh>  // For atomicAdd on complex

#include <cub/block/block_load.cuh>
#include <cub/block/block_store.cuh>
#include <cub/block/block_scan.cuh>
#include <cub/block/block_reduce.cuh>

#include "selective_scan_oflex.h"
#include "selective_scan_common.h"
#include "reverse_scan.cuh"
#include "static_switch.h"

template<int kNThreads_, int kNItems_, bool kIsEvenLen_, bool kDeltaSoftplus_, typename input_t_, typename weight_t_, typename output_t_>
struct Selective_Scan_bwd_kernel_traits {
    static_assert(kNItems_ % 4 == 0);
    using input_t = input_t_;
    using weight_t = weight_t_;
    using output_t = output_t_;

    static constexpr int kNThreads = kNThreads_;
    static constexpr int kNItems = kNItems_;
    static constexpr int MaxDState = MAX_DSTATE;
    static constexpr int kNBytes = sizeof(input_t);
    static_assert(kNBytes == 2 || kNBytes == 4);
    static constexpr int kNElts = kNBytes == 4 ? 4 : std::min(8, kNItems);
    static_assert(kNItems % kNElts == 0);
    static constexpr int kNLoads = kNItems / kNElts;
    static constexpr bool kIsEvenLen = kIsEvenLen_;
    static constexpr bool kDeltaSoftplus = kDeltaSoftplus_;
    // Setting MinBlocksPerMP to be 3 (instead of 2) for 128 threads with float improves occupancy.
    // For complex this would lead to massive register spilling, so we keep it at 2.
    static constexpr int kMinBlocks = kNThreads == 128 && 3;
    static constexpr int kNLoadsOutput = sizeof(output_t) * kNLoads / kNBytes;
    using vec_t = typename BytesToType<kNBytes * kNElts>::Type;
    using scan_t = float2;
    using BlockLoadT = cub::BlockLoad<input_t, kNThreads, kNItems, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadVecT = cub::BlockLoad<vec_t, kNThreads, kNLoads, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadWeightT = cub::BlockLoad<input_t, kNThreads, kNItems, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadWeightVecT = cub::BlockLoad<vec_t, kNThreads, kNLoads, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadOutputT = cub::BlockLoad<output_t, kNThreads, kNItems, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockLoadOutputVecT = cub::BlockLoad<vec_t, kNThreads, kNLoadsOutput, cub::BLOCK_LOAD_WARP_TRANSPOSE>;
    using BlockStoreT = cub::BlockStore<input_t, kNThreads, kNItems, cub::BLOCK_STORE_WARP_TRANSPOSE>;
    using BlockStoreVecT = cub::BlockStore<vec_t, kNThreads, kNLoads, cub::BLOCK_STORE_WARP_TRANSPOSE>;
    // using BlockScanT = cub::BlockScan<scan_t, kNThreads, cub::BLOCK_SCAN_RAKING_MEMOIZE>;
    using BlockScanT = cub::BlockScan<scan_t, kNThreads, cub::BLOCK_SCAN_RAKING>;
    // using BlockScanT = cub::BlockScan<scan_t, kNThreads, cub::BLOCK_SCAN_WARP_SCANS>;
    using BlockReverseScanT = BlockReverseScan<scan_t, kNThreads>;
    using BlockReduceT = cub::BlockReduce<scan_t, kNThreads>;
    using BlockReduceFloatT = cub::BlockReduce<float, kNThreads>;
    using BlockExchangeT = cub::BlockExchange<float, kNThreads, kNItems>;
    static constexpr int kSmemIOSize = std::max({sizeof(typename BlockLoadT::TempStorage),
                                                 sizeof(typename BlockLoadVecT::TempStorage),
                                                 2 * sizeof(typename BlockLoadWeightT::TempStorage),
                                                 2 * sizeof(typename BlockLoadWeightVecT::TempStorage),
                                                 sizeof(typename BlockLoadOutputT::TempStorage),
                                                 sizeof(typename BlockLoadOutputVecT::TempStorage),
                                                 sizeof(typename BlockStoreT::TempStorage),
                                                 sizeof(typename BlockStoreVecT::TempStorage)});
    static constexpr int kSmemExchangeSize = 2 * sizeof(typename BlockExchangeT::TempStorage);
    static constexpr int kSmemReduceSize = sizeof(typename BlockReduceT::TempStorage);
    static constexpr int kSmemSize = kSmemIOSize + kSmemExchangeSize + kSmemReduceSize + sizeof(typename BlockScanT::TempStorage) + sizeof(typename BlockReverseScanT::TempStorage);
};

template<typename Ktraits>
__global__ __launch_bounds__(Ktraits::kNThreads, Ktraits::kMinBlocks)
void selective_scan_bwd_kernel(SSMParamsBwd params) {
    constexpr bool kDeltaSoftplus = Ktraits::kDeltaSoftplus;
    constexpr int kNThreads = Ktraits::kNThreads;
    constexpr int kNItems = Ktraits::kNItems;
    using input_t = typename Ktraits::input_t;
    using weight_t = typename Ktraits::weight_t;
    using output_t = typename Ktraits::output_t;
    using scan_t = typename Ktraits::scan_t;

    // Shared memory.
    extern __shared__ char smem_[];
    auto& smem_load = reinterpret_cast<typename Ktraits::BlockLoadT::TempStorage&>(smem_);
    auto& smem_load1 = reinterpret_cast<typename Ktraits::BlockLoadOutputT::TempStorage&>(smem_);
    auto& smem_load_weight = reinterpret_cast<typename Ktraits::BlockLoadWeightT::TempStorage&>(smem_);
    auto& smem_load_weight1 = *reinterpret_cast<typename Ktraits::BlockLoadWeightT::TempStorage*>(smem_ + sizeof(typename Ktraits::BlockLoadWeightT::TempStorage));
    auto& smem_store = reinterpret_cast<typename Ktraits::BlockStoreT::TempStorage&>(smem_);
    auto& smem_exchange = *reinterpret_cast<typename Ktraits::BlockExchangeT::TempStorage*>(smem_ + Ktraits::kSmemIOSize);
    auto& smem_exchange1 = *reinterpret_cast<typename Ktraits::BlockExchangeT::TempStorage*>(smem_ + Ktraits::kSmemIOSize + sizeof(typename Ktraits::BlockExchangeT::TempStorage));
    auto& smem_reduce = *reinterpret_cast<typename Ktraits::BlockReduceT::TempStorage*>(reinterpret_cast<char *>(&smem_exchange) + Ktraits::kSmemExchangeSize);
    auto& smem_reduce_float = *reinterpret_cast<typename Ktraits::BlockReduceFloatT::TempStorage*>(&smem_reduce);
    auto& smem_scan = *reinterpret_cast<typename Ktraits::BlockScanT::TempStorage*>(reinterpret_cast<char *>(&smem_reduce) + Ktraits::kSmemReduceSize);
    auto& smem_reverse_scan = *reinterpret_cast<typename Ktraits::BlockReverseScanT::TempStorage*>(reinterpret_cast<char *>(&smem_scan) + sizeof(typename Ktraits::BlockScanT::TempStorage));
    weight_t *smem_delta_a = reinterpret_cast<weight_t *>(smem_ + Ktraits::kSmemSize);
    scan_t *smem_running_postfix = reinterpret_cast<scan_t *>(smem_delta_a + 2 * Ktraits::MaxDState + kNThreads);
    weight_t *smem_da = reinterpret_cast<weight_t *>(smem_running_postfix + Ktraits::MaxDState);

    const int batch_id = blockIdx.x;
    const int dim_id = blockIdx.y;
    const int group_id = dim_id / (params.dim_ngroups_ratio);
    const int delta_group_id = dim_id / (params.dim_deltagroups_ratio);
    input_t *u = reinterpret_cast<input_t *>(params.u_ptr) + batch_id * params.u_batch_stride
        + dim_id * params.u_d_stride;
    input_t *delta = reinterpret_cast<input_t *>(params.delta_ptr) + batch_id * params.delta_batch_stride
        + delta_group_id * params.delta_d_stride;

    weight_t *A = reinterpret_cast<weight_t *>(params.A_ptr) + dim_id * params.A_d_stride;
    input_t *Bvar = reinterpret_cast<input_t *>(params.B_ptr) + batch_id * params.B_batch_stride + group_id * params.B_group_stride;
    input_t *Cvar = reinterpret_cast<input_t *>(params.C_ptr) + batch_id * params.C_batch_stride + group_id * params.C_group_stride;
    weight_t *dA = reinterpret_cast<weight_t *>(params.dA_ptr) + dim_id * params.dA_d_stride;
    weight_t *dB = reinterpret_cast<weight_t *>(params.dB_ptr)
        + (batch_id * params.dB_batch_stride + group_id * params.dB_group_stride);
    weight_t *dC = reinterpret_cast<weight_t *>(params.dC_ptr)
        + (batch_id * params.dC_batch_stride + group_id * params.dC_group_stride);
    float *dD = params.dD_ptr == nullptr ? nullptr : reinterpret_cast<float *>(params.dD_ptr) + dim_id;
    float D_val = params.D_ptr == nullptr ? 0 : reinterpret_cast<float *>(params.D_ptr)[dim_id];
    float *ddelta_bias = params.ddelta_bias_ptr == nullptr ? nullptr : reinterpret_cast<float *>(params.ddelta_bias_ptr) + dim_id;
    float delta_bias = params.delta_bias_ptr == nullptr ? 0 : reinterpret_cast<float *>(params.delta_bias_ptr)[delta_group_id];
    scan_t *x = params.x_ptr == nullptr
        ? nullptr
        : reinterpret_cast<scan_t *>(params.x_ptr) + (batch_id * params.dim + dim_id) * (params.n_chunks) * params.dstate;
    float dD_val = 0;
    float ddelta_bias_val = 0;

    output_t *dout = reinterpret_cast<output_t *>(params.dout_ptr) + batch_id * params.dout_batch_stride + dim_id * params.dout_d_stride;

    constexpr int kChunkSize = kNThreads * kNItems;
    u += (params.n_chunks - 1) * kChunkSize;
    delta += (params.n_chunks - 1) * kChunkSize;
    dout += (params.n_chunks - 1) * kChunkSize;
    Bvar += (params.n_chunks - 1) * kChunkSize;
    Cvar += (params.n_chunks - 1) * kChunkSize;
    for (int chunk = params.n_chunks - 1; chunk >= 0; --chunk) {
        input_t u_vals[kNItems];
        input_t delta_vals_load[kNItems];
        float dout_vals[kNItems];
        __syncthreads();
        load_input<Ktraits>(u, u_vals, smem_load, params.seqlen - chunk * kChunkSize);
        __syncthreads();
        load_input<Ktraits>(delta, delta_vals_load, smem_load, params.seqlen - chunk * kChunkSize);
        __syncthreads();
        if constexpr (std::is_same_v<output_t, input_t>) {
            input_t dout_vals_load[kNItems];
            load_input<Ktraits>(reinterpret_cast<input_t *>(dout), dout_vals_load, smem_load, params.seqlen - chunk * kChunkSize);
            Converter<typename Ktraits::input_t, kNItems>::to_float(dout_vals_load, dout_vals);
        } else {
            static_assert(std::is_same_v<output_t, float>);
            load_output<Ktraits>(dout, dout_vals, smem_load1, params.seqlen - chunk * kChunkSize);
        }
        u -= kChunkSize;
        // Will reload delta at the same location if kDeltaSoftplus
        if constexpr (!kDeltaSoftplus) { delta -= kChunkSize; }
        dout -= kChunkSize;

        float delta_vals[kNItems];
        float du_vals[kNItems];
        #pragma unroll
        for (int i = 0; i < kNItems; ++i) {
            delta_vals[i] = float(delta_vals_load[i]) + delta_bias;
            if constexpr (kDeltaSoftplus) {
                delta_vals[i] = delta_vals[i] <= 20.f ? log1pf(expf(delta_vals[i])) : delta_vals[i];
            }
        }
        #pragma unroll
        for (int i = 0; i < kNItems; ++i) { du_vals[i] = D_val * dout_vals[i]; }
        #pragma unroll
        for (int i = 0; i < kNItems; ++i) { dD_val += dout_vals[i] * float(u_vals[i]); }

        float ddelta_vals[kNItems] = {0};
        __syncthreads();
        for (int state_idx = 0; state_idx < params.dstate; ++state_idx) {
            constexpr float kLog2e = M_LOG2E;
            weight_t A_val = A[state_idx * params.A_dstate_stride];
            weight_t A_scaled = A_val * kLog2e;
            weight_t B_vals[kNItems], C_vals[kNItems];
            load_weight<Ktraits>(Bvar + state_idx * params.B_dstate_stride, B_vals,
                    smem_load_weight, (params.seqlen - chunk * kChunkSize));
            auto &smem_load_weight_C = smem_load_weight1;
            load_weight<Ktraits>(Cvar + state_idx * params.C_dstate_stride, C_vals,
                    smem_load_weight_C, (params.seqlen - chunk * kChunkSize));
            scan_t thread_data[kNItems], thread_reverse_data[kNItems];
            #pragma unroll
            for (int i = 0; i < kNItems; ++i) {
                const float delta_a_exp = exp2f(delta_vals[i] * A_scaled);
                thread_data[i] = make_float2(delta_a_exp, delta_vals[i] * float(u_vals[i]) * B_vals[i]);
                if (i == 0) {
                    smem_delta_a[threadIdx.x == 0 ? state_idx + (chunk % 2) * Ktraits::MaxDState: threadIdx.x + 2 * Ktraits::MaxDState] = delta_a_exp;
                } else {
                    thread_reverse_data[i - 1].x = delta_a_exp;
                }
                thread_reverse_data[i].y = dout_vals[i] * C_vals[i];
            }
            __syncthreads();
            thread_reverse_data[kNItems - 1].x = threadIdx.x == kNThreads - 1
                ? (chunk == params.n_chunks - 1 ? 1.f : smem_delta_a[state_idx + ((chunk + 1) % 2) * Ktraits::MaxDState])
                    : smem_delta_a[threadIdx.x + 1 + 2 * Ktraits::MaxDState];
            // Initialize running total
            scan_t running_prefix = chunk > 0 && threadIdx.x % 32 == 0 ? x[(chunk - 1) * params.dstate + state_idx] : make_float2(1.f, 0.f);
            SSMScanPrefixCallbackOp<weight_t> prefix_op(running_prefix);
            Ktraits::BlockScanT(smem_scan).InclusiveScan(
                thread_data, thread_data, SSMScanOp<weight_t>(), prefix_op
            );
            scan_t running_postfix = chunk < params.n_chunks - 1 && threadIdx.x % 32 == 0 ? smem_running_postfix[state_idx] : make_float2(1.f, 0.f);
            SSMScanPrefixCallbackOp<weight_t> postfix_op(running_postfix);
            Ktraits::BlockReverseScanT(smem_reverse_scan).InclusiveReverseScan(
                thread_reverse_data, thread_reverse_data, SSMScanOp<weight_t>(), postfix_op
            );
            if (threadIdx.x == 0) { smem_running_postfix[state_idx] = postfix_op.running_prefix; }
            weight_t dA_val = 0;
            weight_t dB_vals[kNItems], dC_vals[kNItems];
            #pragma unroll
            for (int i = 0; i < kNItems; ++i) {
                const float dx = thread_reverse_data[i].y;
                const float ddelta_u = dx * B_vals[i];
                du_vals[i] += ddelta_u * delta_vals[i];
                const float a = thread_data[i].y - (delta_vals[i] * float(u_vals[i]) * B_vals[i]);
                ddelta_vals[i] += ddelta_u * float(u_vals[i]) + dx * A_val * a;
                dA_val += dx * delta_vals[i] * a;
                dB_vals[i] = dx * delta_vals[i] * float(u_vals[i]);
                dC_vals[i] = dout_vals[i] * thread_data[i].y;
            }
            // Block-exchange to make the atomicAdd's coalesced, otherwise they're much slower
            Ktraits::BlockExchangeT(smem_exchange).BlockedToStriped(dB_vals, dB_vals);
            auto &smem_exchange_C = smem_exchange1;
            Ktraits::BlockExchangeT(smem_exchange_C).BlockedToStriped(dC_vals, dC_vals);
            const int seqlen_remaining = params.seqlen - chunk * kChunkSize - threadIdx.x;
            weight_t *dB_cur = dB + state_idx * params.dB_dstate_stride + chunk * kChunkSize + threadIdx.x;
            weight_t *dC_cur = dC + state_idx * params.dC_dstate_stride + chunk * kChunkSize + threadIdx.x;
            #pragma unroll
            for (int i = 0; i < kNItems; ++i) {
                if (i * kNThreads < seqlen_remaining) {
                    { gpuAtomicAdd(dB_cur + i * kNThreads, dB_vals[i]); }
                    { gpuAtomicAdd(dC_cur + i * kNThreads, dC_vals[i]); }
                }
            }
            dA_val = Ktraits::BlockReduceFloatT(smem_reduce_float).Sum(dA_val);
            if (threadIdx.x == 0) {
                smem_da[state_idx] = chunk == params.n_chunks - 1 ? dA_val : dA_val + smem_da[state_idx];
            }
        }

        if constexpr (kDeltaSoftplus) {
            input_t delta_vals_load[kNItems];
            __syncthreads();
            load_input<Ktraits>(delta, delta_vals_load, smem_load, params.seqlen - chunk * kChunkSize);
            delta -= kChunkSize;
            #pragma unroll
            for (int i = 0; i < kNItems; ++i) {
                float delta_val = float(delta_vals_load[i]) + delta_bias;
                float delta_val_neg_exp = expf(-delta_val);
                ddelta_vals[i] = delta_val <= 20.f
                    ? ddelta_vals[i] / (1.f + delta_val_neg_exp)
                        : ddelta_vals[i];
            }
        }
        
        __syncthreads();
        #pragma unroll
        for (int i = 0; i < kNItems; ++i) { ddelta_bias_val += ddelta_vals[i]; }

        input_t *du = reinterpret_cast<input_t *>(params.du_ptr) + batch_id * params.du_batch_stride
            + dim_id * params.du_d_stride + chunk * kChunkSize;
        input_t *ddelta = reinterpret_cast<input_t *>(params.ddelta_ptr) + batch_id * params.ddelta_batch_stride
            + dim_id * params.ddelta_d_stride + chunk * kChunkSize;
        __syncthreads();
        store_output<Ktraits>(du, du_vals, smem_store, params.seqlen - chunk * kChunkSize);
        __syncthreads();
        store_output<Ktraits>(ddelta, ddelta_vals, smem_store, params.seqlen - chunk * kChunkSize);
        Bvar -= kChunkSize;
        Cvar -= kChunkSize;
    }

    if (params.dD_ptr != nullptr) {
        __syncthreads();
        dD_val = Ktraits::BlockReduceFloatT(smem_reduce_float).Sum(dD_val);
        if (threadIdx.x == 0) { gpuAtomicAdd(dD, dD_val); }
    }
    if (params.ddelta_bias_ptr != nullptr) {
        __syncthreads();
        ddelta_bias_val = Ktraits::BlockReduceFloatT(smem_reduce_float).Sum(ddelta_bias_val);
        if (threadIdx.x == 0) { gpuAtomicAdd(ddelta_bias, ddelta_bias_val); }
    }
    __syncthreads();
    for (int state_idx = threadIdx.x; state_idx < params.dstate; state_idx += blockDim.x) {
        gpuAtomicAdd(&(dA[state_idx * params.dA_dstate_stride]), smem_da[state_idx]);
    }
}

template<int kNThreads, int kNItems, typename input_t, typename weight_t, typename output_t>
void selective_scan_bwd_launch(SSMParamsBwd &params, cudaStream_t stream) {
    BOOL_SWITCH(params.seqlen % (kNThreads * kNItems) == 0, kIsEvenLen, [&] {
        BOOL_SWITCH(params.delta_softplus, kDeltaSoftplus, [&] {
            using Ktraits = Selective_Scan_bwd_kernel_traits<kNThreads, kNItems, kIsEvenLen, kDeltaSoftplus, input_t, weight_t, output_t>;
            constexpr int kSmemSize = Ktraits::kSmemSize + Ktraits::MaxDState * sizeof(typename Ktraits::scan_t) + (kNThreads + 4 * Ktraits::MaxDState) * sizeof(typename Ktraits::weight_t);
            // printf("smem_size = %d\n", kSmemSize);
            dim3 grid(params.batch, params.dim);
            auto kernel = &selective_scan_bwd_kernel<Ktraits>;
            if (kSmemSize >= 48 * 1024) {
                C10_CUDA_CHECK(cudaFuncSetAttribute(kernel, cudaFuncAttributeMaxDynamicSharedMemorySize, kSmemSize));
            }
            kernel<<<grid, Ktraits::kNThreads, kSmemSize, stream>>>(params);
            C10_CUDA_KERNEL_LAUNCH_CHECK();
        });
    });
}

template<int knrows, typename input_t, typename weight_t, typename output_t>
void selective_scan_bwd_cuda(SSMParamsBwd &params, cudaStream_t stream) {
    if (params.seqlen <= 128) {
        selective_scan_bwd_launch<32, 4, input_t, weight_t, output_t>(params, stream);
    } else if (params.seqlen <= 256) {
        selective_scan_bwd_launch<32, 8, input_t, weight_t, output_t>(params, stream);
    } else if (params.seqlen <= 512) {
        selective_scan_bwd_launch<32, 16, input_t, weight_t, output_t>(params, stream);
    } else if (params.seqlen <= 1024) {
        selective_scan_bwd_launch<64, 16, input_t, weight_t, output_t>(params, stream);
    } else {
        selective_scan_bwd_launch<128, 16, input_t, weight_t, output_t>(params, stream);
    }
}


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cusoflex/selective_scan_core_fwd.cu
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/
#include "selective_scan_fwd_kernel_oflex.cuh"

template void selective_scan_fwd_cuda<1, float, float, float>(SSMParamsBase &params, cudaStream_t stream);
template void selective_scan_fwd_cuda<1, at::Half, float, float>(SSMParamsBase &params, cudaStream_t stream);
template void selective_scan_fwd_cuda<1, at::BFloat16, float, float>(SSMParamsBase &params, cudaStream_t stream);
template void selective_scan_fwd_cuda<1, at::Half, float, at::Half>(SSMParamsBase &params, cudaStream_t stream);
template void selective_scan_fwd_cuda<1, at::BFloat16, float, at::BFloat16>(SSMParamsBase &params, cudaStream_t stream);


In [ ]:
%%writefile {SRC_DIR}/selective_scan/csrc/selective_scan/cusoflex/selective_scan_core_bwd.cu
/******************************************************************************
 * Copyright (c) 2023, Tri Dao.
 ******************************************************************************/
#include "selective_scan_bwd_kernel_oflex.cuh"

template void selective_scan_bwd_cuda<1, float, float, float>(SSMParamsBwd &params, cudaStream_t stream);
template void selective_scan_bwd_cuda<1, at::Half, float, float>(SSMParamsBwd &params, cudaStream_t stream);
template void selective_scan_bwd_cuda<1, at::BFloat16, float, float>(SSMParamsBwd &params, cudaStream_t stream);
template void selective_scan_bwd_cuda<1, at::Half, float, at::Half>(SSMParamsBwd &params, cudaStream_t stream);
template void selective_scan_bwd_cuda<1, at::BFloat16, float, at::BFloat16>(SSMParamsBwd &params, cudaStream_t stream);


### Build
This is the same extension, sources and compiler flags as the repo's `setup.py` (`MODES = ["oflex"]`), with three
build-level adaptations. **None of them changes the kernel source.**
* The GPU architecture is set explicitly from the detected device (`-arch=sm_XX`).
* The C++ standard is raised to C++20 when the installed PyTorch headers require it (recent PyTorch refuses
  C++17). The repo pins c++17 for torch 2.2.
* A tiny compatibility header is force-included (`-include tev_cub_compat.cuh`). Newer CUDA (CCCL/CUB 3.x) removed
  `cub::LaneId()` and `cub::CTA_SYNC()`, which `reverse_scan.cuh` calls. The header re-defines them (lane-id
  register read / `__syncthreads()`) only when CUB ≥ 3.0, so older toolkits are unaffected.

In [ ]:
def _so_tag():
    cc = "".join(map(str, torch.cuda.get_device_capability(0))) if DEVICE == "cuda" else "cpu"
    return f"torch{torch.__version__.split('+')[0]}_cu{torch.version.cuda}_sm{cc}_py{sys.version_info[0]}{sys.version_info[1]}"


def _try_import_kernel(path_dir):
    if path_dir not in sys.path:
        sys.path.insert(0, path_dir)
    try:
        import importlib
        mod = importlib.import_module("selective_scan_cuda_oflex")
        return mod
    except Exception as e:  # noqa: BLE001
        print("   import failed:", type(e).__name__, str(e)[:200])
        return None


def build_selective_scan():
    if DEVICE != "cuda":
        print("[kernel] no GPU -> skipping CUDA build (pure-PyTorch scan will be used)")
        return False
    tag = _so_tag()
    cache_dir = osp.join(OUT_DIR, "kernel_cache", tag)
    # 1) cached build from this session or an attached previous output
    cands = glob.glob(osp.join(cache_dir, "selective_scan_cuda_oflex*.so")) + \
        glob.glob(osp.join(INPUT_ROOT, "**", "kernel_cache", tag, "selective_scan_cuda_oflex*.so"), recursive=True)
    for so in cands:
        d = osp.join(CACHE_DIR, "kernel_import"); os.makedirs(d, exist_ok=True)
        shutil.copy(so, d)
        if _try_import_kernel(d):
            print(f"[kernel] using cached build {so}")
            return True
    # 2) compile
    nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if osp.exists("/usr/local/cuda/bin/nvcc") else None)
    if nvcc is None:
        print("[kernel] nvcc not found -> cannot compile; pure-PyTorch scan will be used")
        return False
    cc = "".join(map(str, torch.cuda.get_device_capability(0)))
    torch_inc = osp.join(osp.dirname(torch.__file__), "include")
    needs20 = any("C++20 or later" in open(f, errors="ignore").read()
                  for f in (osp.join(torch_inc, "ATen", "ATen.h"), osp.join(torch_inc, "torch", "all.h")) if osp.isfile(f))
    std = "c++20" if needs20 else "c++17"
    compat = osp.join(KERNEL_DIR, "tev_cub_compat.cuh")
    open(compat, "w").write(
        "#pragma once\n#include <cub/version.cuh>\n"
        "#if defined(CUB_VERSION) && CUB_VERSION >= 300000\n"
        "namespace cub {\n"
        "__device__ __forceinline__ unsigned int LaneId() { unsigned int r; asm volatile(\"mov.u32 %0, %%laneid;\" : \"=r\"(r)); return r; }\n"
        "__device__ __forceinline__ void CTA_SYNC() { __syncthreads(); }\n"
        "}\n#endif\n")
    setup_py = f'''
import os
from pathlib import Path
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension
this_dir = os.path.dirname(os.path.abspath(__file__))
setup(
    name="selective_scan", version="0.0.2", packages=[],
    ext_modules=[CUDAExtension(
        name="selective_scan_cuda_oflex",
        sources=["csrc/selective_scan/cusoflex/selective_scan_oflex.cpp",
                 "csrc/selective_scan/cusoflex/selective_scan_core_fwd.cu",
                 "csrc/selective_scan/cusoflex/selective_scan_core_bwd.cu"],
        extra_compile_args={{"cxx": ["-O3", "-std={std}"],
            "nvcc": ["-O3", "-std={std}", "-include", "{compat}", "-U__CUDA_NO_HALF_OPERATORS__", "-U__CUDA_NO_HALF_CONVERSIONS__",
                     "-U__CUDA_NO_BFLOAT16_OPERATORS__", "-U__CUDA_NO_BFLOAT16_CONVERSIONS__",
                     "-U__CUDA_NO_BFLOAT162_OPERATORS__", "-U__CUDA_NO_BFLOAT162_CONVERSIONS__",
                     "--expt-relaxed-constexpr", "--expt-extended-lambda", "--use_fast_math", "-lineinfo",
                     "-arch=sm_{cc}", "--threads", "4"]}},
        include_dirs=[Path(this_dir) / "csrc" / "selective_scan"])],
    cmdclass={{"build_ext": BuildExtension}})
'''
    open(osp.join(KERNEL_DIR, "setup.py"), "w").write(setup_py)
    env = dict(os.environ, MAX_JOBS="4", CUDA_HOME=osp.dirname(osp.dirname(nvcc)))
    env["PATH"] = osp.dirname(nvcc) + ":" + env.get("PATH", "")
    print(f"[kernel] compiling for sm_{cc} with {nvcc}, -std={std} (5-10 min) ...")
    t0 = time.time()
    p = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=KERNEL_DIR, env=env,
                       capture_output=True, text=True)
    if p.returncode != 0:
        print("[kernel] BUILD FAILED. Last lines:\n" + "\n".join((p.stdout + p.stderr).splitlines()[-25:]))
        return False
    print(f"[kernel] built in {timedelta(seconds=int(time.time() - t0))}")
    so = glob.glob(osp.join(KERNEL_DIR, "selective_scan_cuda_oflex*.so"))[0]
    os.makedirs(cache_dir, exist_ok=True); shutil.copy(so, cache_dir)
    return _try_import_kernel(KERNEL_DIR) is not None


KERNEL_OK = build_selective_scan()
if not KERNEL_OK and DEVICE == "cuda":
    print("!" * 100 + "\n!! The CUDA selective-scan kernel is NOT available: training will use a chunked pure-PyTorch scan,"
          "\n!! which gives the same maths but is many times slower. Fix the build before a full run.\n" + "!" * 100)

## TE-VMamba backbone (embedded from `model/backbones/`)
Five files are written verbatim, except for three compatibility patches that are clearly marked in the code
(`# [notebook patch]`):
1. `vmamba.py`: the `fvcore` import (only used by the FLOP counter) and the `mamba2` import (only used by the
   unused `m0` forward types; its triton files are not embedded) are made optional.
2. `csms6s.py`: if the CUDA extension is missing, the scan falls back to VMamba's own chunked PyTorch
   implementation (`selective_scan_easy` from `kernels/selective_scan/test_selective_scan_easy.py`, chunk 16)
   instead of the 1-step-at-a-time loop, which is unusably slow at 3,136 tokens.
3. `csm_triton.py`: `torch.cuda.device(...)` is skipped for CPU tensors, so the notebook can be smoke-tested on
   a CPU. Behaviour on GPU is unchanged.

**What these files implement** (see the report for details):
* `VSSM` / `VSSBlock` / `SS2D` (`v05_noz` forward): VMamba backbone with 4-direction cross-scan + selective scan.
* `LayerTokenPrune`: token importance = |Δ| (mean SSM step size), threshold θ = μ − α·σ with α = softplus(0.7)
  ≈ 1.10. It keeps tokens with importance ≥ θ, packs them into a smaller H × ⌈K/H⌉ grid, and later blocks run on
  fewer tokens.
* `LayerTokenMerge` + `FixedPairThresholdMerge`: pairs neighbouring tokens (alternating horizontal/vertical), with
  cosine similarity s and a per-image threshold θ = sigmoid(logit(0.5) + MLP(mean token)). Pairs with s > θ are
  averaged into one token.

In [ ]:
BB_DIR = osp.join(SRC_DIR, "tevmamba", "backbones")
os.makedirs(BB_DIR, exist_ok=True)
open(osp.join(SRC_DIR, "tevmamba", "__init__.py"), "w").write("")
open(osp.join(BB_DIR, "__init__.py"), "w").write("")
print("backbone sources ->", BB_DIR)

In [ ]:
%%writefile {SRC_DIR}/tevmamba/backbones/AdaptivePairMergeGate.py
# =========================================================
# AdaptivePairMergeGate.py  
# - even(a_k) <-> odd(b_k) 고정 페어만 고려 (O(P))
# - similarity: cosine (normalized dot)
# - theta: two-layer MLP + sigmoid
# - merge rule: sim > theta  
# - merge result: (a+b)/2
# - 구현은 GPU-friendly: prefix-sum + scatter/scatter_add (배치 loop 제거)
# =========================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Callable, Optional


def do_nothing(x, mode=None):
    return x


def _norm(x: torch.Tensor, dim=-1, eps=1e-12):
    return x / x.norm(dim=dim, keepdim=True).clamp_min(eps)


class FixedPairThresholdMerge(nn.Module):
    """
    논문 Eq.(6)-(7) 형태:
      s_k = cos(a_k, b_k)
      theta = sigmoid( MLP( mean(metric) ) )
      merge if s_k > theta
      z_k = (a_k + b_k)/2
    """

    def __init__(
        self,
        theta_min: float = 0.0,
        theta_max: float = 1.0,
        gate_hidden: int = 64,
        tau_init=0.1
    ):
        super().__init__()
        self.theta_min = float(theta_min)
        self.theta_max = float(theta_max)
        self.gate_hidden = int(gate_hidden)

        # lazy init (C 모름)
        self.gate_head: Optional[nn.Module] = None
        # tau_init은 (0,1)이어야 함
        eps = 1e-6
        t = float(max(eps, min(1.0 - eps, tau_init)))
        # sigmoid(theta_base) = tau_init 되도록 logit으로 초기화
        self.theta_base = nn.Parameter(torch.tensor(math.log(t / (1.0 - t)), dtype=torch.float32))


        # logging
        self.last_theta: Optional[float] = None
        self.keep_ratio_est: Optional[float] = None

    def _theta_from_gate(self, metric: torch.Tensor) -> torch.Tensor:
        """
        metric: (B, T, C)
        theta: (B,) in [theta_min, theta_max]
        """
        B, T, C = metric.shape
        if self.gate_head is None:
            self.gate_head = nn.Sequential(
                nn.Linear(C, self.gate_hidden),
                nn.GELU(),
                nn.Linear(self.gate_hidden, 1),
            ).to(metric.device)

        g = metric.mean(dim=1)  # (B, C)
        raw = self.gate_head(g).squeeze(-1)  # (B,)
        theta = torch.sigmoid(self.theta_base + raw)  # (B,) in (0,1)
        theta = theta.clamp(min=self.theta_min, max=self.theta_max)
        return theta

    def forward(
        self,
        metric: torch.Tensor,            # (B, T, C)
        class_token: bool = False,
        distill_token: bool = False,
    ):
        B, T, C = metric.shape
        protected = (1 if class_token else 0) + (1 if distill_token else 0)

        # (T-protected) 홀수면 마지막 제거 (페어 맞춤)
        if (T - protected) % 2 == 1:
            metric = metric[:, :-1, :]
            T = metric.size(1)

        P = (T - protected) // 2
        if P <= 0:
            return do_nothing, {"merge_mask": None, "ratio": 1.0, "theta": 0.0}

        # 1) cosine similarity
        with torch.no_grad():
            m = _norm(metric)
            a = m[:, protected:, :][:, ::2, :]   # (B,P,C)
            b = m[:, protected:, :][:, 1::2, :]  # (B,P,C)
            sim = (a * b).sum(dim=-1)            # (B,P)

        # 2) theta via gate (two-layer MLP + sigmoid)
        theta = self._theta_from_gate(metric)            # (B,)
        mask_bool = (sim > theta.unsqueeze(-1))          # (B,P) True=merge

        # logging
        with torch.no_grad():
            self.last_theta = float(theta.mean().item())
            # keep ratio(토큰 수 기준): merge면 1개, no-merge면 2개
            exp_keep_per_pair = 1.0 * mask_bool + 2.0 * (~mask_bool)
            self.keep_ratio_est = float((exp_keep_per_pair.float().mean().item() / 2.0))

        # 3) order-preserving merge (벡터화)
        def merge(x: torch.Tensor, mode: str = "mean") -> torch.Tensor:
            """
            x: (B, T, C)
            return: (B, T', C)  (batch마다 T' 다르면 max로 패딩)
            """
            Bx, Tx, Cx = x.shape
            if protected > 0:
                prot = x[:, :protected, :]
                x_ = x[:, protected:, :]
            else:
                prot = None
                x_ = x

            if (x_.shape[1] % 2) == 1:
                x_ = x_[:, :-1, :]

            src = x_[:, 0::2, :]   # (B,P,C)
            dst = x_[:, 1::2, :]   # (B,P,C)
            P_ = src.shape[1]

            mask = mask_bool.to(x_.device)
            if mask.shape[1] != P_:
                raise RuntimeError(f"mask P={mask.shape[1]} != src P={P_}")

            if mode == "mean":
                pair = (src + dst) * 0.5
            elif mode == "sum":
                pair = (src + dst)
            elif mode == "amax":
                pair = torch.maximum(src, dst)
            else:
                raise ValueError(f"Unsupported mode: {mode}")

            token1 = torch.where(mask.unsqueeze(-1), pair, src)  # (B,P,C)

            # out positions (prefix-sum)
            extra = (~mask).long()                      # unmerged면 1
            shift = torch.cumsum(extra, dim=1) - extra  # 이전 extra 개수
            base = torch.arange(P_, device=x_.device)[None, :]   # (1,P)
            pos1 = base + shift                         # (B,P)
            pos2 = pos1 + 1                             # (B,P) unmerged에서만 의미
            out_len = P_ + extra.sum(dim=1)             # (B,)
            out_max = int(out_len.max().item())

            out = x_.new_zeros(Bx, out_max, Cx)

            # (1) token1은 scatter_ 유지해도 안전 (pos1은 항상 0..out_len-1)
            out.scatter_(1, pos1.unsqueeze(-1).expand(-1, -1, Cx), token1)

            # (2) token2는 "unmerged만" 뽑아서 index_add (merge된 pos2는 아예 안 씀)
            unm = (~mask)  # (B,P) bool
            if unm.any():
                # out을 (B*out_max, C)로 펴서 1D index로 더하기
                out_flat = out.view(Bx * out_max, Cx)

                b = torch.arange(Bx, device=x_.device)[:, None].expand(Bx, P_)  # (B,P)
                idx2 = (b * out_max + pos2).masked_select(unm).long()           # (Nu,)
                val2 = dst.masked_select(unm.unsqueeze(-1)).view(-1, Cx)        # (Nu,C)

                # 안전 체크 (디버그용, 터지면 여기서 원인 바로 확정)
                # assert int(idx2.max().item()) < Bx * out_max and int(idx2.min().item()) >= 0

                out_flat.index_add_(0, idx2, val2)


            if prot is not None:
                out = torch.cat([prot, out], dim=1)
            return out

        return merge, {
            "merge_mask": mask_bool,
            "ratio": float(mask_bool.float().mean().item()),
            "theta": float(theta.mean().item()),
        }


# ===== helpers =====
def merge_wavg(merge: Callable, x: torch.Tensor, size: torch.Tensor = None):
    if size is None:
        size = torch.ones_like(x[..., 0, None])
    x = merge(x * size, mode="sum")
    size = merge(size, mode="sum")
    x = x / size
    return x, size


def merge_source(merge: Callable, x: torch.Tensor, source: torch.Tensor = None):
    if source is None:
        n, t, _ = x.shape
        source = torch.eye(t, device=x.device)[None, ...].expand(n, t, t)
    source = merge(source, mode="amax")
    return source


In [ ]:
%%writefile {SRC_DIR}/tevmamba/backbones/LayerTokenReducer.py
# =========================================================
# LayerTokenReducer.py  
# - PRUNE: |Δ| 기반, theta = mu - alpha*sigma (alpha learnable)
# - 배치 loop 제거: prefix-sum + scatter_add
# - MERGE: AdaptivePairMergeGate.FixedPairThresholdMerge 사용
# =========================================================

from typing import Optional, Tuple
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from .AdaptivePairMergeGate import FixedPairThresholdMerge


# -------------------------------
# 공통 유틸
# -------------------------------
def _flatten_bchw_to_btc(x: torch.Tensor) -> Tuple[torch.Tensor, int, int]:
    B, C, H, W = x.shape
    tok = x.permute(0, 2, 3, 1).reshape(B, H * W, C).contiguous()
    return tok, H, W


def _restore_tokens_to_grid(y_tok: torch.Tensor, H: int, W: int, prefer_axis: str = "W") -> torch.Tensor:
    B, K, C = y_tok.shape
    if K <= 0:
        return y_tok.new_zeros(B, C, H, W)

    if prefer_axis == "W":
        Hc, Wc = H, (K + H - 1) // H
    else:
        Wc, Hc = W, (K + W - 1) // W

    need = Hc * Wc - K
    if need > 0:
        y_tok = torch.cat([y_tok, y_tok.new_zeros(B, need, C)], dim=1)

    y = y_tok[:, :Hc * Wc, :].reshape(B, Hc, Wc, C).permute(0, 3, 1, 2).contiguous()
    return y


# =========================================================
# ① PRUNE MODULE (Δ 기반 adaptive pruning)
#   논문: theta = mu - alpha*sigma (alpha learnable)
# =========================================================
class LayerTokenPrune(nn.Module):
    def __init__(
        self,
        dim: int,
        norm_layer: nn.Module,
        layer_idx: int = 1,
        alpha_init: float = 0.7,   # 초기값
        min_keep_ratio: float = 0.05,
        alpha_positive: bool = True,  # alpha>0 강제 여부(권장)
    ):
        super().__init__()
        self.dim = dim
        self.norm = norm_layer(dim)
        self.layer_idx = layer_idx
        self.min_keep_ratio = float(min_keep_ratio)
        self.alpha_positive = bool(alpha_positive)

        # learnable alpha (논문 "learnable threshold parameter")
        self.alpha_raw = nn.Parameter(torch.tensor(float(alpha_init)))

        # logging/cache
        self.cached_delta = None
        self.keep_ratio_est = None
        self.last_theta = None

    def _alpha(self):
        if self.alpha_positive:
            return F.softplus(self.alpha_raw)  # >0
        return self.alpha_raw

    def forward(
        self,
        x_bchw: torch.Tensor,
        delta: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, dict]:
        if delta is None:
            raise ValueError("LayerTokenPrune requires delta input")

        B, C, H, W = x_bchw.shape
        L = H * W
        tok = x_bchw.permute(0, 2, 3, 1).reshape(B, L, C)

        # delta -> delta_tok: (B, L)
        if delta.dim() == 4:
            if delta.shape[-2:] == (H, W):
                delta_tok = delta.abs().mean(dim=1).reshape(B, L)
            elif delta.shape[-1] == L:
                delta_tok = delta.abs().mean(dim=(1, 2))
            else:
                raise ValueError(f"Unexpected 4D delta shape {delta.shape} for H,W={H,W}, L={L}")
        elif delta.dim() == 3:
            if delta.shape[-1] == L:
                delta_tok = delta.abs().mean(dim=1)
            else:
                raise ValueError(f"Unexpected 3D delta shape {delta.shape}, expected last dim L={L}")
        else:
            raise ValueError(f"Unexpected delta shape: {delta.shape}")

        if delta_tok.shape[1] != L:
            raise ValueError(f"delta_tok length {delta_tok.shape[1]} != L {L}. delta shape={delta.shape}")

        # importance = |Δ|
        imp = delta_tok.abs()  # (B,L)

        # theta = mu - alpha*sigma
        mu = imp.mean(dim=1, keepdim=True)
        sigma = imp.std(dim=1, keepdim=True)
        alpha = self._alpha()
        theta = mu - alpha * sigma

        mask = imp >= theta  # (B,L) bool

        # 최소 유지 토큰 수(안정성)
        min_keep = max(1, int(L * self.min_keep_ratio))

        keep_cnt = mask.sum(dim=1)         # (B,)
        need_fix = keep_cnt < min_keep     # (B,)

        if need_fix.any():
            # topk로 min_keep 확보 (벡터화)
            topk_idx = imp.topk(k=min_keep, dim=1).indices  # (B,min_keep)
            topk_mask = torch.zeros_like(mask, dtype=torch.bool, device=mask.device)
            topk_mask.scatter_(1, topk_idx, True)
            mask = torch.where(need_fix[:, None], topk_mask, mask)

        keep_cnt = mask.sum(dim=1)               # (B,)
        max_len = int(keep_cnt.max().item())     # python int

        # prefix-sum으로 압축 위치
        pos = mask.long().cumsum(dim=1) - 1      # (B,L)
        pos = pos.clamp_min(0)                  # keep 아닌 곳도 0

        # scatter_add로 compact (B, max_len, C)
        val = tok * mask.unsqueeze(-1).to(tok.dtype)       # keep 아니면 0
        idx = pos.unsqueeze(-1).expand(B, L, C).long()

        tok_new = tok.new_zeros(B, max_len, C)
        tok_new.scatter_add_(1, idx, val)

        # grid 복원 (논문상 토큰 수 줄었다는 효과를 유지하기 위해 "패딩 최소")
        K = tok_new.size(1)
        H_new = H
        W_new = math.ceil(K / H_new)

        need = H_new * W_new - K
        if need > 0:
            tok_new = torch.cat([tok_new, tok_new.new_zeros(B, need, C)], dim=1)

        y = tok_new.view(B, H_new, W_new, C).permute(0, 3, 1, 2).contiguous()

        if not torch.isfinite(y).all():
            y = torch.nan_to_num(y)

        self.keep_ratio_est = float((keep_cnt.float() / L).mean().item())
        self.last_theta = float(theta.mean().item())
        self.cached_delta = delta.detach()

        return y, {
            "keep_ratio": self.keep_ratio_est,
            "theta": self.last_theta,
            "alpha": float(alpha.detach().item()),
        }


# =========================================================
# ② MERGE MODULE (FixedPairThresholdMerge 기반)
# =========================================================
class LayerTokenMerge(nn.Module):
    def __init__(
        self,
        dim: int,
        norm_layer: nn.Module,
        layer_idx: int = 1,
        merge_gate_hidden: int = 64,
        tau_init=0.1
    ):
        super().__init__()
        self.layer_idx = layer_idx
        self.dim = dim
        self.norm = norm_layer(dim)

        self.merge_core = FixedPairThresholdMerge(
            gate_hidden=merge_gate_hidden,
            theta_min=0.0,
            theta_max=1.0,
            tau_init=tau_init,
        )
        self.last_merge_stats = None

    def _get_merge_fn(self, metric_tokens, hw, direction=None, **kwargs):
        B, T, C = metric_tokens.shape
        H, W = hw

        # pairing 가능한 영역만
        H_eff, W_eff = H - (H % 2), W - (W % 2)
        x2d = metric_tokens.view(B, H, W, C)[:, :H_eff, :W_eff, :]

        # 논문처럼 “고정 pairing”만. 방향 교대는 구현 선택(논문이 강제하진 않음)
        if direction is None:
            direction = "horizontal" if (self.layer_idx % 2 == 0) else "vertical"

        if direction in ("horizontal", "h"):
            a, b = x2d[:, :, 0::2, :], x2d[:, :, 1::2, :]
        else:
            a, b = x2d[:, 0::2, :, :], x2d[:, 1::2, :, :]

        a = a.reshape(B, -1, C)
        b = b.reshape(B, -1, C)
        inter = torch.stack([a, b], dim=2).reshape(B, -1, C)  # even/odd interleave

        merge_core_fn, stats = self.merge_core.forward(metric=inter, **kwargs)
        self.last_merge_stats = stats

        def wrapped(x):
            if isinstance(x, (tuple, list)):
                x = x[0]

            x2 = x.permute(0, 2, 3, 1)[:, :H_eff, :W_eff, :]

            if direction in ("horizontal", "h"):
                ax, bx = x2[:, :, 0::2, :], x2[:, :, 1::2, :]
            else:
                ax, bx = x2[:, 0::2, :, :], x2[:, 1::2, :, :]

            inter2 = torch.stack([ax.reshape(B, -1, C), bx.reshape(B, -1, C)], dim=2).reshape(B, -1, C)

            y_tok = merge_core_fn(inter2, mode="mean")  # 논문: (a+b)/2
            prefer = "W" if W >= H else "H"
            return _restore_tokens_to_grid(y_tok, H, W, prefer_axis=prefer)

        return wrapped

    def forward(self, x_bchw: torch.Tensor, **kwargs):
        if isinstance(x_bchw, (tuple, list)):
            x_bchw = x_bchw[0]

        tok2, H2, W2 = _flatten_bchw_to_btc(x_bchw)

        if tok2.shape[1] > 0:
            fn_merge = self._get_merge_fn(tok2, hw=(H2, W2), **kwargs)
            merged = fn_merge(x_bchw)
            return merged, self.last_merge_stats

        return x_bchw, None


In [ ]:
%%writefile {SRC_DIR}/tevmamba/backbones/csms6s.py
import time
import torch
import warnings


WITH_SELECTIVESCAN_OFLEX = True
WITH_SELECTIVESCAN_CORE = False
WITH_SELECTIVESCAN_MAMBA = True
try:
    import selective_scan_cuda_oflex
except ImportError:
    WITH_SELECTIVESCAN_OFLEX = False
    warnings.warn("Can not import selective_scan_cuda_oflex. This affects speed.")
    print("Can not import selective_scan_cuda_oflex. This affects speed.", flush=True)
try:
    import selective_scan_cuda_core
except ImportError:
    WITH_SELECTIVESCAN_CORE = False
try:
    import selective_scan_cuda
except ImportError:
    WITH_SELECTIVESCAN_MAMBA = False


def selective_scan_torch(
    u: torch.Tensor, # (B, K * C, L)
    delta: torch.Tensor, # (B, K * C, L)
    A: torch.Tensor, # (K * C, N)
    B: torch.Tensor, # (B, K, N, L)
    C: torch.Tensor, # (B, K, N, L)
    D: torch.Tensor = None, # (K * C)
    delta_bias: torch.Tensor = None, # (K * C)
    delta_softplus=True, 
    oflex=True, 
    *args,
    **kwargs
):
    dtype_in = u.dtype
    Batch, K, N, L = B.shape
    KCdim = u.shape[1]
    Cdim = int(KCdim / K)
    assert u.shape == (Batch, KCdim, L)
    assert delta.shape == (Batch, KCdim, L)
    assert A.shape == (KCdim, N)
    assert C.shape == B.shape

    if delta_bias is not None:
        delta = delta + delta_bias[..., None]
    if delta_softplus:
        delta = torch.nn.functional.softplus(delta)
            
    u, delta, A, B, C = u.float(), delta.float(), A.float(), B.float(), C.float()
    B = B.view(Batch, K, 1, N, L).repeat(1, 1, Cdim, 1, 1).view(Batch, KCdim, N, L)
    C = C.view(Batch, K, 1, N, L).repeat(1, 1, Cdim, 1, 1).view(Batch, KCdim, N, L)
    deltaA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A))
    deltaB_u = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)
    
    if True:
        x = A.new_zeros((Batch, KCdim, N))
        ys = []
        for i in range(L):
            x = deltaA[:, :, i, :] * x + deltaB_u[:, :, i, :]
            y = torch.einsum('bdn,bdn->bd', x, C[:, :, :, i])
            ys.append(y)
        y = torch.stack(ys, dim=2) # (B, C, L)
    
    out = y if D is None else y + u * D.unsqueeze(-1)
    return out if oflex else out.to(dtype=dtype_in)


class SelectiveScanCuda(torch.autograd.Function):
    @staticmethod
    @torch.cuda.amp.custom_fwd
    def forward(ctx, u, delta, A, B, C, D=None, delta_bias=None, delta_softplus=False, oflex=True, backend=None):
        ctx.delta_softplus = delta_softplus
        backend = "oflex" if WITH_SELECTIVESCAN_OFLEX and (backend is None) else backend
        backend = "core" if WITH_SELECTIVESCAN_CORE and (backend is None) else backend
        backend = "mamba" if WITH_SELECTIVESCAN_MAMBA and (backend is None) else backend
        ctx.backend = backend
        if backend == "oflex":
            out, x, *rest = selective_scan_cuda_oflex.fwd(u, delta, A, B, C, D, delta_bias, delta_softplus, 1, oflex)
        elif backend == "core":
            out, x, *rest = selective_scan_cuda_core.fwd(u, delta, A, B, C, D, delta_bias, delta_softplus, 1)
        elif backend == "mamba":
            out, x, *rest = selective_scan_cuda.fwd(u, delta, A, B, C, D, None, delta_bias, delta_softplus)
        ctx.save_for_backward(u, delta, A, B, C, D, delta_bias, x)
        return out
    
    @staticmethod
    @torch.cuda.amp.custom_bwd
    def backward(ctx, dout, *args):
        u, delta, A, B, C, D, delta_bias, x = ctx.saved_tensors
        backend = ctx.backend
        if dout.stride(-1) != 1:
            dout = dout.contiguous()
        if backend == "oflex":
            du, ddelta, dA, dB, dC, dD, ddelta_bias, *rest = selective_scan_cuda_oflex.bwd(
                u, delta, A, B, C, D, delta_bias, dout, x, ctx.delta_softplus, 1
            )
        elif backend == "core":
            du, ddelta, dA, dB, dC, dD, ddelta_bias, *rest = selective_scan_cuda_core.bwd(
                u, delta, A, B, C, D, delta_bias, dout, x, ctx.delta_softplus, 1
            )
        elif backend == "mamba":
            du, ddelta, dA, dB, dC, dD, ddelta_bias, *rest = selective_scan_cuda.bwd(
                u, delta, A, B, C, D, None, delta_bias, dout, x, None, None, ctx.delta_softplus,
                False
            )
        return du, ddelta, dA, dB, dC, dD, ddelta_bias, None, None, None


def selective_scan_fn(
    u: torch.Tensor, # (B, K * C, L)
    delta: torch.Tensor, # (B, K * C, L)
    A: torch.Tensor, # (K * C, N)
    B: torch.Tensor, # (B, K, N, L)
    C: torch.Tensor, # (B, K, N, L)
    D: torch.Tensor = None, # (K * C)
    delta_bias: torch.Tensor = None, # (K * C)
    delta_softplus=True, 
    oflex=True,
    backend=None,
):
    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    if backend != "torch" and not WITH_CUDA:  # [notebook patch] fast chunked fallback instead of the per-step loop
        return selective_scan_easy_fallback(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex)
    fn = selective_scan_torch if backend == "torch" or (not WITH_CUDA) else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)


# [notebook patch] chunked pure-PyTorch selective scan, used only when no CUDA extension could be imported.
# Copied from kernels/selective_scan/test_selective_scan_easy.py::selective_scan_easy (VMamba authors); code unchanged, comments removed.
def selective_scan_easy(us, dts, As, Bs, Cs, Ds, delta_bias=None, delta_softplus=False, return_last_state=False, chunksize=64):
    def selective_scan_chunk(us, dts, As, Bs, Cs, hprefix):
        ts = dts.cumsum(dim=0)
        Ats = torch.einsum("gdn,lbgd->lbgdn", As, ts).exp()
        scale = 1
        rAts = Ats / scale
        duts = dts * us
        dtBus = torch.einsum("lbgd,lbgn->lbgdn", duts, Bs)
        hs_tmp = rAts * (dtBus / rAts).cumsum(dim=0)
        hs = hs_tmp + Ats * hprefix.unsqueeze(0)
        ys = torch.einsum("lbgn,lbgdn->lbgd", Cs, hs)
        return ys, hs

    dtype = torch.float32
    inp_dtype = us.dtype
    has_D = Ds is not None
    if chunksize < 1:
        chunksize = Bs.shape[-1]
    dts = dts.to(dtype)
    if delta_bias is not None:
        dts = dts + delta_bias.view(1, -1, 1).to(dtype)
    if delta_softplus:
        dts = torch.nn.functional.softplus(dts)
    if len(Bs.shape) == 3:
        Bs = Bs.unsqueeze(1)
    if len(Cs.shape) == 3:
        Cs = Cs.unsqueeze(1)
    B, G, N, L = Bs.shape
    us = us.view(B, G, -1, L).permute(3, 0, 1, 2).to(dtype)
    dts = dts.view(B, G, -1, L).permute(3, 0, 1, 2).to(dtype)
    As = As.view(G, -1, N).to(dtype)
    Bs = Bs.permute(3, 0, 1, 2).to(dtype)
    Cs = Cs.permute(3, 0, 1, 2).to(dtype)
    Ds = Ds.view(G, -1).to(dtype) if has_D else None
    D = As.shape[1]
    oys = []
    hprefix = us.new_zeros((B, G, D, N), dtype=dtype)
    for i in range(0, L, chunksize):
        ys, hs = selective_scan_chunk(us[i:i + chunksize], dts[i:i + chunksize], As, Bs[i:i + chunksize], Cs[i:i + chunksize], hprefix)
        oys.append(ys)
        hprefix = hs[-1]
    oys = torch.cat(oys, dim=0)
    if has_D:
        oys = oys + Ds * us
    oys = oys.permute(1, 2, 3, 0).view(B, -1, L)
    return oys.to(inp_dtype) if not return_last_state else (oys.to(inp_dtype), hprefix.view(B, G * D, N).float())


def selective_scan_easy_fallback(u, delta, A, B, C, D=None, delta_bias=None, delta_softplus=True, oflex=True):
    out = selective_scan_easy(u, delta, A, B, C, D, delta_bias, delta_softplus, chunksize=16)
    return out.float() if oflex else out   # oflex=True: the CUDA kernel returns fp32


# fvcore flops =======================================
def print_jit_input_names(inputs):
    print("input params: ", end=" ", flush=True)
    try: 
        for i in range(10):
            print(inputs[i].debugName(), end=" ", flush=True)
    except Exception as e:
        pass
    print("", flush=True)

def flops_selective_scan_fn(B=1, L=256, D=768, N=16, with_D=True, with_Z=False, with_complex=False):
    """
    u: r(B D L)
    delta: r(B D L)
    A: r(D N)
    B: r(B N L)
    C: r(B N L)
    D: r(D)
    z: r(B D L)
    delta_bias: r(D), fp32
    
    ignores:
        [.float(), +, .softplus, .shape, new_zeros, repeat, stack, to(dtype), silu] 
    """
    assert not with_complex 
    # https://github.com/state-spaces/mamba/issues/110
    flops = 9 * B * L * D * N
    if with_D:
        flops += B * D * L
    if with_Z:
        flops += B * D * L    
    return flops

# this is only for selective_scan_ref...
def flops_selective_scan_ref(B=1, L=256, D=768, N=16, with_D=True, with_Z=False, with_Group=True, with_complex=False):
    """
    u: r(B D L)
    delta: r(B D L)
    A: r(D N)
    B: r(B N L)
    C: r(B N L)
    D: r(D)
    z: r(B D L)
    delta_bias: r(D), fp32
    
    ignores:
        [.float(), +, .softplus, .shape, new_zeros, repeat, stack, to(dtype), silu] 
    """
    import numpy as np
    
    # fvcore.nn.jit_handles
    def get_flops_einsum(input_shapes, equation):
        np_arrs = [np.zeros(s) for s in input_shapes]
        optim = np.einsum_path(equation, *np_arrs, optimize="optimal")[1]
        for line in optim.split("\n"):
            if "optimized flop" in line.lower():
                # divided by 2 because we count MAC (multiply-add counted as one flop)
                flop = float(np.floor(float(line.split(":")[-1]) / 2))
                return flop
    

    assert not with_complex

    flops = 0 # below code flops = 0

    flops += get_flops_einsum([[B, D, L], [D, N]], "bdl,dn->bdln")
    if with_Group:
        flops += get_flops_einsum([[B, D, L], [B, N, L], [B, D, L]], "bdl,bnl,bdl->bdln")
    else:
        flops += get_flops_einsum([[B, D, L], [B, D, N, L], [B, D, L]], "bdl,bdnl,bdl->bdln")
  
    in_for_flops = B * D * N   
    if with_Group:
        in_for_flops += get_flops_einsum([[B, D, N], [B, D, N]], "bdn,bdn->bd")
    else:
        in_for_flops += get_flops_einsum([[B, D, N], [B, N]], "bdn,bn->bd")
    flops += L * in_for_flops 
    if with_D:
        flops += B * D * L
    if with_Z:
        flops += B * D * L  
    return flops

def selective_scan_flop_jit(inputs, outputs, backend="prefixsum", verbose=True):
    if verbose:
        print_jit_input_names(inputs)
    flops_fn = flops_selective_scan_ref if backend == "naive" else flops_selective_scan_fn
    B, D, L = inputs[0].type().sizes()
    N = inputs[2].type().sizes()[1]
    flops = flops_fn(B=B, L=L, D=D, N=N, with_D=True, with_Z=False)
    return flops


if __name__ == "__main__":
    def params(B, K, C, N, L, device = torch.device("cuda"), itype = torch.float):
        As = (-0.5 * torch.rand(K * C, N, device=device, dtype=torch.float32)).requires_grad_()
        Bs = torch.randn((B, K, N, L), device=device, dtype=itype).requires_grad_()
        Cs = torch.randn((B, K, N, L), device=device, dtype=itype).requires_grad_()
        Ds = torch.randn((K * C), device=device, dtype=torch.float32).requires_grad_()
        u = torch.randn((B, K * C, L), device=device, dtype=itype).requires_grad_()
        delta = (0.5 * torch.rand((B, K * C, L),  device=device, dtype=itype)).requires_grad_()
        delta_bias = (0.5 * torch.rand((K * C), device=device, dtype=torch.float32)).requires_grad_()
        return u, delta, As, Bs, Cs, Ds, delta_bias

    def bench(func, xs, Warmup=30, NTimes=20):
        import time
        torch.cuda.synchronize()
        for r in range(Warmup):
            for x in xs:
                func(x)
        torch.cuda.synchronize()
        tim0 = time.time()
        for r in range(NTimes):
            for x in xs:
                func(x)
        torch.cuda.synchronize()
        return (time.time() - tim0) / NTimes

    def check():
        u, delta, As, Bs, Cs, Ds, delta_bias = params(1, 4, 16, 8, 512, itype=torch.float16)
        u1, delta1, As1, Bs1, Cs1, Ds1, delta_bias1 = [x.clone().detach().requires_grad_() for x in [u, delta, As, Bs, Cs, Ds, delta_bias]]
        
        # out_ref = selective_scan_fn(u, delta, As, Bs, Cs, Ds, delta_bias, True, backend="torch")
        out = selective_scan_fn(u1, delta1, As1, Bs1, Cs1, Ds1, delta_bias1, True, backend="oflex")
        out_ref = selective_scan_fn(u, delta, As, Bs, Cs, Ds, delta_bias, True, backend="mamba")
        print((out_ref - out).abs().max())
        out.sum().backward()
        out_ref.sum().backward()
        for x, y in zip([u, As, Bs, Cs, Ds, delta, delta_bias], [u1, As1, Bs1, Cs1, Ds1, delta1, delta_bias1]):
            print((x.grad - y.grad).abs().max())

        u, delta, As, Bs, Cs, Ds, delta_bias = params(128, 4, 96, 8, 56 * 56)
        print(bench(lambda x: selective_scan_fn(x[0], x[1], x[2], x[3], x[4], x[5], x[6], True, backend="oflex"), [(u, delta, As, Bs, Cs, Ds, delta_bias),]))
        print(bench(lambda x: selective_scan_fn(x[0], x[1], x[2], x[3], x[4], x[5], x[6], True, backend="mamba"), [(u, delta, As, Bs, Cs, Ds, delta_bias),]))
        print(bench(lambda x: selective_scan_fn(x[0], x[1], x[2], x[3], x[4], x[5], x[6], True, backend="torch"), [(u, delta, As, Bs, Cs, Ds, delta_bias),]))

    check()


In [ ]:
%%writefile {SRC_DIR}/tevmamba/backbones/csm_triton.py
import torch
import warnings
import contextlib  # [notebook patch]

WITH_TRITON = True
# WITH_TRITON = False
try:
    import triton
    import triton.language as tl
except:
    WITH_TRITON = False
    warnings.warn("Triton not installed, fall back to pytorch implements.")

# to make sure cached_property can be loaded for triton
if WITH_TRITON:
    try:
        from functools import cached_property
    except:
        warnings.warn("if you are using py37, add this line to functools.py: "
            "cached_property = lambda func: property(lru_cache()(func))")

# torch implementation ========================================
def cross_scan_fwd(x: torch.Tensor, in_channel_first=True, out_channel_first=True, scans=0):
    if in_channel_first:
        B, C, H, W = x.shape
        if scans == 0:
            y = x.new_empty((B, 4, C, H * W))
            y[:, 0, :, :] = x.flatten(2, 3)
            y[:, 1, :, :] = x.transpose(dim0=2, dim1=3).flatten(2, 3)
            y[:, 2:4, :, :] = torch.flip(y[:, 0:2, :, :], dims=[-1])
        elif scans == 1:
            y = x.view(B, 1, C, H * W).repeat(1, 4, 1, 1)
        elif scans == 2:
            y = x.view(B, 1, C, H * W).repeat(1, 2, 1, 1)
            y = torch.cat([y, y.flip(dims=[-1])], dim=1)
    else:
        B, H, W, C = x.shape
        if scans == 0:
            y = x.new_empty((B, H * W, 4, C))
            y[:, :, 0, :] = x.flatten(1, 2)
            y[:, :, 1, :] = x.transpose(dim0=1, dim1=2).flatten(1, 2)
            y[:, :, 2:4, :] = torch.flip(y[:, :, 0:2, :], dims=[1])
        elif scans == 1:
            y = x.view(B, H * W, 1, C).repeat(1, 1, 4, 1)
        elif scans == 2:
            y = x.view(B, H * W, 1, C).repeat(1, 1, 2, 1)
            y = torch.cat([y, y.flip(dims=[1])], dim=2)

    if in_channel_first and (not out_channel_first):
        y = y.permute(0, 3, 1, 2).contiguous()
    elif (not in_channel_first) and out_channel_first:
        y = y.permute(0, 2, 3, 1).contiguous()

    return y


def cross_merge_fwd(y: torch.Tensor, in_channel_first=True, out_channel_first=True, scans=0):
    if out_channel_first:
        B, K, D, H, W = y.shape
        y = y.view(B, K, D, -1)
        if scans == 0:
            y = y[:, 0:2] + y[:, 2:4].flip(dims=[-1]).view(B, 2, D, -1)
            y = y[:, 0] + y[:, 1].view(B, -1, W, H).transpose(dim0=2, dim1=3).contiguous().view(B, D, -1)
        elif scans == 1:
            y = y.sum(1)
        elif scans == 2:
            y = y[:, 0:2] + y[:, 2:4].flip(dims=[-1]).view(B, 2, D, -1)
            y = y.sum(1)
    else:
        B, H, W, K, D = y.shape
        y = y.view(B, -1, K, D)
        if scans == 0:
            y = y[:, :, 0:2] + y[:, :, 2:4].flip(dims=[1]).view(B, -1, 2, D)
            y = y[:, :, 0] + y[:, :, 1].view(B, W, H, -1).transpose(dim0=1, dim1=2).contiguous().view(B, -1, D)        
        elif scans == 1:
            y = y.sum(2)
        elif scans == 2:
            y = y[:, :, 0:2] + y[:, :, 2:4].flip(dims=[1]).view(B, -1, 2, D)
            y = y.sum(2)

    if in_channel_first and (not out_channel_first):
        y = y.permute(0, 2, 1).contiguous()
    elif (not in_channel_first) and out_channel_first:
        y = y.permute(0, 2, 1).contiguous()
    
    return y


def cross_scan1b1_fwd(x: torch.Tensor, in_channel_first=True, out_channel_first=True, scans=0):
    if in_channel_first:
        B, _, C, H, W = x.shape
        if scans == 0:
            y = torch.stack([
                x[:, 0].flatten(2, 3),
                x[:, 1].transpose(dim0=2, dim1=3).flatten(2, 3),
                torch.flip(x[:, 2].flatten(2, 3), dims=[-1]),
                torch.flip(x[:, 3].transpose(dim0=2, dim1=3).flatten(2, 3), dims=[-1]),
            ], dim=1)
        elif scans == 1:
            y = x.flatten(2, 3)
        elif scans == 2:
            y = torch.stack([
                x[:, 0].flatten(2, 3),
                x[:, 1].flatten(2, 3),
                torch.flip(x[:, 2].flatten(2, 3), dims=[-1]),
                torch.flip(x[:, 3].flatten(2, 3), dims=[-1]),
            ], dim=1)
    else:
        B, H, W, _, C = x.shape
        if scans == 0:
            y = torch.stack([
                x[:, :, :, 0].flatten(1, 2),
                x[:, :, :, 1].transpose(dim0=1, dim1=2).flatten(1, 2),
                torch.flip(x[:, :, :, 2].flatten(1, 2), dims=[1]),
                torch.flip(x[:, :, :, 3].transpose(dim0=1, dim1=2).flatten(1, 2), dims=[1]),
            ], dim=2)
        elif scans == 1:
            y = x.flatten(1, 2)
        elif scans == 2:
            y = torch.stack([
                x[:, 0].flatten(1, 2),
                x[:, 1].flatten(1, 2),
                torch.flip(x[:, 2].flatten(1, 2), dims=[-1]),
                torch.flip(x[:, 3].flatten(1, 2), dims=[-1]),
            ], dim=2)

    if in_channel_first and (not out_channel_first):
        y = y.permute(0, 3, 1, 2).contiguous()
    elif (not in_channel_first) and out_channel_first:
        y = y.permute(0, 2, 3, 1).contiguous()

    return y


def cross_merge1b1_fwd(y: torch.Tensor, in_channel_first=True, out_channel_first=True, scans=0):
    if out_channel_first:
        B, K, D, H, W = y.shape
        y = y.view(B, K, D, -1)
        if scans == 0:
            y = torch.stack([
                y[:, 0],
                y[:, 1].view(B, -1, W, H).transpose(dim0=2, dim1=3).flatten(2, 3),
                torch.flip(y[:, 2], dims=[-1]),
                torch.flip(y[:, 3].view(B, -1, W, H).transpose(dim0=2, dim1=3).flatten(2, 3), dims=[-1]),
            ], dim=1)
        elif scans == 1:
            y = y
        elif scans == 2:
            y = torch.stack([
                y[:, 0],
                y[:, 1],
                torch.flip(y[:, 2], dims=[-1]),
                torch.flip(y[:, 3], dims=[-1]),
            ], dim=1)
    else:
        B, H, W, K, D = y.shape
        y = y.view(B, -1, K, D)
        if scans == 0:
            y = torch.stack([
                y[:, :, 0],
                y[:, :, 1].view(B, W, H, -1).transpose(dim0=1, dim1=2).flatten(1, 2),
                torch.flip(y[:, :, 2], dims=[1]),
                torch.flip(y[:, :, 3].view(B, W, H, -1).transpose(dim0=1, dim1=2).flatten(1, 2), dims=[1]),
            ], dim=2)
        elif scans == 1:
            y = y
        elif scans == 2:
            y = torch.stack([
                y[:, :, 0],
                y[:, :, 1],
                torch.flip(y[:, :, 2], dims=[1]),
                torch.flip(y[:, :, 3], dims=[1]),
            ], dim=2)

    if out_channel_first and (not in_channel_first):
        y = y.permute(0, 3, 1, 2).contiguous()
    elif (not out_channel_first) and in_channel_first:
        y = y.permute(0, 2, 3, 1).contiguous()

    return y


class CrossScanF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0):
        # x: (B, C, H, W) | (B, H, W, C) | (B, 4, C, H, W) | (B, H, W, 4, C)
        # y: (B, 4, C, H * W) | (B, H * W, 4, C)
        ctx.in_channel_first = in_channel_first
        ctx.out_channel_first = out_channel_first
        ctx.one_by_one = one_by_one
        ctx.scans = scans

        if one_by_one:
            B, K, C, H, W = x.shape
            if not in_channel_first:
                B, H, W, K, C = x.shape
        else:
            B, C, H, W = x.shape
            if not in_channel_first:
                B, H, W, C = x.shape
        ctx.shape = (B, C, H, W)

        _fn = cross_scan1b1_fwd if one_by_one else cross_scan_fwd
        y = _fn(x, in_channel_first, out_channel_first, scans)

        return y
    
    @staticmethod
    def backward(ctx, ys: torch.Tensor):
        # out: (b, k, d, l)
        in_channel_first = ctx.in_channel_first
        out_channel_first = ctx.out_channel_first
        one_by_one = ctx.one_by_one
        scans = ctx.scans
        B, C, H, W = ctx.shape

        ys = ys.view(B, -1, C, H, W) if out_channel_first else ys.view(B, H, W, -1, C)
        _fn = cross_merge1b1_fwd if one_by_one else cross_merge_fwd
        y = _fn(ys, in_channel_first, out_channel_first, scans)
        
        if one_by_one:
            y = y.view(B, 4, -1, H, W) if in_channel_first else y.view(B, H, W, 4, -1)
        else:
            y = y.view(B, -1, H, W) if in_channel_first else y.view(B, H, W, -1)

        return y, None, None, None, None


class CrossMergeF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, ys: torch.Tensor, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0):
        # x: (B, C, H, W) | (B, H, W, C) | (B, 4, C, H, W) | (B, H, W, 4, C)
        # y: (B, 4, C, H * W) | (B, H * W, 4, C)
        ctx.in_channel_first = in_channel_first
        ctx.out_channel_first = out_channel_first
        ctx.one_by_one = one_by_one
        ctx.scans = scans

        B, K, C, H, W = ys.shape
        if not out_channel_first:
            B, H, W, K, C = ys.shape
        ctx.shape = (B, C, H, W)
        
        _fn = cross_merge1b1_fwd if one_by_one else cross_merge_fwd
        y = _fn(ys, in_channel_first, out_channel_first, scans)

        return y
    
    @staticmethod
    def backward(ctx, x: torch.Tensor):
        # B, D, L = x.shape
        # out: (b, k, d, h, w)
        in_channel_first = ctx.in_channel_first
        out_channel_first = ctx.out_channel_first
        one_by_one = ctx.one_by_one
        scans = ctx.scans
        B, C, H, W = ctx.shape
    
        if not one_by_one:
            if in_channel_first:
                x = x.view(B, C, H, W)
            else:
                x = x.view(B, H, W, C)
        else:
            if in_channel_first:
                x = x.view(B, 4, C, H, W)
            else:
                x = x.view(B, H, W, 4, C)   
                     
        _fn = cross_scan1b1_fwd if one_by_one else cross_scan_fwd
        x = _fn(x, in_channel_first, out_channel_first, scans)
        x = x.view(B, 4, C, H, W) if out_channel_first else x.view(B, H, W, 4, C)
    
        return x, None, None, None, None


# triton implements ========================================

@triton.jit
def triton_cross_scan_flex(
    x: tl.tensor, # (B, C, H, W) | (B, H, W, C) | (B, 4, C, H, W) | (B, H, W, 4, C)
    y: tl.tensor, # (B, 4, C, H, W) | (B, H, W, 4, C)
    x_layout: tl.constexpr,
    y_layout: tl.constexpr,
    operation: tl.constexpr,
    onebyone: tl.constexpr,
    scans: tl.constexpr,
    BC: tl.constexpr,
    BH: tl.constexpr,
    BW: tl.constexpr,
    DC: tl.constexpr,
    DH: tl.constexpr,
    DW: tl.constexpr,
    NH: tl.constexpr,
    NW: tl.constexpr,
):
    # x_layout = 0
    # y_layout = 1 # 0 BCHW, 1 BHWC
    # operation = 0 # 0 scan, 1 merge
    # onebyone = 0 # 0 false, 1 true
    # scans = 0 # 0 cross scan, 1 unidirectional, 2 bidirectional

    i_hw, i_c, i_b = tl.program_id(0), tl.program_id(1), tl.program_id(2)
    i_h, i_w = (i_hw // NW), (i_hw % NW)
    _mask_h = (i_h * BH + tl.arange(0, BH)) < DH
    _mask_w = (i_w * BW + tl.arange(0, BW)) < DW
    _mask_hw = _mask_h[:, None] & _mask_w[None, :]
    _for_C = min(DC - i_c * BC, BC)

    pos_h = (i_h * BH + tl.arange(0, BH)[:, None])
    pos_w = (i_w * BW + tl.arange(0, BW)[None, :])
    neg_h = (DH - i_h * BH - 1 - tl.arange(0, BH)[:, None])
    neg_w = (DW - i_w * BW - 1 - tl.arange(0, BW)[None, :])
    if scans == 0:
        # none; trans; flip; trans + flip;
        HWRoute0 = pos_h * DW + pos_w
        HWRoute1 = pos_w * DH + pos_h # trans
        HWRoute2 = neg_h * DW + neg_w # flip
        HWRoute3 = neg_w * DH + neg_h # trans + flip
    elif scans == 1:
        # none; none; none; none;
        HWRoute0 = pos_h * DW + pos_w
        HWRoute1 = HWRoute0
        HWRoute2 = HWRoute0
        HWRoute3 = HWRoute0
    elif scans == 2:
        # none; none; flip; flip;
        HWRoute0 = pos_h * DW + pos_w
        HWRoute1 = HWRoute0
        HWRoute2 = neg_h * DW + neg_w # flip
        HWRoute3 = HWRoute2      

    _tmp1 = DC * DH * DW

    y_ptr_base = y + i_b * 4 * _tmp1 + (i_c * BC * DH * DW if y_layout == 0 else i_c * BC)
    if y_layout == 0:
        p_y1 = y_ptr_base + HWRoute0
        p_y2 = y_ptr_base + _tmp1 + HWRoute1
        p_y3 = y_ptr_base + 2 * _tmp1 + HWRoute2
        p_y4 = y_ptr_base + 3 * _tmp1 + HWRoute3
    else:
        p_y1 = y_ptr_base + HWRoute0 * 4 * DC
        p_y2 = y_ptr_base + DC + HWRoute1 * 4 * DC
        p_y3 = y_ptr_base + 2 * DC + HWRoute2 * 4 * DC
        p_y4 = y_ptr_base + 3 * DC + HWRoute3 * 4 * DC       
    
    if onebyone == 0:
        x_ptr_base = x + i_b * _tmp1 + (i_c * BC * DH * DW if x_layout == 0 else i_c * BC)
        if x_layout == 0:
            p_x = x_ptr_base + HWRoute0
        else:
            p_x = x_ptr_base + HWRoute0 * DC

        if operation == 0:
            for idxc in range(_for_C):
                _idx_x = idxc * DH * DW if x_layout == 0 else idxc
                _idx_y = idxc * DH * DW if y_layout == 0 else idxc
                _x = tl.load(p_x + _idx_x, mask=_mask_hw)
                tl.store(p_y1 + _idx_y, _x, mask=_mask_hw)
                tl.store(p_y2 + _idx_y, _x, mask=_mask_hw)
                tl.store(p_y3 + _idx_y, _x, mask=_mask_hw)
                tl.store(p_y4 + _idx_y, _x, mask=_mask_hw)
        elif operation == 1:
            for idxc in range(_for_C):
                _idx_x = idxc * DH * DW if x_layout == 0 else idxc
                _idx_y = idxc * DH * DW if y_layout == 0 else idxc
                _y1 = tl.load(p_y1 + _idx_y, mask=_mask_hw)
                _y2 = tl.load(p_y2 + _idx_y, mask=_mask_hw)
                _y3 = tl.load(p_y3 + _idx_y, mask=_mask_hw)
                _y4 = tl.load(p_y4 + _idx_y, mask=_mask_hw)
                tl.store(p_x + _idx_x, _y1 + _y2 + _y3 + _y4, mask=_mask_hw)

    else:
        x_ptr_base = x + i_b * 4 * _tmp1 + (i_c * BC * DH * DW if x_layout == 0 else i_c * BC)
        if x_layout == 0:
            p_x1 = x_ptr_base + HWRoute0
            p_x2 = p_x1 + _tmp1
            p_x3 = p_x2 + _tmp1
            p_x4 = p_x3 + _tmp1  
        else:
            p_x1 = x_ptr_base + HWRoute0 * 4 * DC
            p_x2 = p_x1 + DC
            p_x3 = p_x2 + DC
            p_x4 = p_x3 + DC        
    
        if operation == 0:
            for idxc in range(_for_C):
                _idx_x = idxc * DH * DW if x_layout == 0 else idxc
                _idx_y = idxc * DH * DW if y_layout == 0 else idxc
                tl.store(p_y1 + _idx_y, tl.load(p_x1 + _idx_x, mask=_mask_hw), mask=_mask_hw)
                tl.store(p_y2 + _idx_y, tl.load(p_x2 + _idx_x, mask=_mask_hw), mask=_mask_hw)
                tl.store(p_y3 + _idx_y, tl.load(p_x3 + _idx_x, mask=_mask_hw), mask=_mask_hw)
                tl.store(p_y4 + _idx_y, tl.load(p_x4 + _idx_x, mask=_mask_hw), mask=_mask_hw)
        else:
            for idxc in range(_for_C):
                _idx_x = idxc * DH * DW if x_layout == 0 else idxc
                _idx_y = idxc * DH * DW if y_layout == 0 else idxc
                tl.store(p_x1 + _idx_x, tl.load(p_y1 + _idx_y), mask=_mask_hw)
                tl.store(p_x2 + _idx_x, tl.load(p_y2 + _idx_y), mask=_mask_hw)
                tl.store(p_x3 + _idx_x, tl.load(p_y3 + _idx_y), mask=_mask_hw)
                tl.store(p_x4 + _idx_x, tl.load(p_y4 + _idx_y), mask=_mask_hw)


class CrossScanTritonF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0):
        if one_by_one:
            if in_channel_first:
                B, _, C, H, W = x.shape
            else:
                B, H, W, _, C = x.shape
        else:
            if in_channel_first:
                B, C, H, W = x.shape
            else:
                B, H, W, C = x.shape
        B, C, H, W = int(B), int(C), int(H), int(W)
        BC, BH, BW = 1, 32, 32
        NH, NW, NC = triton.cdiv(H, BH), triton.cdiv(W, BW), triton.cdiv(C, BC)
        
        ctx.in_channel_first = in_channel_first
        ctx.out_channel_first = out_channel_first
        ctx.one_by_one = one_by_one
        ctx.scans = scans
        ctx.shape = (B, C, H, W)
        ctx.triton_shape = (BC, BH, BW, NC, NH, NW)

        y = x.new_empty((B, 4, C, H * W)) if out_channel_first else x.new_empty((B, H * W, 4, C))
        triton_cross_scan_flex[(NH * NW, NC, B)](
            x.contiguous(), y, 
            (0 if in_channel_first else 1), (0 if out_channel_first else 1), 0, (0 if not one_by_one else 1), scans, 
            BC, BH, BW, C, H, W, NH, NW
        )
        return y
        
    @staticmethod
    def backward(ctx, y: torch.Tensor):
        in_channel_first = ctx.in_channel_first
        out_channel_first = ctx.out_channel_first
        one_by_one = ctx.one_by_one
        scans = ctx.scans
        B, C, H, W = ctx.shape
        BC, BH, BW, NC, NH, NW = ctx.triton_shape
        if one_by_one:
            x = y.new_empty((B, 4, C, H, W)) if in_channel_first else y.new_empty((B, H, W, 4, C))
        else:
            x = y.new_empty((B, C, H, W)) if in_channel_first else y.new_empty((B, H, W, C))
        
        triton_cross_scan_flex[(NH * NW, NC, B)](
            x, y.contiguous(), 
            (0 if in_channel_first else 1), (0 if out_channel_first else 1), 1, (0 if not one_by_one else 1), scans,
            BC, BH, BW, C, H, W, NH, NW
        )
        return x, None, None, None, None


class CrossMergeTritonF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, y: torch.Tensor, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0):
        if out_channel_first:
            B, _, C, H, W = y.shape
        else:
            B, H, W, _, C = y.shape
        B, C, H, W = int(B), int(C), int(H), int(W)
        BC, BH, BW = 1, 32, 32
        NH, NW, NC = triton.cdiv(H, BH), triton.cdiv(W, BW), triton.cdiv(C, BC)
        ctx.in_channel_first = in_channel_first
        ctx.out_channel_first = out_channel_first
        ctx.one_by_one = one_by_one
        ctx.scans = scans
        ctx.shape = (B, C, H, W)
        ctx.triton_shape = (BC, BH, BW, NC, NH, NW)
        if one_by_one:
            x = y.new_empty((B, 4, C, H * W)) if in_channel_first else y.new_empty((B, H * W, 4, C))
        else:
            x = y.new_empty((B, C, H * W)) if in_channel_first else y.new_empty((B, H * W, C))
        triton_cross_scan_flex[(NH * NW, NC, B)](
            x, y.contiguous(), 
            (0 if in_channel_first else 1), (0 if out_channel_first else 1), 1, (0 if not one_by_one else 1), scans,
            BC, BH, BW, C, H, W, NH, NW
        )
        return x
        
    @staticmethod
    def backward(ctx, x: torch.Tensor):
        in_channel_first = ctx.in_channel_first
        out_channel_first = ctx.out_channel_first
        one_by_one = ctx.one_by_one
        scans = ctx.scans
        B, C, H, W = ctx.shape
        BC, BH, BW, NC, NH, NW = ctx.triton_shape
        y = x.new_empty((B, 4, C, H, W)) if out_channel_first else x.new_empty((B, H, W, 4, C))
        triton_cross_scan_flex[(NH * NW, NC, B)](
            x.contiguous(), y, 
            (0 if in_channel_first else 1), (0 if out_channel_first else 1), 0, (0 if not one_by_one else 1), scans,
            BC, BH, BW, C, H, W, NH, NW
        )
        return y, None, None, None, None, None


# @torch.compile(options={"triton.cudagraphs": True}, fullgraph=True)
def cross_scan_fn(x: torch.Tensor, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0, force_torch=False):
    # x: (B, C, H, W) | (B, H, W, C) | (B, 4, C, H, W) | (B, H, W, 4, C)
    # y: (B, 4, C, L) | (B, L, 4, C)
    # scans: 0: cross scan; 1 unidirectional; 2: bidirectional;
    CSF = CrossScanTritonF if WITH_TRITON and x.is_cuda and (not force_torch) else CrossScanF
    with (torch.cuda.device(x.device) if x.is_cuda else contextlib.nullcontext()):  # [notebook patch] CPU-safe
        return CSF.apply(x, in_channel_first, out_channel_first, one_by_one, scans)


# @torch.compile(options={"triton.cudagraphs": True}, fullgraph=True)
def cross_merge_fn(y: torch.Tensor, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0, force_torch=False):
    # y: (B, 4, C, L) | (B, L, 4, C)
    # x: (B, C, H * W) | (B, H * W, C) | (B, 4, C, H * W) | (B, H * W, 4, C)
    # scans: 0: cross scan; 1 unidirectional; 2: bidirectional;
    CMF = CrossMergeTritonF if WITH_TRITON and y.is_cuda and (not force_torch) else CrossMergeF
    with (torch.cuda.device(y.device) if y.is_cuda else contextlib.nullcontext()):  # [notebook patch] CPU-safe
        return CMF.apply(y, in_channel_first, out_channel_first, one_by_one, scans)


# checks =================================================================

class CHECK:
    def check_csm_triton():
        B, C, H, W = 256, 192, 56, 57
        dtype=torch.float16
        dtype=torch.float32
        x = torch.randn((B, C, H, W), dtype=dtype, device=torch.device("cuda")).requires_grad_(True)
        y = torch.randn((B, 4, C, H, W), dtype=dtype, device=torch.device("cuda")).requires_grad_(True)
        x1 = x.clone().detach().requires_grad_(True)
        y1 = y.clone().detach().requires_grad_(True)

        def cross_scan(x: torch.Tensor):
            B, C, H, W = x.shape
            L = H * W
            xs = torch.stack([
                x.view(B, C, L),
                torch.transpose(x, dim0=2, dim1=3).contiguous().view(B, C, L),
                torch.flip(x.contiguous().view(B, C, L), dims=[-1]),
                torch.flip(torch.transpose(x, dim0=2, dim1=3).contiguous().view(B, C, L), dims=[-1]),
            ], dim=1).view(B, 4, C, L)
            return xs
        
        def cross_merge(out_y: torch.Tensor):
            B, K, D, H, W = out_y.shape
            L = H * W
            out_y = out_y.view(B, K, D, L)
            inv_y = torch.flip(out_y[:, 2:4], dims=[-1]).view(B, 2, -1, L)
            wh_y = torch.transpose(out_y[:, 1].view(B, -1, W, H), dim0=2, dim1=3).contiguous().view(B, -1, L)
            invwh_y = torch.transpose(inv_y[:, 1].view(B, -1, W, H), dim0=2, dim1=3).contiguous().view(B, -1, L)
            y = out_y[:, 0] + inv_y[:, 0] + wh_y + invwh_y
            return y

        def cross_scan_1b1(x: torch.Tensor):
            B, K, C, H, W = x.shape
            L = H * W
            xs = torch.stack([
                x[:, 0].view(B, C, L),
                torch.transpose(x[:, 1], dim0=2, dim1=3).contiguous().view(B, C, L),
                torch.flip(x[:, 2].contiguous().view(B, C, L), dims=[-1]),
                torch.flip(torch.transpose(x[:, 3], dim0=2, dim1=3).contiguous().view(B, C, L), dims=[-1]),
            ], dim=1).view(B, 4, C, L)
            return xs
        
        def unidi_scan(x):
            B, C, H, W = x.shape
            x = x.view(B, 1, C, H * W).repeat(1, 4, 1, 1)
            return x
        
        def unidi_merge(ys):
            B, K, C, H, W = ys.shape
            return ys.view(B, 4, -1, H * W).sum(1)

        def bidi_scan(x):
            B, C, H, W = x.shape
            x = x.view(B, 1, C, H * W).repeat(1, 2, 1, 1)
            x = torch.cat([x, x.flip(dims=[-1])], dim=1)
            return x
        
        def bidi_merge(ys):
            B, K, D, H, W = ys.shape
            ys = ys.view(B, K, D, -1)
            ys = ys[:, 0:2] + ys[:, 2:4].flip(dims=[-1]).view(B, 2, D, -1)
            return ys.contiguous().sum(1)

        if True:
            res0 = triton.testing.do_bench(lambda :cross_scan(x))
            res1 = triton.testing.do_bench(lambda :cross_scan_fn(x, True, True, False))
            # res2 = triton.testing.do_bench(lambda :CrossScanTriton.apply(x))
            res3 = triton.testing.do_bench(lambda :cross_merge(y))
            res4 = triton.testing.do_bench(lambda :cross_merge_fn(y, True, True, False))
            # res5 = triton.testing.do_bench(lambda :CrossMergeTriton.apply(y))
            # print(res0, res1, res2, res3, res4, res5)
            print(res0, res1, res3, res4)
            res0 = triton.testing.do_bench(lambda :cross_scan(x).sum().backward())
            res1 = triton.testing.do_bench(lambda :cross_scan_fn(x, True, True, False).sum().backward())
            # res2 = triton.testing.do_bench(lambda :CrossScanTriton.apply(x).sum().backward())
            res3 = triton.testing.do_bench(lambda :cross_merge(y).sum().backward())
            res4 = triton.testing.do_bench(lambda :cross_merge_fn(y, True, True, False).sum().backward())
            # res5 = triton.testing.do_bench(lambda :CrossMergeTriton.apply(y).sum().backward())
            # print(res0, res1, res2, res3, res4, res5)
            print(res0, res1, res3, res4)

        print("test cross scan")
        for (cs0, cm0, cs1, cm1) in [
            # channel_first -> channel_first
            (cross_scan, cross_merge, cross_scan_fn, cross_merge_fn),
            (unidi_scan, unidi_merge, lambda x: cross_scan_fn(x, scans=1), lambda x: cross_merge_fn(x, scans=1)),
            (bidi_scan, bidi_merge, lambda x: cross_scan_fn(x, scans=2), lambda x: cross_merge_fn(x, scans=2)),
            
            # flex: BLC->BCL; BCL->BLC; BLC->BLC;
            (cross_scan, cross_merge, lambda x: cross_scan_fn(x.permute(0, 2, 3, 1), in_channel_first=False), lambda x: cross_merge_fn(x, in_channel_first=False).permute(0, 2, 1)),
            (cross_scan, cross_merge, lambda x: cross_scan_fn(x, out_channel_first=False).permute(0, 2, 3, 1), lambda x: cross_merge_fn(x.permute(0, 3, 4, 1, 2), out_channel_first=False)),
            (cross_scan, cross_merge, lambda x: cross_scan_fn(x.permute(0, 2, 3, 1), in_channel_first=False, out_channel_first=False).permute(0, 2, 3, 1), lambda x: cross_merge_fn(x.permute(0, 3, 4, 1, 2), in_channel_first=False, out_channel_first=False).permute(0, 2, 1)),
            
            # previous
            # (cross_scan, cross_merge, lambda x: CrossScanTriton.apply(x), lambda x: CrossMergeTriton.apply(x)),
            # (unidi_scan, unidi_merge, lambda x: getCSM(1)[0].apply(x), lambda x: getCSM(1)[1].apply(x)),
            # (bidi_scan, bidi_merge, lambda x: getCSM(2)[0].apply(x), lambda x: getCSM(2)[1].apply(x)),
        ]:
            x.grad, x1.grad, y.grad, y1.grad = None, None, None, None
            o0 = cs0(x)
            o1 = cs1(x1)
            o0.backward(y.view(B, 4, C, H * W))
            o1.backward(y.view(B, 4, C, H * W))
            print((o0 - o1).abs().max())
            print((x.grad - x1.grad).abs().max())
            o0 = cm0(y)
            o1 = cm1(y1)
            o0.backward(x.view(B, C, H * W))
            o1.backward(x.view(B, C, H * W))
            print((o0 - o1).abs().max())
            print((y.grad - y1.grad).abs().max())
            x.grad, x1.grad, y.grad, y1.grad = None, None, None, None
            print("===============", flush=True)

        print("test cross scan one by one")
        for (cs0, cs1) in [
            (cross_scan_1b1, lambda x: cross_scan_fn(x, one_by_one=True)),
            # (cross_scan_1b1, lambda x: CrossScanTriton1b1.apply(x)),
        ]:
            o0 = cs0(y)
            o1 = cs1(y1)
            o0.backward(y.view(B, 4, C, H * W))
            o1.backward(y.view(B, 4, C, H * W))
            print((o0 - o1).abs().max())
            print((y.grad - y1.grad).abs().max())
            x.grad, x1.grad, y.grad, y1.grad = None, None, None, None
            print("===============", flush=True)

    def check_csm_scan3():
        if False:
            x = torch.arange(0, 16).view(1, 1, 4, 4).cuda()
            out1 = cross_scan_fn(x, scans=3, force_torch=True).view(1, 4, 1, 4, 4)
            out2 = cross_merge_fn(out1, scans=3, force_torch=True).view(1, 1, 4, 4)
            out4 = cross_merge_fn(out1, one_by_one=True, scans=3, force_torch=True).view(1, 4, 1, 4, 4)
            out3 = cross_scan_fn(out4, one_by_one=True, scans=3, force_torch=True).view(1, 4, 1, 4, 4)
            out5 = cross_scan_fn(x.view(1, 4, 4, 1), in_channel_first=False, out_channel_first=False, scans=3, force_torch=True).view(1, 4, 4, 4, 1)
            out6 = cross_merge_fn(out5, in_channel_first=False, out_channel_first=False, scans=3, force_torch=True).view(1, 4, 4, 1)
            out8 = cross_merge_fn(out5, in_channel_first=False, out_channel_first=False, one_by_one=True, scans=3, force_torch=True).view(1, 4, 4, 4, 1)
            out7 = cross_scan_fn(out8, in_channel_first=False, out_channel_first=False, one_by_one=True, scans=3, force_torch=True).view(1, 4, 4, 4, 1)
            print(out1.view(4, -1))
            print(out2.view(-1))
            print(out3.view(4, -1))
            print(out4.view(4, -1))
            print(out5.view(-1, 4).t())
            print(out6.view(-1))
            print(out7.view(-1, 4).t())
            print(out8.view(-1, 4).t())

        B, C, H, W = 27, 253, 57, 58
        x = torch.randn((B, C, H, W)).cuda()

        for scans in [0, 1, 2, 3]:
            o1 = cross_scan_fn(x, scans=scans, force_torch=True).view(B, 4, C, H, W)
            print((cross_scan_fn(x, scans=scans) == cross_scan_fn(x, scans=scans, force_torch=True)).all())
            print((cross_merge_fn(o1, scans=scans) == cross_merge_fn(o1, scans=scans, force_torch=True)).all())

            kwargs = dict(in_channel_first=False, out_channel_first=False)
            x2 = x.permute(0, 2, 3, 1).contiguous()
            o2 = o1.permute(0, 3, 4, 1, 2).contiguous()
            print((cross_scan_fn(x, scans=scans, **kwargs) == cross_scan_fn(x, scans=scans, force_torch=True, **kwargs)).all())
            print((cross_merge_fn(o2, scans=scans, **kwargs) == cross_merge_fn(o2, scans=scans, force_torch=True, **kwargs)).all())            

        breakpoint()


if __name__ == "__main__":
    CHECK.check_csm_scan3()
    CHECK.check_csm_triton()


In [ ]:
%%writefile {SRC_DIR}/tevmamba/backbones/vmamba.py

# vmamba
import os
import time
import math
import copy
from functools import partial
from typing import Optional, Callable, Any
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from timm.models.layers import DropPath, trunc_normal_
try:  # [notebook patch] fvcore is only used by the FLOP counter
    from fvcore.nn import FlopCountAnalysis, flop_count_str, flop_count, parameter_count
except ImportError:
    FlopCountAnalysis = flop_count_str = flop_count = parameter_count = None

from .LayerTokenReducer import LayerTokenPrune, LayerTokenMerge
from .AdaptivePairMergeGate import do_nothing  # no-op 병합 시 체크 용

DropPath.__repr__ = lambda self: f"timm.DropPath({self.drop_prob})"
# train speed is slower after enabling this opts.
# torch.backends.cudnn.enabled = True
# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.deterministic = True

try:
    from .csm_triton import cross_scan_fn, cross_merge_fn
except:
    from csm_triton import cross_scan_fn, cross_merge_fn

try:
    from .csms6s import selective_scan_fn, selective_scan_flop_jit
except:
    from csms6s import selective_scan_fn, selective_scan_flop_jit

# FLOPs counter not prepared fro mamba2
try:  # [notebook patch] mamba2 (triton) is only used by the unused 'm0' forward types; not embedded
    from .mamba2.ssd_minimal import selective_scan_chunk_fn
except Exception:
    selective_scan_chunk_fn = None

import inspect




# ============================
# 전역 플래그: 토큰 머징/프루닝 적용 여부
# ============================
ENABLE_TOKEN_ADAPTATION = True





# --- Token Merge Utils (horizontal/vertical alternate) ---
def _alt_mode_from_layer_idx(layer_idx: int) -> str:
    """짝수 레이어: horizontal(가로), 홀수 레이어: vertical(세로)"""
    return "horizontal" if (layer_idx % 2 == 0) else "vertical"
    #return "vertical" if (layer_idx % 2 == 0) else "horizontal"

def _flatten_for_merge(x: torch.Tensor, channel_first: bool):
    """
    x: (B,C,H,W) or (B,H,W,C)  ->  (B, H*W, C),  H, W 반환
    """
    if channel_first:
        B, C, H, W = x.shape
        metric = x.permute(0, 2, 3, 1).contiguous().view(B, H * W, C)
        return metric, H, W
    else:
        B, H, W, C = x.shape
        metric = x.view(B, H * W, C)
        return metric, H, W

def _apply_prune_once(x, pruner, delta=None, channel_first=True, state=None, **kwargs):
    """
    LayerTokenPrune 실행 (Δ 기반 프루닝)
    - delta: 이전 블록에서 전달된 Δ
    - channel_first: 입력이 (B,C,H,W)인지 여부
    - state: optional state 전달용
    """
    if pruner is None:
        return x, state

    # === Δ 유효성 확인 ===
    if delta is None:
        print(f"[PRUNE-SKIP] layer={getattr(pruner, 'layer_idx', '?')} delta=None")
        return x, state

    # === 디버그용 로그 ===
    # if getattr(pruner, "layer_idx", None) is not None:
    #     print(f"[PRUNE DEBUG] layer={pruner.layer_idx} delta={tuple(delta.shape)}")

    with torch.no_grad():
        # contiguous 제거 → 메모리 절감
        ret, stats = pruner(x_bchw=x, delta=delta)

    if not isinstance(ret, torch.Tensor):
        raise RuntimeError("prune 결과는 torch.Tensor여야 합니다.")

    # keep mask 전달 (선택)
    if hasattr(pruner, "last_keep_mask"):
        state = {"keep_mask": pruner.last_keep_mask}

    return ret, state


def _apply_merge_once(x, merger, mode, channel_first: bool, state=None, **kwargs):
    """
    LayerTokenMerge 실행
    - mode: horizontal / vertical
    - state: 이전 단계 정보
    """
    if merger is None:
        return x, state

    # === 디버그용 ===
    # if getattr(merger, "layer_idx", None) is not None:
    #     print(f"[MERGE DEBUG] layer={merger.layer_idx} mode={mode}")

    with torch.no_grad():
        sig = merger.forward.__code__.co_varnames
        if "state" in sig:
            ret, new_state = merger(x_bchw=x, state=state, **kwargs)
        else:
            ret, new_state = merger(x_bchw=x, **kwargs)

    if not isinstance(ret, torch.Tensor):
        raise RuntimeError("merge 결과는 torch.Tensor여야 합니다.")

    return ret, new_state



# =====================================================
# we have this class as linear and conv init differ from each other
# this function enable loading from both conv2d or linear
class Linear2d(nn.Linear):
    def forward(self, x: torch.Tensor):
        # B, C, H, W = x.shape
        return F.conv2d(x, self.weight[:, :, None, None], self.bias)

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):
        state_dict[prefix + "weight"] = state_dict[prefix + "weight"].view(self.weight.shape)
        return super()._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)


class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor):
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x


class PatchMerging2D(nn.Module):
    def __init__(self, dim, out_dim=-1, norm_layer=nn.LayerNorm, channel_first=False):
        super().__init__()
        self.dim = dim
        Linear = Linear2d if channel_first else nn.Linear
        self._patch_merging_pad = self._patch_merging_pad_channel_first if channel_first else self._patch_merging_pad_channel_last
        self.reduction = Linear(4 * dim, (2 * dim) if out_dim < 0 else out_dim, bias=False)
        self.norm = norm_layer(4 * dim)

    @staticmethod
    def _patch_merging_pad_channel_last(x: torch.Tensor):
        H, W, _ = x.shape[-3:]
        if (W % 2 != 0) or (H % 2 != 0):
            x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))
        x0 = x[..., 0::2, 0::2, :]  # ... H/2 W/2 C
        x1 = x[..., 1::2, 0::2, :]  # ... H/2 W/2 C
        x2 = x[..., 0::2, 1::2, :]  # ... H/2 W/2 C
        x3 = x[..., 1::2, 1::2, :]  # ... H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # ... H/2 W/2 4*C
        return x

    @staticmethod
    def _patch_merging_pad_channel_first(x: torch.Tensor):
        H, W = x.shape[-2:]
        if (W % 2 != 0) or (H % 2 != 0):
            x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))
        x0 = x[..., 0::2, 0::2]  # ... H/2 W/2
        x1 = x[..., 1::2, 0::2]  # ... H/2 W/2
        x2 = x[..., 0::2, 1::2]  # ... H/2 W/2
        x3 = x[..., 1::2, 1::2]  # ... H/2 W/2
        x = torch.cat([x0, x1, x2, x3], 1)  # ... H/2 W/2 4*C
        return x

    def forward(self, x):
        x = self._patch_merging_pad(x)
        x = self.norm(x)
        x = self.reduction(x)

        return x


class Permute(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.args = args

    def forward(self, x: torch.Tensor):
        return x.permute(*self.args)


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.,channels_first=False):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        Linear = Linear2d if channels_first else nn.Linear
        self.fc1 = Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class gMlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.,channels_first=False):
        super().__init__()
        self.channel_first = channels_first
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        Linear = Linear2d if channels_first else nn.Linear
        self.fc1 = Linear(in_features, 2 * hidden_features)
        self.act = act_layer()
        self.fc2 = Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor):
        x = self.fc1(x)
        x, z = x.chunk(2, dim=(1 if self.channel_first else -1))
        x = self.fc2(x * self.act(z))
        x = self.drop(x)
        return x


class SoftmaxSpatial(nn.Softmax):
    def forward(self, x: torch.Tensor):
        if self.dim == -1:
            B, C, H, W = x.shape
            return super().forward(x.view(B, C, -1)).view(B, C, H, W)
        elif self.dim == 1:
            B, H, W, C = x.shape
            return super().forward(x.view(B, -1, C)).view(B, H, W, C)
        else:
            raise NotImplementedError


# =====================================================
class mamba_init:
    @staticmethod
    def dt_init(dt_rank, d_inner, dt_scale=1.0, dt_init="random", dt_min=0.001, dt_max=0.1, dt_init_floor=1e-4):
        dt_proj = nn.Linear(dt_rank, d_inner, bias=True)

        # Initialize special dt projection to preserve variance at initialization
        dt_init_std = dt_rank**-0.5 * dt_scale
        if dt_init == "constant":
            nn.init.constant_(dt_proj.weight, dt_init_std)
        elif dt_init == "random":
            nn.init.uniform_(dt_proj.weight, -dt_init_std, dt_init_std)
        else:
            raise NotImplementedError

        # Initialize dt bias so that F.softplus(dt_bias) is between dt_min and dt_max
        dt = torch.exp(
            torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        ).clamp(min=dt_init_floor)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            dt_proj.bias.copy_(inv_dt)
        # Our initialization would set all Linear.bias to zero, need to mark this one as _no_reinit
        # dt_proj.bias._no_reinit = True
        
        return dt_proj

    @staticmethod
    def A_log_init(d_state, d_inner, copies=-1, device=None, merge=True):
        # S4D real initialization
        A = torch.arange(1, d_state + 1, dtype=torch.float32, device=device).view(1, -1).repeat(d_inner, 1).contiguous()
        A_log = torch.log(A)  # Keep A_log in fp32
        if copies > 0:
            A_log = A_log[None].repeat(copies, 1, 1).contiguous()
            if merge:
                A_log = A_log.flatten(0, 1)
        A_log = nn.Parameter(A_log)
        A_log._no_weight_decay = True
        return A_log

    @staticmethod
    def D_init(d_inner, copies=-1, device=None, merge=True):
        # D "skip" parameter
        D = torch.ones(d_inner, device=device)
        if copies > 0:
            D = D[None].repeat(copies, 1).contiguous()
            if merge:
                D = D.flatten(0, 1)
        D = nn.Parameter(D)  # Keep in fp32
        D._no_weight_decay = True
        return D

    @classmethod
    def init_dt_A_D(cls, d_state, dt_rank, d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor, k_group=4):
        # dt proj ============================
        dt_projs = [
            cls.dt_init(dt_rank, d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor)
            for _ in range(k_group)
        ]
        dt_projs_weight = nn.Parameter(torch.stack([t.weight for t in dt_projs], dim=0)) # (K, inner, rank)
        dt_projs_bias = nn.Parameter(torch.stack([t.bias for t in dt_projs], dim=0)) # (K, inner)
        del dt_projs
            
        # A, D =======================================
        A_logs = cls.A_log_init(d_state, d_inner, copies=k_group, merge=True) # (K * D, N)
        Ds = cls.D_init(d_inner, copies=k_group, merge=True) # (K * D)  
        return A_logs, Ds, dt_projs_weight, dt_projs_bias


# support: v0, v0seq
class SS2Dv0:
    def __initv0__(
        self,
        # basic dims ===========
        d_model=96,
        d_state=16,
        ssm_ratio=2.0,
        dt_rank="auto",
        # ======================
        dropout=0.0,
        # ======================
        seq=False,
        force_fp32=True,
        **kwargs,
    ):
        if "channel_first" in kwargs:
            assert not kwargs["channel_first"]
        act_layer = nn.SiLU
        dt_min = 0.001
        dt_max = 0.1
        dt_init = "random"
        dt_scale = 1.0
        dt_init_floor = 1e-4
        bias = False
        conv_bias = True
        d_conv = 3
        k_group = 4
        factory_kwargs = {"device": None, "dtype": None}
        super().__init__()
        d_inner = int(ssm_ratio * d_model)
        dt_rank = math.ceil(d_model / 16) if dt_rank == "auto" else dt_rank

        self.forward = self.forwardv0 
        if seq:
            self.forward = partial(self.forwardv0, seq=True)
        if not force_fp32:
            self.forward = partial(self.forwardv0, force_fp32=False)

        # in proj ============================
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=bias)
        self.act: nn.Module = act_layer()
        self.conv2d = nn.Conv2d(
            in_channels=d_inner,
            out_channels=d_inner,
            groups=d_inner,
            bias=conv_bias,
            kernel_size=d_conv,
            padding=(d_conv - 1) // 2,
            **factory_kwargs,
        )

        # x proj ============================
        self.x_proj = [
            nn.Linear(d_inner, (dt_rank + d_state * 2), bias=False)
            for _ in range(k_group)
        ]
        self.x_proj_weight = nn.Parameter(torch.stack([t.weight for t in self.x_proj], dim=0)) # (K, N, inner)
        del self.x_proj

        # dt proj, A, D ============================
        self.A_logs, self.Ds, self.dt_projs_weight, self.dt_projs_bias = mamba_init.init_dt_A_D(
            d_state, dt_rank, d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor, k_group=4,
        )

        # out proj =======================================
        self.out_norm = nn.LayerNorm(d_inner)
        self.out_proj = nn.Linear(d_inner, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0. else nn.Identity()

    def forwardv0(self, x: torch.Tensor, seq=False, force_fp32=True, **kwargs):
        x = self.in_proj(x)
        x, z = x.chunk(2, dim=-1) # (b, h, w, d)
        z = self.act(z)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self.conv2d(x) # (b, d, h, w)
        x = self.act(x)
        selective_scan = partial(selective_scan_fn, backend="mamba")
        
        B, D, H, W = x.shape
        D, N = self.A_logs.shape
        K, D, R = self.dt_projs_weight.shape
        L = H * W

        x_hwwh = torch.stack([x.view(B, -1, L), torch.transpose(x, dim0=2, dim1=3).contiguous().view(B, -1, L)], dim=1).view(B, 2, -1, L)
        xs = torch.cat([x_hwwh, torch.flip(x_hwwh, dims=[-1])], dim=1) # (b, k, d, l)

        x_dbl = torch.einsum("b k d l, k c d -> b k c l", xs, self.x_proj_weight)
        if hasattr(self, "x_proj_bias"):
            x_dbl = x_dbl + self.x_proj_bias.view(1, K, -1, 1)
        dts, Bs, Cs = torch.split(x_dbl, [R, N, N], dim=2)
        dts = torch.einsum("b k r l, k d r -> b k d l", dts, self.dt_projs_weight)

        xs = xs.view(B, -1, L) # (b, k * d, l)
        dts = dts.contiguous().view(B, -1, L) # (b, k * d, l)
        Bs = Bs.contiguous() # (b, k, d_state, l)
        Cs = Cs.contiguous() # (b, k, d_state, l)
        
        As = -self.A_logs.float().exp() # (k * d, d_state)
        Ds = self.Ds.float() # (k * d)
        dt_projs_bias = self.dt_projs_bias.float().view(-1) # (k * d)

        # assert len(xs.shape) == 3 and len(dts.shape) == 3 and len(Bs.shape) == 4 and len(Cs.shape) == 4
        # assert len(As.shape) == 2 and len(Ds.shape) == 1 and len(dt_projs_bias.shape) == 1
        to_fp32 = lambda *args: (_a.to(torch.float32) for _a in args)
        
        if force_fp32:
            xs, dts, Bs, Cs = to_fp32(xs, dts, Bs, Cs)

        if seq:
            out_y = []
            for i in range(4):
                yi = selective_scan(
                    xs.view(B, K, -1, L)[:, i], dts.view(B, K, -1, L)[:, i], 
                    As.view(K, -1, N)[i], Bs[:, i].unsqueeze(1), Cs[:, i].unsqueeze(1), Ds.view(K, -1)[i],
                    delta_bias=dt_projs_bias.view(K, -1)[i],
                    delta_softplus=True,
                ).view(B, -1, L)
                out_y.append(yi)
            out_y = torch.stack(out_y, dim=1)
        else:
            out_y = selective_scan(
                xs, dts, 
                As, Bs, Cs, Ds,
                delta_bias=dt_projs_bias,
                delta_softplus=True,
            ).view(B, K, -1, L)
        assert out_y.dtype == torch.float

        inv_y = torch.flip(out_y[:, 2:4], dims=[-1]).view(B, 2, -1, L)
        wh_y = torch.transpose(out_y[:, 1].view(B, -1, W, H), dim0=2, dim1=3).contiguous().view(B, -1, L)
        invwh_y = torch.transpose(inv_y[:, 1].view(B, -1, W, H), dim0=2, dim1=3).contiguous().view(B, -1, L)
        y = out_y[:, 0] + inv_y[:, 0] + wh_y + invwh_y
        
        y = y.transpose(dim0=1, dim1=2).contiguous() # (B, L, C)
        y = self.out_norm(y).view(B, H, W, -1)

        y = y * z
        out = self.dropout(self.out_proj(y))

        return out


# support: v01-v05; v051d,v052d,v052dc; 
# postfix: _onsigmoid,_onsoftmax,_ondwconv3,_onnone;_nozact,_noz;_oact;_no32;
# history support: v2,v3;v31d,v32d,v32dc;
class SS2Dv2:
    def __initv2__(
        self,
        # basic dims ===========
        d_model=96,
        d_state=16,
        ssm_ratio=2.0,
        dt_rank="auto",
        act_layer=nn.SiLU,
        # dwconv ===============
        d_conv=3, # < 2 means no conv 
        conv_bias=True,
        # ======================
        dropout=0.0,
        bias=False,
        # dt init ==============
        dt_min=0.001,
        dt_max=0.1,
        dt_init="random",
        dt_scale=1.0,
        dt_init_floor=1e-4,
        initialize="v0",
        # ======================
        forward_type="v2",
        channel_first=False,
        # ======================
        **kwargs,    
    ):
        factory_kwargs = {"device": None, "dtype": None}
        super().__init__()
        self.k_group = 4
        self.d_model = int(d_model)
        self.d_state = int(d_state)
        self.d_inner = int(ssm_ratio * d_model)
        self.dt_rank = int(math.ceil(self.d_model / 16) if dt_rank == "auto" else dt_rank)
        self.channel_first = channel_first
        self.with_dconv = d_conv > 1
        Linear = Linear2d if channel_first else nn.Linear
        self.forward = self.forwardv2

        # tags for forward_type ==============================
        checkpostfix = self.checkpostfix
        self.disable_force32, forward_type = checkpostfix("_no32", forward_type)
        self.oact, forward_type = checkpostfix("_oact", forward_type)
        self.disable_z, forward_type = checkpostfix("_noz", forward_type)
        self.disable_z_act, forward_type = checkpostfix("_nozact", forward_type)
        self.out_norm, forward_type = self.get_outnorm(forward_type, self.d_inner, channel_first)

        # forward_type debug =======================================
        FORWARD_TYPES = dict(
            v01=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="mamba", scan_force_torch=True),
            v02=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="mamba"),
            v03=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="oflex"),
            v04=partial(self.forward_corev2, force_fp32=False), # selective_scan_backend="oflex", scan_mode="cross2d"
            v05=partial(self.forward_corev2, force_fp32=False, no_einsum=True),  # selective_scan_backend="oflex", scan_mode="cross2d"
            # ===============================
            v051d=partial(self.forward_corev2, force_fp32=False, no_einsum=True, scan_mode="unidi"),
            v052d=partial(self.forward_corev2, force_fp32=False, no_einsum=True, scan_mode="bidi"),
            v052dc=partial(self.forward_corev2, force_fp32=False, no_einsum=True, scan_mode="cascade2d"),
            v052d3=partial(self.forward_corev2, force_fp32=False, no_einsum=True, scan_mode=3), # debug
            # ===============================
            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="core"),
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="oflex"),
        )
        self.forward_core = FORWARD_TYPES.get(forward_type, None)
        if self.forward_core is not None and hasattr(self.forward_core, 'keywords'):
            self.backend = self.forward_core.keywords.get("selective_scan_backend", None)
        else:
            self.backend = None


        # in proj =======================================
        d_proj = self.d_inner if self.disable_z else (self.d_inner * 2)
        self.in_proj = Linear(self.d_model, d_proj, bias=bias)
        self.act: nn.Module = act_layer()
        
        # conv =======================================
        if self.with_dconv:
            self.conv2d = nn.Conv2d(
                in_channels=self.d_inner,
                out_channels=self.d_inner,
                groups=self.d_inner,
                bias=conv_bias,
                kernel_size=d_conv,
                padding=(d_conv - 1) // 2,
                **factory_kwargs,
            )

        # x proj ============================
        self.x_proj = [
            nn.Linear(self.d_inner, (self.dt_rank + self.d_state * 2), bias=False)
            for _ in range(self.k_group)
        ]
        self.x_proj_weight = nn.Parameter(torch.stack([t.weight for t in self.x_proj], dim=0)) # (K, N, inner)
        del self.x_proj
        
        # out proj =======================================
        self.out_act = nn.GELU() if self.oact else nn.Identity()
        self.out_proj = Linear(self.d_inner, self.d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0. else nn.Identity()

        if initialize in ["v0"]:
            self.A_logs, self.Ds, self.dt_projs_weight, self.dt_projs_bias = mamba_init.init_dt_A_D(
                self.d_state, self.dt_rank, self.d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor, k_group=self.k_group,
            )
        elif initialize in ["v1"]:
            # simple init dt_projs, A_logs, Ds
            self.Ds = nn.Parameter(torch.ones((self.k_group * self.d_inner)))
            self.A_logs = nn.Parameter(torch.randn((self.k_group * self.d_inner, self.d_state))) # A == -A_logs.exp() < 0; # 0 < exp(A * dt) < 1
            self.dt_projs_weight = nn.Parameter(0.1 * torch.randn((self.k_group, self.d_inner, self.dt_rank))) # 0.1 is added in 0430
            self.dt_projs_bias = nn.Parameter(0.1 * torch.randn((self.k_group, self.d_inner))) # 0.1 is added in 0430
        elif initialize in ["v2"]:
            # simple init dt_projs, A_logs, Ds
            self.Ds = nn.Parameter(torch.ones((self.k_group * self.d_inner)))
            self.A_logs = nn.Parameter(torch.zeros((self.k_group * self.d_inner, self.d_state))) # A == -A_logs.exp() < 0; # 0 < exp(A * dt) < 1
            self.dt_projs_weight = nn.Parameter(0.1 * torch.rand((self.k_group, self.d_inner, self.dt_rank)))
            self.dt_projs_bias = nn.Parameter(0.1 * torch.rand((self.k_group, self.d_inner)))

    def forward_corev2(
        self,
        x: torch.Tensor=None, 
        # ==============================
        force_fp32=False, # True: input fp32
        # ==============================
        ssoflex=True, # True: input 16 or 32 output 32 False: output dtype as input
        no_einsum=False, # replace einsum with linear or conv1d to raise throughput
        # ==============================
        selective_scan_backend = None,
        # ==============================
        scan_mode = "cross2d",
        scan_force_torch = False,
        # ==============================
        **kwargs,
    ):
        # 자동 매핑 추가
        if selective_scan_backend == "core":
            selective_scan_backend = "oflex"
            
        assert selective_scan_backend in [None, "oflex", "mamba", "torch"]
        _scan_mode = dict(cross2d=0, unidi=1, bidi=2, cascade2d=-1).get(scan_mode, None) if isinstance(scan_mode, str) else scan_mode # for debug
        assert isinstance(_scan_mode, int)
        delta_softplus = True
        out_norm = self.out_norm
        channel_first = self.channel_first
        to_fp32 = lambda *args: (_a.to(torch.float32) for _a in args)

        B, D, H, W = x.shape
        N = self.d_state
        K, D, R = self.k_group, self.d_inner, self.dt_rank
        L = H * W

        def selective_scan(u, delta, A, B, C, D=None, delta_bias=None, delta_softplus=True):
            return selective_scan_fn(u, delta, A, B, C, D, delta_bias, delta_softplus, ssoflex, backend=selective_scan_backend)
        
        if _scan_mode == -1:
            x_proj_bias = getattr(self, "x_proj_bias", None)
            def scan_rowcol(
                x: torch.Tensor, 
                proj_weight: torch.Tensor, 
                proj_bias: torch.Tensor, 
                dt_weight: torch.Tensor, 
                dt_bias: torch.Tensor, # (2*c)
                _As: torch.Tensor, # As = -torch.exp(A_logs.to(torch.float))[:2,] # (2*c, d_state)
                _Ds: torch.Tensor,
                width = True,
            ):
                # x: (B, D, H, W)
                # proj_weight: (2 * D, (R+N+N))
                XB, XD, XH, XW = x.shape
                if width:
                    _B, _D, _L = XB * XH, XD, XW
                    xs = x.permute(0, 2, 1, 3).contiguous()
                else:
                    _B, _D, _L = XB * XW, XD, XH
                    xs = x.permute(0, 3, 1, 2).contiguous()
                xs = torch.stack([xs, xs.flip(dims=[-1])], dim=2) # (B, H, 2, D, W)
                if no_einsum:
                    x_dbl = F.conv1d(xs.view(_B, -1, _L), proj_weight.view(-1, _D, 1), bias=(proj_bias.view(-1) if proj_bias is not None else None), groups=2)
                    dts, Bs, Cs = torch.split(x_dbl.view(_B, 2, -1, _L), [R, N, N], dim=2)
                    dts = F.conv1d(dts.contiguous().view(_B, -1, _L), dt_weight.view(2 * _D, -1, 1), groups=2)
                else:
                    x_dbl = torch.einsum("b k d l, k c d -> b k c l", xs, proj_weight)
                    if x_proj_bias is not None:
                        x_dbl = x_dbl + x_proj_bias.view(1, 2, -1, 1)
                    dts, Bs, Cs = torch.split(x_dbl, [R, N, N], dim=2)
                    dts = torch.einsum("b k r l, k d r -> b k d l", dts, dt_weight)

                xs = xs.view(_B, -1, _L)
                dts = dts.contiguous().view(_B, -1, _L)
                As = _As.view(-1, N).to(torch.float)
                Bs = Bs.contiguous().view(_B, 2, N, _L)
                Cs = Cs.contiguous().view(_B, 2, N, _L)
                Ds = _Ds.view(-1)
                delta_bias = dt_bias.view(-1).to(torch.float)

                if force_fp32:
                    xs = xs.to(torch.float)
                dts = dts.to(xs.dtype)
                Bs = Bs.to(xs.dtype)
                Cs = Cs.to(xs.dtype)

                ys: torch.Tensor = selective_scan(
                    xs, dts, As, Bs, Cs, Ds, delta_bias, delta_softplus
                ).view(_B, 2, -1, _L)
                return ys
            
            As = -self.A_logs.to(torch.float).exp().view(4, -1, N)
            x = F.layer_norm(x.permute(0, 2, 3, 1), normalized_shape=(int(x.shape[1]),)).permute(0, 3, 1, 2).contiguous() # added0510 to avoid nan
            y_row = scan_rowcol(
                x,
                proj_weight = self.x_proj_weight.view(4, -1, D)[:2].contiguous(), 
                proj_bias = (x_proj_bias.view(4, -1)[:2].contiguous() if x_proj_bias is not None else None),
                dt_weight = self.dt_projs_weight.view(4, D, -1)[:2].contiguous(),
                dt_bias = (self.dt_projs_bias.view(4, -1)[:2].contiguous() if self.dt_projs_bias is not None else None),
                _As = As[:2].contiguous().view(-1, N),
                _Ds = self.Ds.view(4, -1)[:2].contiguous().view(-1),
                width=True,
            ).view(B, H, 2, -1, W).sum(dim=2).permute(0, 2, 1, 3) # (B,C,H,W)
            y_row = F.layer_norm(y_row.permute(0, 2, 3, 1), normalized_shape=(int(y_row.shape[1]),)).permute(0, 3, 1, 2).contiguous() # added0510 to avoid nan
            y_col = scan_rowcol(
                y_row,
                proj_weight = self.x_proj_weight.view(4, -1, D)[2:].contiguous().to(y_row.dtype), 
                proj_bias = (x_proj_bias.view(4, -1)[2:].contiguous().to(y_row.dtype) if x_proj_bias is not None else None),
                dt_weight = self.dt_projs_weight.view(4, D, -1)[2:].contiguous().to(y_row.dtype),
                dt_bias = (self.dt_projs_bias.view(4, -1)[2:].contiguous().to(y_row.dtype) if self.dt_projs_bias is not None else None),
                _As = As[2:].contiguous().view(-1, N),
                _Ds = self.Ds.view(4, -1)[2:].contiguous().view(-1),
                width=False,
            ).view(B, W, 2, -1, H).sum(dim=2).permute(0, 2, 3, 1)
            y = y_col
        else:
            x_proj_bias = getattr(self, "x_proj_bias", None)
            xs = cross_scan_fn(x, in_channel_first=True, out_channel_first=True, scans=_scan_mode, force_torch=scan_force_torch)
            if no_einsum:
                x_dbl = F.conv1d(xs.view(B, -1, L), self.x_proj_weight.view(-1, D, 1), bias=(x_proj_bias.view(-1) if x_proj_bias is not None else None), groups=K)
                dts, Bs, Cs = torch.split(x_dbl.view(B, K, -1, L), [R, N, N], dim=2)
                if hasattr(self, "dt_projs_weight"):
                    dts = F.conv1d(dts.contiguous().view(B, -1, L), self.dt_projs_weight.view(K * D, -1, 1), groups=K)
            else:
                x_dbl = torch.einsum("b k d l, k c d -> b k c l", xs, self.x_proj_weight)
                if x_proj_bias is not None:
                    x_dbl = x_dbl + x_proj_bias.view(1, K, -1, 1)
                dts, Bs, Cs = torch.split(x_dbl, [R, N, N], dim=2)
                if hasattr(self, "dt_projs_weight"):
                    dts = torch.einsum("b k r l, k d r -> b k d l", dts, self.dt_projs_weight)

            xs = xs.view(B, -1, L)
            if hasattr(self, "prune_module") and getattr(self.prune_module, "hsa_meta", None) is not None:
                pos_map = self.prune_module.hsa_meta.get("position_map", None)
                if (pos_map is not None) and (pos_map.shape[1] == L):
                    keep = pos_map.to(xs.dtype).unsqueeze(1)
                    xs = xs * keep

            dts = dts.contiguous().view(B, -1, L)



            with torch.no_grad():
                # Δ를 공간 구조로 유지 (B, H, W)
                # dts: (B, KD, L)  (이미 위에서 dts = dts.view(B, -1, L) 했으니 KD축 평균)
                delta_map = F.softplus(dts).abs().mean(dim=1).view(B, 1, H, W).contiguous()
                self.cached_delta = delta_map.detach().to(torch.float16)

                

            As = -self.A_logs.to(torch.float).exp() # (k * c, d_state)
            Ds = self.Ds.to(torch.float) # (K * c)
            Bs = Bs.contiguous().view(B, K, N, L)
            Cs = Cs.contiguous().view(B, K, N, L)
            delta_bias = self.dt_projs_bias.view(-1).to(torch.float)

            if force_fp32:
                xs, dts, Bs, Cs = to_fp32(xs, dts, Bs, Cs)

            ys: torch.Tensor = selective_scan(
                xs, dts, As, Bs, Cs, Ds, delta_bias, delta_softplus
            ).view(B, K, -1, H, W)

                        
            y: torch.Tensor = cross_merge_fn(ys, in_channel_first=True, out_channel_first=True, scans=_scan_mode, force_torch=scan_force_torch)

            if getattr(self, "__DEBUG__", False):
                setattr(self, "__data__", dict(
                    A_logs=self.A_logs, Bs=Bs, Cs=Cs, Ds=Ds,
                    us=xs, dts=dts, delta_bias=delta_bias,
                    ys=ys, y=y, H=H, W=W,
                ))

        y = y.view(B, -1, H, W)
        if not channel_first:
            y = y.view(B, -1, H * W).transpose(dim0=1, dim1=2).contiguous().view(B, H, W, -1) # (B, L, C)
        y = out_norm(y)

        return y.to(x.dtype)

    def forwardv2(self, x: torch.Tensor, **kwargs):
        x = self.in_proj(x)
        if not self.disable_z:
            x, z = x.chunk(2, dim=(1 if self.channel_first else -1)) # (b, h, w, d)
            if not self.disable_z_act:
                z = self.act(z)

        # 채널-퍼스트 정렬
        if not self.channel_first:
            x = x.permute(0, 3, 1, 2).contiguous()

        if self.with_dconv:
            x = self.conv2d(x) # (b, d, h, w)
        x = self.act(x)

        y = self.forward_corev2(x, selective_scan_backend=self.backend)

        state = None
        layer_idx = getattr(self, "_layer_idx", 0)
        mode = _alt_mode_from_layer_idx(layer_idx)
        H, W = x.shape[-2], x.shape[-1]

        
        if ENABLE_TOKEN_ADAPTATION and hasattr(self, "merge_module"):
            y, state = _apply_merge_once(
                y,
                self.merge_module,
                channel_first=self.channel_first,
                state=state,
                mode=mode,
            )


        # --- PRUNE (SS2D 후 Δ 기반 프루닝) ---
        delta_cur = getattr(self, "cached_delta", None)  # self 안에 Δ 직접 존재

        # if delta_cur is None:
        #     print(f"[PRUNE-SKIP] layer={self._layer_idx} delta=None")
        # else:
        #     print(f"[PRUNE DEBUG] layer={self._layer_idx} delta={tuple(delta_cur.shape)}")
        

        if ENABLE_TOKEN_ADAPTATION and hasattr(self, "prune_module") and self.prune_module is not None:
            y, state = _apply_prune_once(
                y,
                self.prune_module,
                delta=delta_cur,
                channel_first=self.channel_first,
                state=state,
            )
            self.cached_delta = delta_cur.detach()


        y = self.out_act(y)
        if not self.disable_z:
            y = y * z
        out = self.dropout(self.out_proj(y))
        return out

    @staticmethod
    def get_outnorm(forward_type="", d_inner=192, channel_first=True):
        def checkpostfix(tag, value):
            ret = value[-len(tag):] == tag
            if ret:
                value = value[:-len(tag)]
            return ret, value

        LayerNorm = LayerNorm2d if channel_first else nn.LayerNorm

        out_norm_none, forward_type = checkpostfix("_onnone", forward_type)
        out_norm_dwconv3, forward_type = checkpostfix("_ondwconv3", forward_type)
        out_norm_cnorm, forward_type = checkpostfix("_oncnorm", forward_type)
        out_norm_softmax, forward_type = checkpostfix("_onsoftmax", forward_type)
        out_norm_sigmoid, forward_type = checkpostfix("_onsigmoid", forward_type)

        out_norm = nn.Identity()
        if out_norm_none:
            out_norm = nn.Identity()
        elif out_norm_cnorm:
            out_norm = nn.Sequential(
                LayerNorm(d_inner),
                (nn.Identity() if channel_first else Permute(0, 3, 1, 2)),
                nn.Conv2d(d_inner, d_inner, kernel_size=3, padding=1, groups=d_inner, bias=False),
                (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
            )
        elif out_norm_dwconv3:
            out_norm = nn.Sequential(
                (nn.Identity() if channel_first else Permute(0, 3, 1, 2)),
                nn.Conv2d(d_inner, d_inner, kernel_size=3, padding=1, groups=d_inner, bias=False),
                (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
            )
        elif out_norm_softmax:
            out_norm = SoftmaxSpatial(dim=(-1 if channel_first else 1))
        elif out_norm_sigmoid:
            out_norm = nn.Sigmoid()
        else:
            out_norm = LayerNorm(d_inner)

        return out_norm, forward_type

    @staticmethod
    def checkpostfix(tag, value):
        ret = value[-len(tag):] == tag
        if ret:
            value = value[:-len(tag)]
        return ret, value


# support: xv1a,xv2a,xv3a; 
# postfix: _cpos;_ocov;_ocov2;_ca,_ca1;_act;_mul;_onsigmoid,_onsoftmax,_ondwconv3,_onnone;
class SS2Dv3:
    def __initxv__(
        self,
        # basic dims ===========
        d_model=96,
        d_state=16,
        ssm_ratio=2.0,
        dt_rank="auto",
        # dwconv ===============
        d_conv=3, # < 2 means no conv 
        conv_bias=True,
        # ======================
        dropout=0.0,
        bias=False,
        # dt init ==============
        dt_min=0.001,
        dt_max=0.1,
        dt_init="random",
        dt_scale=1.0,
        dt_init_floor=1e-4,
        initialize="v0",
        # ======================
        forward_type="v2",
        channel_first=False,
        # ======================
        **kwargs,
    ):
        super().__init__()
        d_inner = int(ssm_ratio * d_model)
        dt_rank = math.ceil(d_model / 16) if dt_rank == "auto" else dt_rank
        self.channel_first = channel_first
        self.d_state = d_state
        self.dt_rank = dt_rank
        self.d_inner = d_inner
        k_group = 4
        self.with_dconv = d_conv > 1
        Linear = Linear2d if channel_first else nn.Linear
        self.forward = self.forwardxv

        # tags for forward_type ==============================
        checkpostfix = SS2Dv2.checkpostfix
        self.out_norm, forward_type = SS2Dv2.get_outnorm(forward_type, d_inner, channel_first)
        self.omul, forward_type = checkpostfix("_mul", forward_type)
        self.oact, forward_type = checkpostfix("_act", forward_type)
        self.f_omul = nn.Identity() if self.omul else None
        self.out_act = nn.GELU() if self.oact else nn.Identity()

        mode = forward_type[:4]
        assert mode in ["xv1a", "xv2a", "xv3a"]

        self.forward = partial(self.forwardxv, mode=mode)
        self.dts_dim = dict(xv1a=self.dt_rank, xv2a=self.d_inner, xv3a=4 * self.dt_rank)[mode]
        d_inner_all = d_inner + self.dts_dim + 8 * d_state
        self.in_proj = Linear(d_model, d_inner_all, bias=bias)
        
        # conv =======================================
        self.cpos = False
        self.iconv = False
        self.oconv = False
        self.oconv2 = False
        if self.with_dconv:
            cact, forward_type = checkpostfix("_ca", forward_type)
            cact1, forward_type = checkpostfix("_ca1", forward_type)
            self.cact = nn.SiLU() if cact else nn.Identity()
            self.cact = nn.GELU() if cact1 else self.cact
                
            self.oconv2, forward_type = checkpostfix("_ocov2", forward_type)
            self.oconv, forward_type = checkpostfix("_ocov", forward_type)
            self.cpos, forward_type = checkpostfix("_cpos", forward_type)
            self.iconv = (not self.oconv) and (not self.oconv2)

            if self.iconv:
                self.conv2d = nn.Conv2d(
                    in_channels=d_model,
                    out_channels=d_model,
                    groups=d_model,
                    bias=conv_bias,
                    kernel_size=d_conv,
                    padding=(d_conv - 1) // 2,
                )
            if self.oconv:
                self.oconv2d = nn.Conv2d(
                    in_channels=d_inner,
                    out_channels=d_inner,
                    groups=d_inner,
                    bias=conv_bias,
                    kernel_size=d_conv,
                    padding=(d_conv - 1) // 2,
                )
            if self.oconv2:
                self.conv2d = nn.Conv2d(
                    in_channels=d_inner_all,
                    out_channels=d_inner_all,
                    groups=d_inner_all,
                    bias=conv_bias,
                    kernel_size=d_conv,
                    padding=(d_conv - 1) // 2,
                )

        # out proj =======================================
        self.out_proj = Linear(d_inner, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        if initialize in ["v0"]:
            self.A_logs, self.Ds, self.dt_projs_weight, self.dt_projs_bias = mamba_init.init_dt_A_D(
                d_state, dt_rank, d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor, k_group=4,
            )
        elif initialize in ["v1"]:
            # simple init dt_projs, A_logs, Ds
            self.Ds = nn.Parameter(torch.ones((k_group * d_inner)))
            self.A_logs = nn.Parameter(torch.randn((k_group * d_inner, d_state))) # A == -A_logs.exp() < 0; # 0 < exp(A * dt) < 1
            self.dt_projs_weight = nn.Parameter(torch.randn((k_group, d_inner, dt_rank)))
            self.dt_projs_bias = nn.Parameter(torch.randn((k_group, d_inner))) 
        elif initialize in ["v2"]:
            # simple init dt_projs, A_logs, Ds
            self.Ds = nn.Parameter(torch.ones((k_group * d_inner)))
            self.A_logs = nn.Parameter(torch.zeros((k_group * d_inner, d_state))) # A == -A_logs.exp() < 0; # 0 < exp(A * dt) < 1
            self.dt_projs_weight = nn.Parameter(0.1 * torch.rand((k_group, d_inner, dt_rank)))
            self.dt_projs_bias = nn.Parameter(0.1 * torch.rand((k_group, d_inner)))


        if forward_type.startswith("xv2"):
            del self.dt_projs_weight
            self.dt_projs_weight = None

    def forwardxv(self, x: torch.Tensor, **kwargs):
        B, (H, W) = x.shape[0], (x.shape[2:4] if self.channel_first else x.shape[1:3])
        L = H * W
        force_fp32 = False
        delta_softplus = True
        out_norm = self.out_norm
        to_dtype = True

        to_fp32 = lambda *args: (_a.to(torch.float32) for _a in args)

        def selective_scan(u, delta, A, B, C, D, delta_bias, delta_softplus):
            return selective_scan_fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex=True, backend=None)


        # === prune → merge ===
        state = None
        layer_idx = getattr(self, "_layer_idx", 0)
        mode = _alt_mode_from_layer_idx(layer_idx)

        

        # MERGE
        if hasattr(self, "merge_module"):
            x, _ = _apply_merge_once(
                x,
                self.merge_module,
                mode=mode,
                channel_first=self.channel_first,
            )

        if torch.isnan(x).any() or torch.isinf(x).any():
            print("NaN DETECTED BEFORE CONV", x.mean(), x.std())
            exit()


        # === conv 단계로 진입 ===
        if self.iconv:
            x = self.cact(self.conv2d(x)) # (b, d, h, w)
        elif self.cpos:
            x = x + self.conv2d(x) # (b, d, h, w)

        x = self.in_proj(x)
        
        if self.oconv2:
            x = self.conv2d(x) # (b, d, h, w)


        us, dts, Bs, Cs = x.split([self.d_inner, self.dts_dim, 4 * self.d_state, 4 * self.d_state], dim=(1 if self.channel_first else -1))

        _us = us
        # Bs, Cs = Bs.view(B, H, W, 4, -1), Cs.view(B, H, W, 4, -1)
        # Bs, Cs = Bs.view(B, 4, -1, H, W), Cs.view(B, 4, -1, H, W)
        us = cross_scan_fn(us.contiguous(), in_channel_first=self.channel_first, out_channel_first=True).view(B, -1, L)
        Bs = cross_scan_fn(Bs.contiguous(), in_channel_first=self.channel_first, out_channel_first=True, one_by_one=True).view(B, 4, -1, L)
        Cs = cross_scan_fn(Cs.contiguous(), in_channel_first=self.channel_first, out_channel_first=True, one_by_one=True).view(B, 4, -1, L)
        dts = cross_scan_fn(dts.contiguous(), in_channel_first=self.channel_first, out_channel_first=True, one_by_one=(self.dts_dim == 4 * self.dt_rank)).view(B, L, -1)
        if self.dts_dim == self.dt_rank:
            dts = F.conv1d(dts, self.dt_projs_weight.view(4 * self.d_inner, self.dt_rank, 1), None, groups=4)
        elif self.dts_dim == 4 * self.dt_rank:
            dts = F.conv1d(dts, self.dt_projs_weight.view(4 * self.d_inner, self.dt_rank, 1), None, groups=4)

        As = -self.A_logs.to(torch.float).exp() # (k * c, d_state)
        Ds = self.Ds.to(torch.float) # (K * c)
        delta_bias = self.dt_projs_bias.view(-1).to(torch.float) # (K * c)

        if force_fp32:
            us, dts, Bs, Cs = to_fp32(us, dts, Bs, Cs)

        ys: torch.Tensor = selective_scan(
            us, dts, As, Bs, Cs, Ds, delta_bias, delta_softplus
        ).view(B, 4, -1, H, W)
        y: torch.Tensor = cross_merge_fn(ys.contiguous(), in_channel_first=self.channel_first, out_channel_first=True)
        y = y.view(B, -1, H, W) if self.channel_first else y.view(B, H, W, -1)
        y = out_norm(y)

        # PRUNE
        if hasattr(self, "prune_module"):
            delta_cur = dts.mean(dim=-1).view(B, 1, H, W)   # Δ 생성
            y, _ = _apply_prune_once(y, self.prune_module, delta=delta_cur, channel_first=self.channel_first
            )

        if getattr(self, "__DEBUG__", False):
            setattr(self, "__data__", dict(
                A_logs=self.A_logs, Bs=Bs, Cs=Cs, Ds=Ds,
                us=us, dts=dts, delta_bias=delta_bias,
                ys=ys, y=y,
            ))

        y = (y.to(x.dtype) if to_dtype else y)
        
        y = self.out_act(y)
        
        if self.omul:
            y = y * _us

        if self.oconv:
            y = y + self.cact(self.oconv2d(_us))

        out = self.dropout(self.out_proj(y))
        return out


# mamba2 support ================================
class SS2Dm0:
    def __initm0__(
        self,
        # basic dims ===========
        d_model=96,
        d_state=16, # now with mamba2, dstate should be bigger...
        ssm_ratio=2.0,
        dt_rank="auto",
        act_layer=nn.GELU,
        # dwconv ===============
        d_conv=3, # < 2 means no conv 
        conv_bias=True,
        # ======================
        dropout=0.0,
        bias=False,
        # dt init ==============
        dt_min=0.001,
        dt_max=0.1,
        dt_init="random",
        dt_scale=1.0,
        dt_init_floor=1e-4,
        initialize="v2",
        # ======================
        forward_type="m0",
        # ======================
        with_initial_state=False,
        # ======================
        **kwargs,    
    ):
        factory_kwargs = {"device": None, "dtype": None}
        super().__init__()
        d_inner = int(ssm_ratio * d_model)
        dt_rank = math.ceil(d_model / 16) if dt_rank == "auto" else dt_rank
        assert d_inner % dt_rank == 0
        self.with_dconv = d_conv > 1
        Linear = nn.Linear
        self.forward = self.forwardm0

        # tags for forward_type ==============================
        checkpostfix = SS2Dv2.checkpostfix
        self.disable_force32, forward_type = checkpostfix("_no32", forward_type)
        self.oact, forward_type = checkpostfix("_oact", forward_type)
        self.disable_z, forward_type = checkpostfix("_noz", forward_type)
        self.disable_z_act, forward_type = checkpostfix("_nozact", forward_type)
        self.out_norm, forward_type = SS2Dv2.get_outnorm(forward_type, d_inner, False)

        # forward_type debug =======================================
        FORWARD_TYPES = dict(
            m0=partial(self.forward_corem0, force_fp32=False, dstate=d_state),
        )
        self.forward_core = FORWARD_TYPES.get(forward_type, None)
        k_group = 4

        # in proj =======================================
        d_proj = d_inner if self.disable_z else (d_inner * 2)
        self.in_proj = Linear(d_model, d_proj, bias=bias)
        self.act: nn.Module = act_layer()
        
        # conv =======================================
        if self.with_dconv:
            self.conv2d = nn.Sequential(
                Permute(0, 3, 1, 2),
                nn.Conv2d(
                    in_channels=d_inner,
                    out_channels=d_inner,
                    groups=d_inner,
                    bias=conv_bias,
                    kernel_size=d_conv,
                    padding=(d_conv - 1) // 2,
                    **factory_kwargs,
                ),
                Permute(0, 2, 3, 1),
            ) 
        
        # x proj ============================
        self.x_proj = [
            nn.Linear(d_inner, (dt_rank + d_state * 2), bias=False)
            for _ in range(k_group)
        ]
        self.x_proj_weight = nn.Parameter(torch.stack([t.weight for t in self.x_proj], dim=0)) # (K, N, inner)
        del self.x_proj
        
        # out proj =======================================
        self.out_act = nn.GELU() if self.oact else nn.Identity()
        self.out_proj = Linear(d_inner, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0. else nn.Identity()

        if initialize in ["v1"]:
            # simple init dt_projs, A_logs, Ds
            self.Ds = nn.Parameter(torch.ones((k_group, dt_rank, int(d_inner // dt_rank))))
            self.A_logs = nn.Parameter(torch.randn((k_group, dt_rank))) # A == -A_logs.exp() < 0; # 0 < exp(A * dt) < 1
            self.dt_projs_bias = nn.Parameter(0.1 * torch.randn((k_group, dt_rank))) # 0.1 is added in 0430
        elif initialize in ["v2"]:
            # simple init dt_projs, A_logs, Ds
            self.Ds = nn.Parameter(torch.ones((k_group, dt_rank, int(d_inner // dt_rank))))
            self.A_logs = nn.Parameter(torch.zeros((k_group, dt_rank))) # A == -A_logs.exp() < 0; # 0 < exp(A * dt) < 1
            self.dt_projs_bias = nn.Parameter(0.1 * torch.rand((k_group, dt_rank)))

        # init state ============================
        self.initial_state = None
        if with_initial_state:
            self.initial_state = nn.Parameter(torch.zeros((1, k_group * dt_rank, int(d_inner // dt_rank), d_state)), requires_grad=False)

    def forward_corem0(
        self,
        x: torch.Tensor=None, 
        # ==============================
        force_fp32=False, # True: input fp32
        chunk_size = 64,
        dstate = 64,        
        # ==============================
        selective_scan_backend = None,
        scan_mode = "cross2d",
        scan_force_torch = False,
        # ==============================
        **kwargs,
    ):
        assert scan_mode in ["unidi", "bidi", "cross2d"]
        assert selective_scan_backend in [None, "triton", "torch"]
        x_proj_bias = getattr(self, "x_proj_bias", None)
        to_fp32 = lambda *args: (_a.to(torch.float32) for _a in args)

        N = dstate
        B, H, W, RD = x.shape
        K, R = self.A_logs.shape
        K, R, D = self.Ds.shape
        assert RD == R * D
        L = H * W
        KR = K * R
        _scan_mode = dict(cross2d=0, unidi=1, bidi=2, cascade2d=3)[scan_mode]

        initial_state = None
        if self.initial_state is not None:
            assert self.initial_state.shape[-1] == dstate
            initial_state = self.initial_state.detach().repeat(B, 1, 1, 1)
        xs = cross_scan_fn(x.view(B, H, W, RD), in_channel_first=False, out_channel_first=False, scans=_scan_mode, force_torch=scan_force_torch) # (B, H, W, 4, D)
        x_dbl = torch.einsum("b l k d, k c d -> b l k c", xs, self.x_proj_weight)
        if x_proj_bias is not None:
            x_dbl = x_dbl + x_proj_bias.view(1, -1, K, 1)
        dts, Bs, Cs = torch.split(x_dbl, [R, N, N], dim=3)
        xs = xs.contiguous().view(B, L, KR, D)
        dts = dts.contiguous().view(B, L, KR)
        Bs = Bs.contiguous().view(B, L, K, N)
        Cs = Cs.contiguous().view(B, L, K, N)
        if force_fp32:
            xs, dts, Bs, Cs = to_fp32(xs, dts, Bs, Cs)

        As = -self.A_logs.to(torch.float).exp().view(KR)
        Ds = self.Ds.to(torch.float).view(KR, D)
        dt_bias = self.dt_projs_bias.view(KR)

        if force_fp32:
            xs, dts, Bs, Cs = to_fp32(xs, dts, Bs, Cs)

        ys, final_state = selective_scan_chunk_fn(
            xs, dts, As, Bs, Cs, chunk_size=chunk_size, D=Ds, dt_bias=dt_bias, 
            initial_states=initial_state, dt_softplus=True, return_final_states=True,
            backend=selective_scan_backend,
        )
        y: torch.Tensor = cross_merge_fn(ys.view(B, H, W, K, RD), in_channel_first=False, out_channel_first=False, scans=_scan_mode, force_torch=scan_force_torch)

        if getattr(self, "__DEBUG__", False):
            setattr(self, "__data__", dict(
                A_logs=self.A_logs, Bs=Bs, Cs=Cs, Ds=self.Ds,
                us=xs, dts=dts, delta_bias=self.dt_projs_bias, 
                initial_state=self.initial_state, final_satte=final_state,
                ys=ys, y=y, H=H, W=W,
            ))
        if self.initial_state is not None:
            self.initial_state = nn.Parameter(final_state.detach().sum(0, keepdim=True), requires_grad=False)

        y = self.out_norm(y.view(B, H, W, -1))

        return y.to(x.dtype)

    def forwardm0(self, x: torch.Tensor, **kwargs):
        x = self.in_proj(x)
        if not self.disable_z:
            x, z = x.chunk(2, dim=(1 if self.channel_first else -1)) # (b, h, w, d)
            if not self.disable_z_act:
                z = self.act(z)
        if self.with_dconv:
            x = self.conv2d(x) # (b, d, h, w)
        x = self.act(x)
        y = self.forward_core(x)
        y = self.out_act(y)
        if not self.disable_z:
            y = y * z
        out = self.dropout(self.out_proj(y))
        return out


class SS2D(nn.Module, SS2Dv0, SS2Dv2, SS2Dv3, SS2Dm0):
    def __init__(
        self,
        # basic dims ===========
        d_model=96,
        d_state=16,
        ssm_ratio=2.0,
        dt_rank="auto",
        act_layer=nn.SiLU,
        # dwconv ===============
        d_conv=3, # < 2 means no conv 
        conv_bias=True,
        # ======================
        dropout=0.0,
        bias=False,
        # dt init ==============
        dt_min=0.001,
        dt_max=0.1,
        dt_init="random",
        dt_scale=1.0,
        dt_init_floor=1e-4,
        initialize="v0",
        # ======================
        forward_type="v2",
        channel_first=False,
        # ======================
        **kwargs,
    ):
        nn.Module.__init__(self)
        self.channel_first = channel_first
        kwargs.update(
            d_model=d_model, d_state=d_state, ssm_ratio=ssm_ratio, dt_rank=dt_rank,
            act_layer=act_layer, d_conv=d_conv, conv_bias=conv_bias, dropout=dropout, bias=bias,
            dt_min=dt_min, dt_max=dt_max, dt_init=dt_init, dt_scale=dt_scale, dt_init_floor=dt_init_floor,
            initialize=initialize, forward_type=forward_type, channel_first=channel_first,
        )
        if forward_type in ["v0", "v0seq"]:
            self.__initv0__(seq=("seq" in forward_type), **kwargs)
        elif forward_type.startswith("xv"):
            self.__initxv__(**kwargs)
        elif forward_type.startswith("m"):
            self.__initm0__(**kwargs)
        else:
            self.__initv2__(**kwargs)
     
class VSSBlock(nn.Module):
    def __init__(
        self,
        hidden_dim: int = 0,
        drop_path: float = 0,
        norm_layer: nn.Module = nn.LayerNorm,
        channel_first=False,
        # =============================
        ssm_d_state: int = 16,
        ssm_ratio=2.0,
        ssm_dt_rank: Any = "auto",
        ssm_act_layer=nn.SiLU,
        ssm_conv: int = 3,
        ssm_conv_bias=True,
        ssm_drop_rate: float = 0,
        ssm_init="v0",
        forward_type="v2",
        # =============================
        mlp_ratio=4.0,
        mlp_act_layer=nn.GELU,
        mlp_drop_rate: float = 0.0,
        gmlp=False,
        # =============================
        use_checkpoint: bool = False,
        post_norm: bool = False,
        # =============================
        token_merge=True, 
        token_prune=True,
        merge_layers: Optional[set]=None,
        prune_layers: Optional[set]=None,
        prune_threshold: float = 0.25,
        prune_sim: str = "cos",
        prune_imp: str = "l2",
        total_blocks=None,
        final_threshold=None,
        block_index: int = 0,
        stage_index: int = 0,
        # =============================
        _SS2D: type = SS2D,
        **kwargs,
    ):
        super().__init__()
        
        self.channel_first = channel_first
        self.ssm_branch = ssm_ratio > 0
        self.mlp_branch = mlp_ratio > 0
        self.use_checkpoint = use_checkpoint
        self.post_norm = post_norm
        self.token_merge = token_merge
        self.token_prune = token_prune
        self.total_blocks = total_blocks
        self.final_threshold = final_threshold
        self.block_index = block_index
        self.stage_index = stage_index

        
        enable_merge = self.token_merge and ((merge_layers is None) or (block_index in merge_layers))
        enable_prune = self.token_prune and ((prune_layers is None) or (block_index in prune_layers))

        # --- 프루닝 모듈 생성 ---
        if enable_prune:
            self.prune_module = LayerTokenPrune(
                dim=hidden_dim,
                norm_layer=norm_layer,
                layer_idx=self.block_index + 1,
                alpha_init=0.7,  # Δ 기반 threshold 조절값 
            )

        # --- 머징 모듈 생성 ---
        if enable_merge:
            self.merge_module = LayerTokenMerge(
                dim=hidden_dim,
                norm_layer=norm_layer,
                layer_idx=self.block_index + 1,
                merge_gate_hidden=64,
                tau_init=0.5,
            )

        if self.ssm_branch:
            self.norm = norm_layer(hidden_dim)
            self.op = _SS2D(
                d_model=hidden_dim, 
                d_state=ssm_d_state, 
                ssm_ratio=ssm_ratio,
                dt_rank=ssm_dt_rank,
                act_layer=ssm_act_layer,
                d_conv=ssm_conv,
                conv_bias=ssm_conv_bias,
                dropout=ssm_drop_rate,
                initialize=ssm_init,
                forward_type=forward_type,
                channel_first=channel_first,
            )
            if hasattr(self, "prune_module"):
                self.op.prune_module = self.prune_module
            if hasattr(self, "merge_module"):
                self.op.merge_module = self.merge_module
            self.op._layer_idx = self.block_index

        self.drop_path = DropPath(drop_path)
        
        if self.mlp_branch:
            _MLP = Mlp if not gmlp else gMlp
            self.norm2 = norm_layer(hidden_dim)
            mlp_hidden_dim = int(hidden_dim * mlp_ratio)
            self.mlp = _MLP(in_features=hidden_dim, hidden_features=mlp_hidden_dim, act_layer=mlp_act_layer, drop=mlp_drop_rate, channels_first=channel_first)

    def _forward(self, input: torch.Tensor, prev_features: Optional[list] = None):
        x = input

        def _match_like(x_in: torch.Tensor, y_like: torch.Tensor) -> torch.Tensor:
            if x_in.shape == y_like.shape:
                return x_in

            Bx, Cx, Hx, Wx = x_in.shape
            By, Cy, Hy, Wy = y_like.shape

            if Cx != Cy:
                proj = getattr(self, "_res_proj", None)
                if proj is None:
                    self._res_proj = nn.Conv2d(Cx, Cy, kernel_size=1, bias=False).to(x_in.device)
                x_in = self._res_proj(x_in)
                Cx = Cy

            if (Hx % Hy == 0) and (Wx % Wy == 0):
                sh, sw = Hx // Hy, Wx // Wy
                x_in = F.avg_pool2d(x_in, kernel_size=(sh, sw), stride=(sh, sw))
            else:
                x_in = F.interpolate(x_in, size=(Hy, Wy), mode="bilinear", align_corners=False)
            return x_in

        if self.ssm_branch:
            res = self.op(self.norm(x)) if not self.post_norm else self.norm(self.op(x))
            x = _match_like(x, res) + self.drop_path(res)
            
        if self.mlp_branch:
            res = self.mlp(self.norm2(x)) if not self.post_norm else self.norm2(self.mlp(x))
            x = _match_like(x, res) + self.drop_path(res)
            
        return x

    def forward(self, input: torch.Tensor, prev_features: Optional[list] = None):
        if self.use_checkpoint:
            return checkpoint.checkpoint(self._forward, input, prev_features)
        else:
            return self._forward(input, prev_features)



class VSSM(nn.Module):
    def __init__(
        self, 
        patch_size=4, 
        in_chans=3, 
        num_classes=1000, 
        depths=[2, 2, 9, 2], 
        dims=[96, 192, 384, 768], 
        # =========================
        ssm_d_state=16,
        ssm_ratio=2.0,
        ssm_dt_rank="auto",
        ssm_act_layer="silu",        
        ssm_conv=3,
        ssm_conv_bias=True,
        ssm_drop_rate=0.0, 
        ssm_init="v0",
        forward_type="v2",
        # =========================
        mlp_ratio=4.0,
        mlp_act_layer="gelu",
        mlp_drop_rate=0.0,
        gmlp=False,
        # =========================
        drop_path_rate=0.1, 
        patch_norm=True, 
        norm_layer="LN", # "BN", "LN2D"
        downsample_version: str = "v2", # "v1", "v2", "v3"
        patchembed_version: str = "v1", # "v1", "v2"
        use_checkpoint=False,  
        # =========================
        posembed=False,
        imgsize=224,
        _SS2D=SS2D,
        # =========================
        token_merge=True,
        token_prune: bool=True,
        final_threshold=0.1,      # ← 기본값 설정 (0~1 사이)

        merge_start: int = 0,

        merge_layers=None,
        prune_layers=None,
        prune_threshold: float = 0.25,
        prune_sim: str = "cos_delta",
        prune_imp: str = "l2",
        # =========================
        **kwargs,
    ):
        super().__init__()
        self.channel_first = (norm_layer.lower() in ["bn", "ln2d"])
        self.num_classes = num_classes
        self.num_layers = len(depths)
        if isinstance(dims, int):
            dims = [int(dims * 2 ** i_layer) for i_layer in range(self.num_layers)]
        self.num_features = dims[-1]
        self.dims = dims

        self.token_merge = token_merge
        self.token_prune = token_prune

        self.final_threshold = final_threshold

        self.merge_start = int(merge_start) 

        self.merge_layers = set(merge_layers) if merge_layers is not None else list(range(8, 20))
        self.prune_layers = set(prune_layers) if prune_layers is not None else list(range(0, 8))
        self.prune_threshold = float(prune_threshold)
        self.prune_sim = prune_sim
        self.prune_imp = prune_imp


        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule
        
        _NORMLAYERS = dict(
            ln=nn.LayerNorm,
            ln2d=LayerNorm2d,
            bn=nn.BatchNorm2d,
        )

        _ACTLAYERS = dict(
            silu=nn.SiLU, 
            gelu=nn.GELU, 
            relu=nn.ReLU, 
            sigmoid=nn.Sigmoid,
        )

        norm_layer: nn.Module = _NORMLAYERS.get(norm_layer.lower(), None)
        ssm_act_layer: nn.Module = _ACTLAYERS.get(ssm_act_layer.lower(), None)
        mlp_act_layer: nn.Module = _ACTLAYERS.get(mlp_act_layer.lower(), None)

        self.pos_embed = self._pos_embed(dims[0], patch_size, imgsize) if posembed else None

        _make_patch_embed = dict(
            v1=self._make_patch_embed, 
            v2=self._make_patch_embed_v2,
        ).get(patchembed_version, None)
        self.patch_embed = _make_patch_embed(in_chans, dims[0], patch_size, patch_norm, norm_layer, channel_first=self.channel_first)

        _make_downsample = dict(
            v1=PatchMerging2D, 
            v2=self._make_downsample, 
            v3=self._make_downsample_v3, 
            none=(lambda *_, **_k: None),
        ).get(downsample_version, None)

        self.layers = nn.ModuleList()
        global_block_index = 0
        for i_layer in range(self.num_layers):
            downsample = _make_downsample(
                self.dims[i_layer], 
                self.dims[i_layer + 1], 
                norm_layer=norm_layer,
                channel_first=self.channel_first,
            ) if (i_layer < self.num_layers - 1) else nn.Identity()

            total_blocks = sum(depths)   # 네트워크 전체 블록 수
            final_threshold = final_threshold  # config.py에 새로 추가한 값 (0~1 사이)

            layer=self._make_layer(
                dim = self.dims[i_layer],
                drop_path = dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                use_checkpoint=use_checkpoint,
                norm_layer=norm_layer,
                downsample=downsample,
                channel_first=self.channel_first,
                # =================
                ssm_d_state=ssm_d_state,
                ssm_ratio=ssm_ratio,
                ssm_dt_rank=ssm_dt_rank,
                ssm_act_layer=ssm_act_layer,
                ssm_conv=ssm_conv,
                ssm_conv_bias=ssm_conv_bias,
                ssm_drop_rate=ssm_drop_rate,
                ssm_init=ssm_init,
                forward_type=forward_type,
                # =================
                mlp_ratio=mlp_ratio,
                mlp_act_layer=mlp_act_layer,
                mlp_drop_rate=mlp_drop_rate,
                gmlp=gmlp,
                # =================
                _SS2D=_SS2D,
                # =================
                token_merge=token_merge,
                token_prune=token_prune,
                total_blocks=total_blocks,
                final_threshold=final_threshold,
                stage_index=i_layer,
                block_index_offset=global_block_index,

                merge_start=self.merge_start,
                merge_layers=self.merge_layers,
                prune_layers=self.prune_layers,
                prune_threshold=self.prune_threshold,
                prune_sim=self.prune_sim,
                prune_imp=self.prune_imp,
            )
            global_block_index += depths[i_layer]
            self.layers.append(layer)

        self.classifier = nn.Sequential(OrderedDict(
            norm=norm_layer(self.num_features), # B,H,W,C
            permute=(Permute(0, 3, 1, 2) if not self.channel_first else nn.Identity()),
            avgpool=nn.AdaptiveAvgPool2d(1),
            flatten=nn.Flatten(1),
            head=nn.Linear(self.num_features, num_classes),
        ))
        self.head = nn.Linear(self.num_features, num_classes)  # 추가

        self.apply(self._init_weights)


    @staticmethod
    def _pos_embed(embed_dims, patch_size, img_size):
        patch_height, patch_width = (img_size // patch_size, img_size // patch_size)
        pos_embed = nn.Parameter(torch.zeros(1, embed_dims, patch_height, patch_width))
        trunc_normal_(pos_embed, std=0.02)
        return pos_embed

    def _init_weights(self, m: nn.Module):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    # used in building optimizer
    @torch.jit.ignore
    def no_weight_decay(self):
        return {"pos_embed"}

    # used in building optimizer
    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {}

    @staticmethod
    def _make_patch_embed(in_chans=3, embed_dim=96, patch_size=4, patch_norm=True, norm_layer=nn.LayerNorm, channel_first=False):
        # if channel first, then Norm and Output are both channel_first
        return nn.Sequential(
            nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size, bias=True),
            (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
            (norm_layer(embed_dim) if patch_norm else nn.Identity()),
        )

    @staticmethod
    def _make_patch_embed_v2(in_chans=3, embed_dim=96, patch_size=4, patch_norm=True, norm_layer=nn.LayerNorm, channel_first=False):
        # if channel first, then Norm and Output are both channel_first
        stride = patch_size // 2
        kernel_size = stride + 1
        padding = 1
        return nn.Sequential(
            nn.Conv2d(in_chans, embed_dim // 2, kernel_size=kernel_size, stride=stride, padding=padding),
            (nn.Identity() if (channel_first or (not patch_norm)) else Permute(0, 2, 3, 1)),
            (norm_layer(embed_dim // 2) if patch_norm else nn.Identity()),
            (nn.Identity() if (channel_first or (not patch_norm)) else Permute(0, 3, 1, 2)),
            nn.GELU(),
            nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=kernel_size, stride=stride, padding=padding),
            (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
            (norm_layer(embed_dim) if patch_norm else nn.Identity()),
        )
    
    @staticmethod
    def _make_downsample(dim=96, out_dim=192, norm_layer=nn.LayerNorm, channel_first=False):
        # if channel first, then Norm and Output are both channel_first
        return nn.Sequential(
            (nn.Identity() if channel_first else Permute(0, 3, 1, 2)),
            nn.Conv2d(dim, out_dim, kernel_size=2, stride=2),
            (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
            norm_layer(out_dim),
        )

    @staticmethod
    def _make_downsample_v3(dim=96, out_dim=192, norm_layer=nn.LayerNorm, channel_first=False):
        # if channel first, then Norm and Output are both channel_first
        return nn.Sequential(
            (nn.Identity() if channel_first else Permute(0, 3, 1, 2)),
            nn.Conv2d(dim, out_dim, kernel_size=3, stride=2, padding=1),
            (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
            norm_layer(out_dim),
        )

    @staticmethod
    def _make_layer(
        dim=96, 
        drop_path=[0.1, 0.1], 
        use_checkpoint=False, 
        norm_layer=nn.LayerNorm,
        downsample=nn.Identity(),
        channel_first=False,
        # ===========================
        ssm_d_state=16,
        ssm_ratio=2.0,
        ssm_dt_rank="auto",       
        ssm_act_layer=nn.SiLU,
        ssm_conv=3,
        ssm_conv_bias=True,
        ssm_drop_rate=0.0, 
        ssm_init="v0",
        forward_type="v2",
        # ===========================
        mlp_ratio=4.0,
        mlp_act_layer=nn.GELU,
        mlp_drop_rate=0.0,
        gmlp=False,
        # ===========================
        _SS2D=SS2D,
        # ===========================
        token_merge=True,
        token_prune=True,
        total_blocks=None,
        final_threshold=None,
        stage_index=0,
        block_index_offset=0,
        merge_start=0,
        merge_layers=None,
        prune_layers=None,
        prune_threshold=0.25,
        prune_sim="cos_delta",
        prune_imp="l2",
        **kwargs,
    ):
        depth = len(drop_path)
        blocks = nn.ModuleList()

        for d in range(depth):
            global_block_idx = block_index_offset + d
            token_merge_this = bool(token_merge and (global_block_idx >= merge_start))

            blk = VSSBlock(
                hidden_dim=dim, 
                drop_path=drop_path[d],
                norm_layer=norm_layer,
                channel_first=channel_first,
                ssm_d_state=ssm_d_state,
                ssm_ratio=ssm_ratio,
                ssm_dt_rank=ssm_dt_rank,
                ssm_act_layer=ssm_act_layer,
                ssm_conv=ssm_conv,
                ssm_conv_bias=ssm_conv_bias,
                ssm_drop_rate=ssm_drop_rate,
                ssm_init=ssm_init,
                forward_type=forward_type,
                mlp_ratio=mlp_ratio,
                mlp_act_layer=mlp_act_layer,
                mlp_drop_rate=mlp_drop_rate,
                gmlp=gmlp,
                use_checkpoint=use_checkpoint,
                _SS2D=_SS2D,

                # ✅ Token Merge 설정
                token_merge=token_merge_this,
                token_prune=token_prune,
                total_blocks=total_blocks,
                final_threshold=final_threshold,
                block_index=global_block_idx,
                stage_index=stage_index,
                merge_layers=merge_layers,
                prune_layers=prune_layers,
                prune_threshold=prune_threshold,
                prune_sim=prune_sim,
                prune_imp=prune_imp,
            )
            blocks.append(blk)

        # ✅ Sequential → nn.Module 로 교체 (forward 포함)
        class Layer(nn.Module):
            def __init__(self, blocks, downsample):
                super().__init__()
                self.blocks = blocks
                self.downsample = downsample

            def forward(self, x):
                for blk in self.blocks:
                    x = blk(x)
                x = self.downsample(x)
                return x

        return Layer(blocks, downsample)



    def forward(self, x: torch.Tensor):
        x = self.patch_embed(x)
        if self.pos_embed is not None:
            pos_embed = self.pos_embed.permute(0, 2, 3, 1) if not self.channel_first else self.pos_embed
            x = x + pos_embed
        for layer in self.layers:
            x = layer(x)

        # Feature 추출
        feat = x
        if not self.channel_first:
            feat = feat.mean(dim=[1, 2])  # B, C
        else:
            feat = F.adaptive_avg_pool2d(feat, 1).squeeze(-1).squeeze(-1)  # B, C

        # Classification head
        logits = self.head(feat)

        return logits, feat
    
    def flops(self, shape=(3, 224, 224), verbose=True):
        # shape = self.__input_shape__[1:]
        supported_ops={
            "aten::silu": None, # as relu is in _IGNORED_OPS
            "aten::neg": None, # as relu is in _IGNORED_OPS
            "aten::exp": None, # as relu is in _IGNORED_OPS
            "aten::flip": None, # as permute is in _IGNORED_OPS
            # "prim::PythonOp.CrossScan": None,
            # "prim::PythonOp.CrossMerge": None,
            "prim::PythonOp.SelectiveScanCuda": partial(selective_scan_flop_jit, backend="prefixsum", verbose=verbose),
        }

        model = copy.deepcopy(self)
        model.cuda().eval()

        input = torch.randn((1, *shape), device=next(model.parameters()).device)
        params = parameter_count(model)[""]
        Gflops, unsupported = flop_count(model=model, inputs=(input,), supported_ops=supported_ops)

        del model, input
        return sum(Gflops.values()) * 1e9
        return f"params {params} GFLOPs {sum(Gflops.values())}"


    # used to load ckpt from previous training code
    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):

        def check_name(src, state_dict: dict = state_dict, strict=False):
            if strict:
                if prefix + src in list(state_dict.keys()):
                    return True
            else:
                key = prefix + src
                for k in list(state_dict.keys()):
                    if k.startswith(key):
                        return True
            return False

        def change_name(src, dst, state_dict: dict = state_dict, strict=False):
            if strict:
                if prefix + src in list(state_dict.keys()):
                    state_dict[prefix + dst] = state_dict[prefix + src]
                    state_dict.pop(prefix + src)
            else:
                key = prefix + src
                for k in list(state_dict.keys()):
                    if k.startswith(key):
                        new_k = prefix + dst + k[len(key):]
                        state_dict[new_k] = state_dict[k]
                        state_dict.pop(k)

        if check_name("pos_embed", strict=True):
            srcEmb: torch.Tensor = state_dict[prefix + "pos_embed"]
            state_dict[prefix + "pos_embed"] = F.interpolate(srcEmb.float(), size=self.pos_embed.shape[2:4], align_corners=False, mode="bicubic").to(srcEmb.device)

        change_name("patch_embed.proj", "patch_embed.0")
        change_name("patch_embed.norm", "patch_embed.2")
        for i in range(100):
            for j in range(100):
                change_name(f"layers.{i}.blocks.{j}.ln_1", f"layers.{i}.blocks.{j}.norm")
                change_name(f"layers.{i}.blocks.{j}.self_attention", f"layers.{i}.blocks.{j}.op")
        change_name("norm", "classifier.norm")
        change_name("head", "classifier.head")

        return super()._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)


# compatible with openmmlab
class Backbone_VSSM(VSSM):
    def __init__(self, out_indices=(0, 1, 2, 3), pretrained=None, norm_layer="ln", **kwargs):
        kwargs.update(norm_layer=norm_layer)
        super().__init__(**kwargs)
        self.channel_first = (norm_layer.lower() in ["bn", "ln2d"])
        _NORMLAYERS = dict(
            ln=nn.LayerNorm,
            ln2d=LayerNorm2d,
            bn=nn.BatchNorm2d,
        )
        norm_layer: nn.Module = _NORMLAYERS.get(norm_layer.lower(), None)        
        
        self.out_indices = out_indices
        for i in out_indices:
            layer = norm_layer(self.dims[i])
            layer_name = f'outnorm{i}'
            self.add_module(layer_name, layer)

        del self.classifier
        self.load_pretrained(pretrained)

    def load_pretrained(self, ckpt=None, key="model"):
        if ckpt is None:
            return
        
        try:
            _ckpt = torch.load(open(ckpt, "rb"), map_location=torch.device("cpu"))
            print(f"Successfully load ckpt {ckpt}")
            incompatibleKeys = self.load_state_dict(_ckpt[key], strict=False)
            print(incompatibleKeys)        
        except Exception as e:
            print(f"Failed loading checkpoint form {ckpt}: {e}")

    def forward(self, x):
        def layer_forward(l, x):
            x = l.blocks(x)
            y = l.downsample(x)
            return x, y

        x = self.patch_embed(x)
        outs = []
        for i, layer in enumerate(self.layers):
            o, x = layer_forward(layer, x) # (B, H, W, C)
            if i in self.out_indices:
                norm_layer = getattr(self, f'outnorm{i}')
                out = norm_layer(o)
                if not self.channel_first:
                    out = out.permute(0, 3, 1, 2)
                outs.append(out.contiguous())

        if len(self.out_indices) == 0:
            return x
        
        return outs


# =====================================================
def vanilla_vmamba_tiny():
    return VSSM(
        depths=[2, 2, 9, 2], dims=96, drop_path_rate=0.2, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=True, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v0", 
        mlp_ratio=0.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln", 
        downsample_version="v1", patchembed_version="v1", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vanilla_vmamba_small():
    return VSSM(
        depths=[2, 2, 27, 2], dims=96, drop_path_rate=0.3, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=True, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v0", 
        mlp_ratio=0.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln", 
        downsample_version="v1", patchembed_version="v1", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vanilla_vmamba_base():
    return VSSM(
        depths=[2, 2, 27, 2], dims=128, drop_path_rate=0.6, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=True, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v0", 
        mlp_ratio=0.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln", 
        downsample_version="v1", patchembed_version="v1", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


# =====================================================
def vmamba_tiny_s2l5(channel_first=True):
    return VSSM(
        depths=[2, 2, 5, 2], dims=96, drop_path_rate=0.2, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=1, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v05_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer=("ln2d" if channel_first else "ln"), 
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vmamba_small_s2l15(channel_first=True):
    return VSSM(
        depths=[2, 2, 15, 2], dims=96, drop_path_rate=0.3, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=1, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v05_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer=("ln2d" if channel_first else "ln"), 
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vmamba_base_s2l15(channel_first=True):
    return VSSM(
        depths=[2, 2, 15, 2], dims=128, drop_path_rate=0.6, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=1, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v05_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer=("ln2d" if channel_first else "ln"), 
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


# =====================================================
def vmamba_tiny_s1l8(channel_first=True):
    return VSSM(
        depths=[2, 2, 8, 2], dims=96, drop_path_rate=0.2, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=1, ssm_ratio=1.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v05_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer=("ln2d" if channel_first else "ln"), 
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vmamba_small_s1l20(channel_first=True):
    return VSSM(
        depths=[2, 2, 20, 2], dims=96, drop_path_rate=0.3, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=1, ssm_ratio=1.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v05_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer=("ln2d" if channel_first else "ln"), 
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vmamba_base_s1l20(channel_first=True):
    return VSSM(
        depths=[2, 2, 20, 2], dims=128, drop_path_rate=0.5, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=1, ssm_ratio=1.0, ssm_dt_rank="auto", ssm_act_layer="silu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v0", forward_type="v05_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer=("ln2d" if channel_first else "ln"), 
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


# mamba2 support =====================================================
# FLOPS count do not work now for mamba2!
def vmamba_tiny_m2():
    return VSSM(
        depths=[2, 2, 4, 2], dims=96, drop_path_rate=0.2, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=64, ssm_ratio=1.0, ssm_dt_rank="auto", ssm_act_layer="gelu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v2", forward_type="m0_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln",
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vmamba_small_m2():
    return VSSM(
        depths=[2, 2, 12, 2], dims=96, drop_path_rate=0.3, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=64, ssm_ratio=1.0, ssm_dt_rank="auto", ssm_act_layer="gelu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v2", forward_type="m0_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln",
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


def vmamba_base_m2():
    return VSSM(
        depths=[2, 2, 12, 2], dims=128, drop_path_rate=0.3, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=64, ssm_ratio=1.0, ssm_dt_rank="auto", ssm_act_layer="gelu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v2", forward_type="m0_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln",
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )


if __name__ == "__main__":
    model_ref = vmamba_tiny_s1l8()

    model = VSSM(
        depths=[2, 2, 15, 2], dims=128, drop_path_rate=0.2, 
        patch_size=4, in_chans=3, num_classes=1000, 
        ssm_d_state=64, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="gelu",
        ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0, 
        ssm_init="v2", forward_type="m0_noz", 
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        patch_norm=True, norm_layer="ln",
        downsample_version="v3", patchembed_version="v2", 
        use_checkpoint=False, posembed=False, imgsize=224, 
    )
    print(parameter_count(model)[""])
    print(model.flops()) # wrong
    model.cuda().train()
    model_ref.cuda().train()

    def bench(model):
        import time
        inp = torch.randn((128, 3, 224, 224)).cuda()
        for _ in range(30):
            model(inp)
        torch.cuda.synchronize()
        tim = time.time()
        for _ in range(30):
            model(inp)
        torch.cuda.synchronize()
        tim1 = time.time() - tim

        for _ in range(30):
            model(inp).sum().backward()
        torch.cuda.synchronize()
        tim = time.time()
        for _ in range(30):
            model(inp).sum().backward()
        torch.cuda.synchronize()
        tim2 = time.time() - tim

        return tim1 / 30, tim2 / 30
    
    print(bench(model_ref))
    print(bench(model))

    breakpoint()



def save_network(network, model_dir, name):
    save_path = os.path.join(model_dir, name + ".pth")
    torch.save(network.cpu().state_dict(), save_path)
    
    if torch.cuda.is_available():
        network.cuda()

class TripletLoss(nn.Module):
    def __init__(self, margin=0.3, mutual_flag = False):
        super(TripletLoss, self).__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)
        self.mutual = mutual_flag

    def forward(self, inputs, targets):
        n = inputs.size(0)

        # float32로 변환하여 dtype mismatch 방지
        inputs = inputs.float()

        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(1, -2, inputs, inputs.t())
        dist = dist.clamp(min=1e-12).sqrt()  # for numerical stability

        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []
        for i in range(n):
            ap = dist[i][mask[i]].max().unsqueeze(0)

            # negative sample이 없을 경우 예외 처리
            if (mask[i] == 0).sum() == 0:
                an = torch.tensor([0.0], device=inputs.device)  # 혹은 float("inf")도 가능
            else:
                an = dist[i][mask[i] == 0].min().unsqueeze(0)

            dist_ap.append(ap)
            dist_an.append(an)

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)

        y = torch.ones_like(dist_an)
        loss = self.ranking_loss(dist_an, dist_ap, y)
        if self.mutual:
            return loss, dist
        return loss


class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes=62, epsilon=0.05, use_gpu=True):
        super(CrossEntropyLabelSmooth, self).__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.use_gpu = use_gpu
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets = torch.zeros(log_probs.size()).scatter_(1, targets.unsqueeze(1).data.cpu(), 1)
        if self.use_gpu: targets = targets.cuda()
        targets = (1 - self.epsilon) * targets + self.epsilon / self.num_classes
        loss = (- targets * log_probs).mean(0).sum()
        return loss


In [ ]:
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
for k in [k for k in sys.modules if k.startswith("tevmamba")]:
    del sys.modules[k]                       # re-import cleanly if the cell is re-run
from tevmamba.backbones import vmamba as vmamba_mod
from tevmamba.backbones import csms6s
from tevmamba.backbones.vmamba import VSSM
print(f"selective scan backend: {'CUDA oflex kernel' if csms6s.WITH_SELECTIVESCAN_OFLEX else 'chunked PyTorch fallback (slow)'} | "
      f"triton cross-scan: {vmamba_mod.__dict__.get('cross_scan_fn') is not None and __import__('tevmamba.backbones.csm_triton', fromlist=['x']).WITH_TRITON}")

## Data preparation
The notebook finds the attached Kaggle datasets by their folder signature and builds the layout the repo's loaders
expect under `DATA_ROOT`:
```
DATA_ROOT/market1501/{bounding_box_train, query, bounding_box_test}       <- symlinks to the Kaggle input
DATA_ROOT/dukemtmcreid/{...}                                               <- symlinks
DATA_ROOT/Occluded_Duke/{...}                                              <- built from DukeMTMC + official lists
DATA_ROOT/Occluded_REID/{train/<pid>, test/query/<pid>, test/gallery/<pid>}/*.jpg   <- converted from the zip
```

In [ ]:
def _list_dirs(root, max_depth=11):
    out, base = [], osp.abspath(root).rstrip("/").count("/")
    for cur, dirs, _ in os.walk(root, followlinks=True):
        if cur.rstrip("/").count("/") - base >= max_depth:
            dirs[:] = []
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__MACOSX"]
        out.append(cur)
    return out


ALL_INPUT_DIRS = _list_dirs(INPUT_ROOT) if osp.isdir(INPUT_ROOT) else []
print(f"scanned {len(ALL_INPUT_DIRS)} directories under {INPUT_ROOT}")


def _has_images(d, exts=(".jpg", ".jpeg", ".png")):
    try:
        return any(e.name.lower().endswith(exts) for e in os.scandir(d))
    except OSError:
        return False


def _find(pred, keyword=None):
    c = [d for d in ALL_INPUT_DIRS if (keyword is None or keyword in d.lower()) and pred(d)]
    return sorted(c, key=lambda p: (p.count("/"), len(p)))[0] if c else None


def _market_style(d):
    return all(_has_images(osp.join(d, s)) for s in ("bounding_box_train", "query", "bounding_box_test"))


def _symlink(src, dst):
    if osp.islink(dst):
        os.unlink(dst)
    elif osp.exists(dst):
        return
    os.makedirs(osp.dirname(dst), exist_ok=True)
    os.symlink(src, dst)


def _download(url, out, sha256=None, timeout=300):
    if not osp.isfile(out):
        print(f"   downloading {url}")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (TE-VMamba reproduction notebook)"})
        with urllib.request.urlopen(req, timeout=timeout) as r, open(out + ".part", "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        os.replace(out + ".part", out)
    if sha256 and hashlib.sha256(open(out, "rb").read()).hexdigest() != sha256:
        os.remove(out)
        raise RuntimeError(f"sha256 mismatch for {out}")
    return out


PREPARED, PREP_NOTES = {}, {}

# ---- Market-1501 / DukeMTMC-reID ---------------------------------------------------------------------------------------
for name, kw in (("market1501", "market"), ("dukemtmcreid", "duke")):
    src = DATA_OVERRIDES.get(name) or _find(lambda d: _market_style(d) and "occluded" not in d.lower(), kw)
    if src:
        for s in ("bounding_box_train", "query", "bounding_box_test"):
            _symlink(osp.join(src, s), osp.join(DATA_ROOT, name, s))
        PREPARED[name] = osp.join(DATA_ROOT, name)
        print(f"[data] {name:14s} <- {src}")
if "dukemtmcreid" in PREPARED:
    PREPARED["dukemtmc"] = PREPARED["dukemtmcreid"]

# ---- Occluded-Duke: official name lists applied to DukeMTMC-reID (== lightas/convert_duke_to_occduke.py) --------------
OCCDUKE_LIST_URL = "https://raw.githubusercontent.com/lightas/Occluded-DukeMTMC-Dataset/master/Occluded_Duke/{}.list"


def build_occluded_duke(duke_dir, out_dir):
    if _market_style(out_dir) and len(os.listdir(osp.join(out_dir, "query"))) == 2210:
        return "already built"
    lists = {}
    for split in ("train", "query", "gallery"):
        attached = _find(lambda d: osp.isfile(osp.join(d, f"{split}.list")))
        p = osp.join(attached, f"{split}.list") if attached else _download(
            OCCDUKE_LIST_URL.format(split), osp.join(CACHE_DIR, f"occduke_{split}.list"))
        lists[split] = [l.strip() for l in open(p) if l.strip()]
    missing = 0
    for split, folder in (("train", "bounding_box_train"), ("gallery", "bounding_box_test"), ("query", "query")):
        os.makedirs(osp.join(out_dir, folder), exist_ok=True)
        for img in lists[split]:
            src = osp.join(duke_dir, folder, img)
            if not osp.isfile(src):                       # image moved to another split by the re-split
                src = osp.join(duke_dir, "bounding_box_test", img)
            if not osp.isfile(src):
                src = osp.join(duke_dir, "query", img)
            if osp.isfile(src):
                _symlink(src, osp.join(out_dir, folder, img))
            else:
                missing += 1
    return f"built (train {len(lists['train'])}, query {len(lists['query'])}, gallery {len(lists['gallery'])}, missing {missing})"


if "dukemtmcreid" in PREPARED:
    try:
        msg = build_occluded_duke(osp.realpath(osp.join(PREPARED["dukemtmcreid"], "bounding_box_train", "..")),
                                  osp.join(DATA_ROOT, "Occluded_Duke"))
        PREPARED["occ_duke"] = osp.join(DATA_ROOT, "Occluded_Duke")
        print(f"[data] occ_duke       <- DukeMTMC-reID + official lists: {msg}")
    except Exception as e:  # noqa: BLE001
        print(f"!! Occluded-Duke could not be built: {type(e).__name__}: {e}")

# ---- Occluded-REID: official zip -> repo layout ------------------------------------------------------------------------
OCCREID_URL = ("https://github.com/tinajia2012/ICME2018_Occluded-Person-Reidentification_datasets/raw/master/"
               "Occluded_REID.zip")
OCCREID_SHA256 = "1c8854e02db775da6a51399df4b7e02c299804850dff9985451d610fc8c6d57d"
OCCREID_SPLIT_SEED = 0


def build_occluded_reid(out_dir):
    if osp.isdir(osp.join(out_dir, "test", "gallery")) and len(os.listdir(osp.join(out_dir, "test", "gallery"))) == 100:
        return "already built"
    zpath = next((osp.join(d, "Occluded_REID.zip") for d in ALL_INPUT_DIRS if osp.isfile(osp.join(d, "Occluded_REID.zip"))), None)
    zpath = zpath or _download(OCCREID_URL, osp.join(CACHE_DIR, "Occluded_REID.zip"), OCCREID_SHA256)
    tmp = out_dir + ".partial"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zpath) as z:
        names = [n for n in z.namelist() if n.lower().endswith((".tif", ".tiff", ".jpg", ".png"))]
        pids = sorted({n.split("/")[-2] for n in names})
        rng = random.Random(OCCREID_SPLIT_SEED)
        train_ids = set(rng.sample(pids, len(pids) // 2))
        for n in names:
            kind, pid, fname = n.split("/")[-3:]
            occluded = kind.startswith("occluded")
            if pid in train_ids:
                sub = osp.join("train", pid)
            else:
                sub = osp.join("test", "query" if occluded else "gallery", pid)
            os.makedirs(osp.join(tmp, sub), exist_ok=True)
            stem = ("occ_" if occluded else "whole_") + osp.splitext(fname)[0]
            with z.open(n) as f:
                Image.open(f).convert("RGB").save(osp.join(tmp, sub, stem + ".jpg"), quality=95)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.replace(tmp, out_dir)
    return f"built ({len(pids)} IDs: {len(train_ids)} train / {len(pids) - len(train_ids)} test, seed {OCCREID_SPLIT_SEED})"


if "occ_reid" in DATASETS_TO_RUN:
    try:
        msg = build_occluded_reid(osp.join(DATA_ROOT, "Occluded_REID"))
        PREPARED["occ_reid"] = osp.join(DATA_ROOT, "Occluded_REID")
        print(f"[data] occ_reid       <- official zip: {msg}")
    except Exception as e:  # noqa: BLE001
        print(f"!! Occluded-REID could not be prepared: {type(e).__name__}: {e}")

missing = [d for d in DATASETS_TO_RUN if d not in PREPARED]
print("\nprepared:", sorted(k for k in PREPARED if k != "dukemtmcreid"))
if missing:
    print(f"!! not available, will be skipped: {missing}")

## Datasets, sampler, transforms (port of `datasets/*.py`)
* Market-1501 / DukeMTMC / Occluded-Duke: filenames `PPPP_cC...jpg` → (pid, camera). Train IDs are relabelled to 0..N−1.
* Occluded-REID: one folder per identity. **Query = camera 0, gallery = camera 1**, so the standard evaluator
  (which ignores same-ID *and* same-camera gallery hits) keeps every true match.
* PK sampler: 16 identities × 4 images = batch 64.
* Train transform: Resize 224×224 (bicubic), flip 0.5, pad 10 + random crop, ImageNet normalisation, random erasing
  0.3 (pixel mode).

Only change from the repo: the `PPPP_cC` pattern is applied to the file name, not the full path (identical results;
avoids accidental matches on digits in Kaggle paths).

In [ ]:
def read_image(img_path):
    if not osp.exists(img_path):
        raise IOError(f"{img_path} does not exist")
    return Image.open(img_path).convert("RGB")


class BaseImageDataset:
    def get_imagedata_info(self, data):
        pids = {d[1] for d in data}; cams = {d[2] for d in data}; tracks = {d[3] for d in data}
        return len(pids), len(data), len(cams), len(tracks)

    def print_dataset_statistics(self, train, query, gallery, tag):
        print(f"=> {tag} loaded\n  subset   | # ids | # images | # cameras")
        for n, d in (("train", train), ("query", query), ("gallery", gallery)):
            p, i, c, _ = self.get_imagedata_info(d)
            print(f"  {n:8s} | {p:5d} | {i:8d} | {c:9d}")

    def _finish(self, train, query, gallery, tag, verbose):
        self.train, self.query, self.gallery = train, query, gallery
        if verbose:
            self.print_dataset_statistics(train, query, gallery, tag)
        self.num_train_pids, self.num_train_imgs, self.num_train_cams, self.num_train_vids = self.get_imagedata_info(train)


class _MarketLike(BaseImageDataset):
    """market1501.py / dukemtmcreid.py / occ_duke.py share this parser."""
    dataset_dir, tag, max_cam, skip_junk = "", "", 8, False

    def __init__(self, root="", verbose=True, pid_begin=0, **kw):
        self.pid_begin = pid_begin
        d = osp.join(root, self.dataset_dir)
        tr = self._process_dir(osp.join(d, "bounding_box_train"), relabel=True)
        q = self._process_dir(osp.join(d, "query"), relabel=False)
        g = self._process_dir(osp.join(d, "bounding_box_test"), relabel=False)
        self._finish(tr, q, g, self.tag, verbose)

    def _process_dir(self, dir_path, relabel=False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d)")
        parsed = [(p, *map(int, pattern.search(osp.basename(p)).groups())) for p in img_paths]
        if self.skip_junk:
            parsed = [x for x in parsed if x[1] != -1]
        pid2label = {pid: lab for lab, pid in enumerate(sorted({x[1] for x in parsed}))}
        out = []
        for p, pid, camid in parsed:
            assert 1 <= camid <= self.max_cam
            out.append((p, self.pid_begin + (pid2label[pid] if relabel else pid), camid - 1, 1))
        return out


class Market1501(_MarketLike):
    dataset_dir, tag, max_cam, skip_junk = "market1501", "Market-1501", 6, True


class DukeMTMCreID(_MarketLike):
    dataset_dir, tag = "dukemtmcreid", "DukeMTMC-reID"


class OCC_DukeMTMCreID(_MarketLike):
    dataset_dir, tag = "Occluded_Duke", "Occluded-DukeMTMC"


class OCC_ReID(BaseImageDataset):
    dataset_dir = "Occluded_REID"

    def __init__(self, root="", verbose=True, pid_begin=0, **kw):
        d = osp.join(root, self.dataset_dir)
        tr = self._process(osp.join(d, "train"), "train")
        q = self._process(osp.join(d, "test/query"), "query")
        g = self._process(osp.join(d, "test/gallery"), "gallery")
        self._finish(tr, q, g, "Occluded-ReID", verbose)

    def _process(self, folder, mode):
        pid_dirs = sorted(os.listdir(folder))
        pid2label = {pid: idx for idx, pid in enumerate(pid_dirs)}
        out = []
        for pid in pid_dirs:
            if not osp.isdir(osp.join(folder, pid)):
                continue
            camid = 1 if mode == "gallery" else 0
            for img in sorted(glob.glob(osp.join(folder, pid, "*.jpg"))):
                out.append((img, pid2label[pid], camid, 1))
        return out


FACTORY = {"market1501": Market1501, "dukemtmc": DukeMTMCreID, "occ_duke": OCC_DukeMTMCreID, "occ_reid": OCC_ReID}


class ImageDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset, self.transform = dataset, transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        img_path, pid, camid, trackid = self.dataset[index]
        img = read_image(img_path)
        if self.transform is not None:
            img = self.transform(img)
        return img, pid, camid, trackid, img_path.split("/")[-1]


class RandomIdentitySampler(Sampler):
    """N identities x K instances per batch (verbatim logic of datasets/sampler.py)."""
    def __init__(self, data_source, batch_size, num_instances):
        self.data_source, self.batch_size, self.num_instances = data_source, batch_size, num_instances
        self.num_pids_per_batch = batch_size // num_instances
        self.index_dic = defaultdict(list)
        for index, (_, pid, _, _) in enumerate(data_source):
            self.index_dic[pid].append(index)
        self.pids = list(self.index_dic.keys())
        self.length = 0
        for pid in self.pids:
            num = max(len(self.index_dic[pid]), num_instances)
            self.length += num - num % num_instances

    def __iter__(self):
        batch_idxs_dict = defaultdict(list)
        for pid in self.pids:
            idxs = copy.deepcopy(self.index_dic[pid])
            if len(idxs) < self.num_instances:
                idxs = np.random.choice(idxs, size=self.num_instances, replace=True)
            random.shuffle(idxs)
            batch = []
            for idx in idxs:
                batch.append(idx)
                if len(batch) == self.num_instances:
                    batch_idxs_dict[pid].append(batch); batch = []
        avai_pids, final_idxs = copy.deepcopy(self.pids), []
        while len(avai_pids) >= self.num_pids_per_batch:
            for pid in random.sample(avai_pids, self.num_pids_per_batch):
                final_idxs.extend(batch_idxs_dict[pid].pop(0))
                if len(batch_idxs_dict[pid]) == 0:
                    avai_pids.remove(pid)
        return iter(final_idxs)

    def __len__(self):
        return self.length


def train_collate_fn(batch):
    imgs, pids, camids, viewids, _ = zip(*batch)
    return (torch.stack(imgs, 0), torch.tensor(pids, dtype=torch.int64), torch.tensor(camids, dtype=torch.int64),
            torch.tensor(viewids, dtype=torch.int64))


def val_collate_fn(batch):
    imgs, pids, camids, viewids, img_paths = zip(*batch)
    return (torch.stack(imgs, 0), pids, camids, torch.tensor(camids, dtype=torch.int64),
            torch.tensor(viewids, dtype=torch.int64), img_paths)


def _quick_eval_subset(query, gallery, seed=0):
    rng = random.Random(seed)
    q = rng.sample(query, min(len(query), QUICK["eval_max_query"]))
    qp = {x[1] for x in q}
    pos = [x for x in gallery if x[1] in qp]; neg = [x for x in gallery if x[1] not in qp]
    return q, pos + rng.sample(neg, max(0, min(len(neg), QUICK["eval_max_gallery"] - len(pos))))


def make_dataloader(cfg):
    train_transforms = T.Compose([
        T.Resize(cfg.INPUT.SIZE_TRAIN, interpolation=T.InterpolationMode.BICUBIC),   # == interpolation=3
        T.RandomHorizontalFlip(p=cfg.INPUT.PROB),
        T.Pad(cfg.INPUT.PADDING),
        T.RandomCrop(cfg.INPUT.SIZE_TRAIN),
        T.ToTensor(),
        T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD),
        RandomErasing(probability=cfg.INPUT.RE_PROB, mode="pixel", max_count=1, device="cpu"),
    ])
    val_transforms = T.Compose([T.Resize(cfg.INPUT.SIZE_TEST), T.ToTensor(),
                                T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD)])
    dataset = FACTORY[cfg.DATASETS.NAMES](root=cfg.DATASETS.ROOT_DIR)
    query, gallery = dataset.query, dataset.gallery
    if RUN_MODE == "quick":
        query, gallery = _quick_eval_subset(query, gallery)
    nw = cfg.DATALOADER.NUM_WORKERS
    train_loader = DataLoader(ImageDataset(dataset.train, train_transforms), batch_size=cfg.SOLVER.IMS_PER_BATCH,
                              sampler=RandomIdentitySampler(dataset.train, cfg.SOLVER.IMS_PER_BATCH, cfg.DATALOADER.NUM_INSTANCE),
                              num_workers=nw, collate_fn=train_collate_fn, pin_memory=True, persistent_workers=nw > 0)
    val_loader = DataLoader(ImageDataset(query + gallery, val_transforms), batch_size=cfg.TEST.IMS_PER_BATCH,
                            shuffle=False, num_workers=nw, collate_fn=val_collate_fn, pin_memory=True)
    return train_loader, val_loader, len(query), dataset.num_train_pids, dataset

## ReID model (port of `model/make_model.py`)
The VSSM backbone feeds global average pooling, then a BNNeck (BatchNorm1d), then a linear ID classifier.
* Training returns (ID logits, pooled feature). Triplet loss uses the feature *before* the BNNeck.
* Testing returns the pooled feature *before* the BNNeck (`TEST.NECK_FEAT = before`), L2-normalised, cosine distance.

**ImageNet initialisation.** The repo's configs say `PRETRAIN_CHOICE: imagenet` but leave `PRETRAIN_PATH` empty, so
the released configs actually train from random weights. Here the official **VMamba-T [s1l8] ImageNet-1K
checkpoint** (82.6% top-1) is loaded when `MODEL_SIZE="tiny"`. It has exactly the v2v_tiny architecture. The download
comes from the VMamba authors' release page; a copy attached as a Kaggle input is used first if present.

In [ ]:
def weights_init_kaiming(m):
    cn = m.__class__.__name__
    if cn.find("Linear") != -1:
        nn.init.kaiming_normal_(m.weight, a=0, mode="fan_out"); nn.init.constant_(m.bias, 0.0)
    elif cn.find("Conv") != -1:
        nn.init.kaiming_normal_(m.weight, a=0, mode="fan_in")
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)
    elif cn.find("BatchNorm") != -1 and getattr(m, "affine", False):
        nn.init.constant_(m.weight, 1.0); nn.init.constant_(m.bias, 0.0)


def weights_init_classifier(m):
    if m.__class__.__name__.find("Linear") != -1:
        nn.init.normal_(m.weight, std=0.001)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)


def resolve_pretrain_path():
    if PRETRAIN != "imagenet":
        return ""
    if MODEL_SIZE not in IMAGENET_CKPTS:
        print(f"[pretrain] no official ImageNet checkpoint matches MODEL_SIZE={MODEL_SIZE} (v2v design) -> random init")
        return ""
    fname, url = IMAGENET_CKPTS[MODEL_SIZE]
    for d in ALL_INPUT_DIRS:
        if osp.isfile(osp.join(d, fname)):
            return osp.join(d, fname)
    try:
        return _download(url, osp.join(CACHE_DIR, fname))
    except Exception as e:  # noqa: BLE001
        print(f"!! [pretrain] download failed ({type(e).__name__}: {e}). Attach {fname} as a Kaggle input. -> random init")
        return ""


class VMambaReID(nn.Module):
    def __init__(self, num_classes, cfg, pretrain_path=""):
        super().__init__()
        self.neck, self.neck_feat, self.ID_LOSS_TYPE = cfg.MODEL.NECK, cfg.TEST.NECK_FEAT, cfg.MODEL.ID_LOSS_TYPE
        vcfg, ta = cfg.MODEL.VSSM, cfg.MODEL.TOKEN_ADAPT
        vmamba_mod.ENABLE_TOKEN_ADAPTATION = bool(ta.ENABLE)
        self.base = VSSM(
            patch_size=vcfg.PATCH_SIZE, in_chans=vcfg.IN_CHANS, num_classes=0, depths=vcfg.DEPTHS,
            dims=[vcfg.EMBED_DIM, vcfg.EMBED_DIM * 2, vcfg.EMBED_DIM * 4, vcfg.EMBED_DIM * 8],
            ssm_d_state=vcfg.SSM_D_STATE, ssm_dt_rank=vcfg.SSM_DT_RANK, ssm_ratio=vcfg.SSM_RATIO,
            ssm_conv=vcfg.SSM_CONV, ssm_conv_bias=vcfg.SSM_CONV_BIAS, forward_type=vcfg.SSM_FORWARDTYPE,
            mlp_ratio=vcfg.MLP_RATIO, drop_path_rate=cfg.MODEL.DROP_PATH_RATE, norm_layer=vcfg.NORM_LAYER,
            downsample_version=vcfg.DOWNSAMPLE, patchembed_version=vcfg.PATCHEMBED, imgsize=cfg.INPUT.SIZE_TRAIN[0],
            token_merge=bool(ta.ENABLE and ta.TOKEN_MERGE), token_prune=bool(ta.ENABLE and ta.TOKEN_PRUNE),
            merge_layers=list(ta.MERGE_LAYERS), prune_layers=list(ta.PRUNE_LAYERS), merge_start=int(ta.MERGE_START),
            prune_threshold=float(ta.PRUNE_THRESHOLD), prune_sim=str(ta.PRUNE_SIM), prune_imp=str(ta.PRUNE_IMP),
            final_threshold=float(ta.FINAL_THRESHOLD),
        )
        self.channel_first = self.base.channel_first
        self.in_planes, self.num_classes = self.base.num_features, num_classes
        self.bottleneck = nn.BatchNorm1d(self.in_planes)
        self.bottleneck.bias.requires_grad_(False)
        self.bottleneck.apply(weights_init_kaiming)
        self.pretrain_report = "random init"
        if cfg.MODEL.PRETRAIN_CHOICE in ("imagenet", "pretrained") and pretrain_path:
            self._load_backbone_pretrained(pretrain_path)
        self.classifier = nn.Linear(self.in_planes, num_classes, bias=False)
        self.classifier.apply(weights_init_classifier)

    def forward_features(self, x):
        x = self.base.patch_embed(x)
        if self.base.pos_embed is not None:
            pos = self.base.pos_embed
            x = x + (pos.permute(0, 2, 3, 1) if not self.base.channel_first else pos)
        for layer in self.base.layers:
            x = layer(x)
        return x

    def pool(self, feat_map):
        if not self.channel_first:
            feat_map = feat_map.permute(0, 3, 1, 2).contiguous()
        return F.adaptive_avg_pool2d(feat_map, 1).flatten(1)

    def forward(self, x, label=None, cam_label=None, view_label=None):
        global_feat = self.pool(self.forward_features(x))
        feat = self.bottleneck(global_feat) if self.neck == "bnneck" else global_feat
        if self.training:
            return self.classifier(feat), global_feat
        return feat if self.neck_feat == "after" else global_feat

    def _load_backbone_pretrained(self, path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict):
            ckpt = ckpt.get("state_dict", ckpt.get("model", ckpt))
        new = {(k[7:] if k.startswith("module.") else k): v for k, v in ckpt.items()}
        for k in list(new):
            if k.startswith("classifier.head.") or k.startswith("head."):
                new.pop(k)
        inc = self.base.load_state_dict(new, strict=False)
        real_missing = [k for k in inc.missing_keys if "prune_module" not in k and "merge_module" not in k
                        and not k.startswith(("head.", "classifier."))]
        self.pretrain_report = (f"loaded {osp.basename(path)}: {len(new) - len(inc.unexpected_keys)} tensors, "
                                f"missing backbone tensors={len(real_missing)}, unexpected={len(inc.unexpected_keys)}")
        print("[pretrain]", self.pretrain_report)
        if real_missing:
            print("   missing (first 10):", real_missing[:10])


def token_stats(model):
    """Average keep ratios reported by the prune / merge modules after the last forward pass."""
    pr, mr = [], []
    for m in model.modules():
        if m.__class__.__name__ == "LayerTokenPrune" and m.keep_ratio_est is not None:
            pr.append(m.keep_ratio_est)
        if m.__class__.__name__ == "FixedPairThresholdMerge" and m.keep_ratio_est is not None:
            mr.append(m.keep_ratio_est)
    return pr, mr

## Losses, optimiser, LR schedule (port of `loss/`, `solver/make_optimizer.py`, `solver/scheduler_factory.py`)
* Loss = ID cross-entropy (label smoothing **off** in the configs) + batch-hard triplet (margin 0.3) on the
  pre-BNNeck feature, each weighted 1.0.
* AdamW, lr 3e-4. Weight decay 0.05 on weights, 0 on biases, norms, and any parameter whose name contains "norm",
  "bn" or "ln" (the repo's name heuristic).
* Cosine schedule over all epochs, linear warm-up for 5 epochs from 1% of lr, floor at 0.2% of lr, stepped once per
  epoch (exact port of the repo's modified timm `CosineLRScheduler`).

In [ ]:
def euclidean_dist(x, y):
    m, n = x.size(0), y.size(0)
    xx = torch.pow(x, 2).sum(1, keepdim=True).expand(m, n)
    yy = torch.pow(y, 2).sum(1, keepdim=True).expand(n, m).t()
    return (xx + yy - 2 * torch.matmul(x, y.t())).clamp(min=1e-12).sqrt()


def hard_example_mining(dist_mat, labels):
    N = dist_mat.size(0)
    is_pos = labels.expand(N, N).eq(labels.expand(N, N).t())
    is_neg = labels.expand(N, N).ne(labels.expand(N, N).t())
    dist_ap, _ = torch.max(dist_mat[is_pos].contiguous().view(N, -1), 1, keepdim=True)
    dist_an, _ = torch.min(dist_mat[is_neg].contiguous().view(N, -1), 1, keepdim=True)
    return dist_ap.squeeze(1), dist_an.squeeze(1)


class TripletLoss:
    def __init__(self, margin=None):
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin) if margin is not None else nn.SoftMarginLoss()

    def __call__(self, global_feat, labels):
        dist_ap, dist_an = hard_example_mining(euclidean_dist(global_feat, global_feat), labels)
        y = torch.ones_like(dist_an)
        loss = self.ranking_loss(dist_an, dist_ap, y) if self.margin is not None else self.ranking_loss(dist_an - dist_ap, y)
        return loss, dist_ap, dist_an


class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes, self.epsilon, self.logsoftmax = num_classes, epsilon, nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        t = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        t = (1 - self.epsilon) * t + self.epsilon / self.num_classes
        return (-t * log_probs).mean(0).sum()


def make_loss(cfg, num_classes):
    triplet = TripletLoss() if cfg.MODEL.NO_MARGIN else TripletLoss(cfg.SOLVER.MARGIN)
    xent = CrossEntropyLabelSmooth(num_classes) if cfg.MODEL.IF_LABELSMOOTH == "on" else None

    def loss_func(score, feat, target, target_cam=None):
        id_loss = xent(score, target) if xent is not None else F.cross_entropy(score, target)
        tri_loss = triplet(feat, target)[0]
        return cfg.MODEL.ID_LOSS_WEIGHT * id_loss + cfg.MODEL.TRIPLET_LOSS_WEIGHT * tri_loss
    return loss_func


def make_optimizer(cfg, model):
    base_lr, wd = cfg.SOLVER.BASE_LR, cfg.SOLVER.WEIGHT_DECAY
    nwd_names, nwd_keywords = set(), set()
    for obj in (model, getattr(model, "base", None)):
        if obj is not None and hasattr(obj, "no_weight_decay"):
            nwd_names |= set(obj.no_weight_decay())
        if obj is not None and hasattr(obj, "no_weight_decay_keywords"):
            nwd_keywords |= set(obj.no_weight_decay_keywords())

    def no_wd(name):
        return (name in nwd_names or any(kw and kw in name for kw in nwd_keywords) or name.endswith(".bias")
                or any(x in name.lower() for x in ["norm", "bn", "ln", "layernorm", "batchnorm"]))

    decay, nodecay = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (nodecay if no_wd(name) else decay).append(p)
    groups = [{"params": decay, "lr": base_lr, "weight_decay": wd}, {"params": nodecay, "lr": base_lr, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=base_lr, weight_decay=0.0)


class RepoCosineLR:
    """Exact port of solver/cosine_lr.py + solver/scheduler_factory.py as configured by the repo
    (t_mul=1, decay_rate=0.1, warmup_prefix=False, cycle_limit=1, no noise). Note the repo quirk kept here: at the
    final epoch (t == t_initial) the cycle has ended and lr drops to lr_min * decay_rate (= 6e-8), whereas timm's
    built-in scheduler would keep lr_min."""
    def __init__(self, optimizer, t_initial, warmup_t, lr_min_ratio=0.002, warmup_ratio=0.01, decay_rate=0.1):
        self.optimizer, self.t_initial, self.warmup_t, self.decay_rate = optimizer, t_initial, int(warmup_t), decay_rate
        self.base_values = [g["lr"] for g in optimizer.param_groups]
        self.lr_min = [lr_min_ratio * b for b in self.base_values]
        self.warmup_lr_init = [warmup_ratio * b for b in self.base_values]
        self.warmup_steps = ([(b - w) / self.warmup_t for b, w in zip(self.base_values, self.warmup_lr_init)]
                             if self.warmup_t else [1.0] * len(self.base_values))
        if self.warmup_t:
            self._set(self.warmup_lr_init)

    def _set(self, lrs):
        for g, lr in zip(self.optimizer.param_groups, lrs):
            g["lr"] = lr

    def _get_lr(self, t):
        if t < self.warmup_t:
            return [w + t * s for w, s in zip(self.warmup_lr_init, self.warmup_steps)]
        i = t // self.t_initial
        t_curr = t - self.t_initial * i
        gamma = self.decay_rate ** i
        lr_max = [v * gamma for v in self.base_values]
        lr_min = [v * gamma for v in self.lr_min]
        if i < 1:   # cycle_limit = 1
            return [lo + 0.5 * (hi - lo) * (1 + math.cos(math.pi * t_curr / self.t_initial)) for hi, lo in zip(lr_max, lr_min)]
        return lr_min

    def step(self, epoch):
        self._set(self._get_lr(epoch))

    def state_dict(self):
        return {}

    def load_state_dict(self, sd):
        pass


def create_scheduler(cfg, optimizer):
    return RepoCosineLR(optimizer, t_initial=cfg.SOLVER.MAX_EPOCHS, warmup_t=cfg.SOLVER.WARMUP_EPOCHS)

## Evaluation (port of `utils/metrics.py`)
Features are L2-normalised, the distance is cosine distance (1 − q·g), and gallery images with the same ID **and**
the same camera as the query are ignored. CMC (Rank-1/5/10) and mAP follow the standard Market-1501 protocol.
Re-ranking is off, as in the repo (the evaluator is created without it). Features are extracted in fp32, as in the repo.

In [ ]:
def eval_func(distmat, q_pids, g_pids, q_camids, g_camids, max_rank=50):
    num_q, num_g = distmat.shape
    max_rank = min(max_rank, num_g)
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, np.newaxis]).astype(np.int32)
    all_cmc, all_AP = [], []
    for q_idx in range(num_q):
        order = indices[q_idx]
        keep = np.invert((g_pids[order] == q_pids[q_idx]) & (g_camids[order] == q_camids[q_idx]))
        orig_cmc = matches[q_idx][keep]
        if not np.any(orig_cmc):
            continue
        cmc = orig_cmc.cumsum(); cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        tmp = orig_cmc.cumsum() / (np.arange(1, orig_cmc.shape[0] + 1) * 1.0)
        all_AP.append((tmp * orig_cmc).sum() / orig_cmc.sum())
    assert all_cmc, "Error: all query identities do not appear in gallery"
    return np.asarray(all_cmc).astype(np.float32).mean(0), float(np.mean(all_AP))


@torch.no_grad()
def evaluate(model, val_loader, num_query, feat_norm=True):
    model.eval()
    feats, pids, camids = [], [], []
    for img, pid, camid, camids_b, _, _ in val_loader:
        f = model(img.to(DEVICE, non_blocking=True))          # fp32, like processor.py's evaluation
        feats.append(f.float().cpu()); pids.extend(int(x) for x in pid); camids.extend(int(x) for x in camid)
    feats = torch.cat(feats)
    if feat_norm:
        feats = F.normalize(feats, dim=1, p=2)
    qf, gf = feats[:num_query], feats[num_query:]
    distmat = (1.0 - qf @ gf.t()).numpy() if feat_norm else torch.cdist(qf, gf).numpy()
    p, c = np.asarray(pids), np.asarray(camids)
    cmc, mAP = eval_func(distmat, p[:num_query], p[num_query:], c[:num_query], c[num_query:])
    return {"mAP": mAP, "R1": float(cmc[0]), "R5": float(cmc[4]), "R10": float(cmc[9])}

## Efficiency measurement (what the paper's "over 60% FLOPs reduction" refers to)
With the trained model, the notebook measures:
* **tokens processed per block** (via forward hooks on every VSS block) with token adaptation on vs off;
* **inference throughput** (images/s, batch 64, fp16) with token adaptation on vs off;
* the average **keep ratios** of the prune modules and of the merge modules.

FLOPs are not counted directly (the repo's FLOP counter needs fvcore plus custom op handlers). The reduction in
tokens per block is a close proxy, because each block's cost grows linearly with its token count.

In [ ]:
@torch.no_grad()
def measure_efficiency(model, size, batch=64, iters=20):
    model.eval()
    x = torch.randn(batch, 3, size[0], size[1], device=DEVICE)
    tokens = []

    def hook(_m, inp, _out):
        t = inp[0]
        tokens.append(t.shape[-1] * t.shape[-2] if t.dim() == 4 else t.shape[1])
    blocks = [m for m in model.modules() if m.__class__.__name__ == "VSSBlock"]
    res = {}
    for flag in (True, False):
        vmamba_mod.ENABLE_TOKEN_ADAPTATION = flag
        handles = [b.register_forward_hook(hook) for b in blocks]
        tokens.clear()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            model(x)
        for h in handles:
            h.remove()
        per_block = list(tokens)
        pr, mr = token_stats(model) if flag else ([], [])
        n = iters if DEVICE == "cuda" else 2
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        res["on" if flag else "off"] = {"tokens_per_block": per_block, "total_tokens": int(sum(per_block)),
                                        "img_per_s": batch * n / (time.time() - t0),
                                        "prune_keep": pr, "merge_keep": mr}
    vmamba_mod.ENABLE_TOKEN_ADAPTATION = True
    res["token_reduction_%"] = 100.0 * (1 - res["on"]["total_tokens"] / max(res["off"]["total_tokens"], 1))
    res["speedup_x"] = res["on"]["img_per_s"] / max(res["off"]["img_per_s"], 1e-9)
    return res

## Training loop (port of `processor/processor.py::do_train` + `train.py`)
Per epoch: `scheduler.step(epoch)`, then for each PK batch run an AMP forward, compute ID + triplet loss, scale,
backward and step. Seed 1234, `cudnn.deterministic=True`, `benchmark=True`, as in `train.py`.

`train.py` runs **one forward pass before building the optimiser** (a debug check). That pass is what lazily
creates each merge module's threshold MLP. It is reproduced here, so those parameters exist when the optimiser and
checkpoints are built.

Additions for Kaggle (no effect on the maths):
* a resumable state (model, optimiser, AMP scaler, epoch, RNG states) saved every `CKPT_EVERY` epochs;
* a clean stop before `MAX_SESSION_HOURS`;
* optional mid-training evaluations (`EVAL_EVERY`).

In [ ]:
T_NOTEBOOK_START = globals().get("T_NOTEBOOK_START", time.time())


def set_seed(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


def setup_logger(out_dir):
    lg = logging.getLogger("tev"); lg.setLevel(logging.INFO); lg.handlers.clear(); lg.propagate = False
    fmt = logging.Formatter("%(asctime)s %(message)s", "%H:%M:%S")
    for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(osp.join(out_dir, "train_log.txt"), mode="a")):
        h.setFormatter(fmt); lg.addHandler(h)
    return lg


def _state_path(name):
    return osp.join(OUT_DIR, name, "last_state.pth")


def _find_resume(name):
    p = _state_path(name)
    if osp.isfile(p):
        return p
    for q in glob.glob(osp.join(INPUT_ROOT, "**", "TEVMamba_outputs", name, "last_state.pth"), recursive=True):
        os.makedirs(osp.dirname(p), exist_ok=True); shutil.copy(q, p)
        for f in ("train_log.txt", "eval_history.json"):
            if osp.isfile(osp.join(osp.dirname(q), f)):
                shutil.copy(osp.join(osp.dirname(q), f), osp.join(osp.dirname(p), f))
        print(f"[resume] copied {q}")
        return p
    return None


def train_and_eval(name):
    cfg = build_cfg(name)
    out_dir = osp.join(OUT_DIR, name); os.makedirs(out_dir, exist_ok=True)
    final_path = osp.join(out_dir, "final_results.json")
    if osp.isfile(final_path) and RESUME:
        print(f"[skip] {name}: already finished -> {final_path}")
        return json.load(open(final_path))
    lg = setup_logger(out_dir)
    set_seed(cfg.SOLVER.SEED)
    train_loader, val_loader, num_query, num_classes, dataset = make_dataloader(cfg)
    model = VMambaReID(num_classes, cfg, resolve_pretrain_path()).to(DEVICE)
    lg.info(f"[{name}] classes={num_classes} | epochs={cfg.SOLVER.MAX_EPOCHS} | input={cfg.INPUT.SIZE_TRAIN} | "
            f"backbone={MODEL_SIZE} | {model.pretrain_report}")

    # == train.py: one forward in eval mode before the optimiser (creates the lazy merge-gate MLPs)
    model.eval()
    with torch.no_grad():
        model(next(iter(train_loader))[0][:8].to(DEVICE))
    pr, mr = token_stats(model)
    lg.info(f"[attach] prune modules={len(pr)} keep~{np.mean(pr) if pr else float('nan'):.2f} | "
            f"merge modules={len(mr)} keep~{np.mean(mr) if mr else float('nan'):.2f}")
    model.train()

    loss_fn = make_loss(cfg, num_classes)
    optimizer = make_optimizer(cfg, model)
    scheduler = create_scheduler(cfg, optimizer)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    start_epoch, history = 1, []
    rp = _find_resume(name) if RESUME else None
    if rp:
        st = torch.load(rp, map_location="cpu", weights_only=False)
        # strict=False like the repo's test.py: VSSM._load_from_state_dict renames "head.*" -> "classifier.head.*"
        # (legacy-checkpoint compatibility), so the unused 0-output `base.head` cannot round-trip strictly.
        inc = model.load_state_dict(st["model"], strict=False)
        bad = [k for k in inc.missing_keys + inc.unexpected_keys if not k.startswith(("base.head.", "base.classifier.head."))]
        assert not bad, f"resume state does not match the model: {bad[:10]}"
        optimizer.load_state_dict(st["optimizer"]); scaler.load_state_dict(st["scaler"])
        start_epoch, history = st["epoch"] + 1, st.get("history", [])
        random.setstate(st["rng_py"]); np.random.set_state(st["rng_np"]); torch.set_rng_state(st["rng_torch"])
        if DEVICE == "cuda" and st.get("rng_cuda") is not None:
            torch.cuda.set_rng_state(st["rng_cuda"])
        lg.info(f"[resume] from epoch {st['epoch']} ({rp})")

    def save_state(epoch):
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch, "history": history, "rng_py": random.getstate(), "rng_np": np.random.get_state(),
                    "rng_torch": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state() if DEVICE == "cuda" else None}, _state_path(name) + ".tmp")
        os.replace(_state_path(name) + ".tmp", _state_path(name))

    epochs = cfg.SOLVER.MAX_EPOCHS
    epoch_times = []
    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        scheduler.step(epoch)
        model.train()
        loss_sum = acc_sum = n_seen = 0.0
        n_iter = 0
        for n_iter, (img, vid, target_cam, target_view) in enumerate(train_loader):
            if RUN_MODE == "quick" and n_iter >= QUICK["max_iters"]:
                break
            optimizer.zero_grad(set_to_none=True)
            img, target = img.to(DEVICE, non_blocking=True), vid.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                score, feat = model(img, target)
                loss = loss_fn(score, feat, target, target_cam)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            acc = (score.max(1)[1] == target).float().mean().item()
            loss_sum += loss.item() * img.shape[0]; acc_sum += acc; n_seen += img.shape[0]
            if (n_iter + 1) % cfg.SOLVER.LOG_PERIOD == 0:
                lg.info(f"Epoch[{epoch}] Iter[{n_iter + 1}/{len(train_loader)}] Loss {loss_sum / n_seen:.3f} "
                        f"Acc {acc_sum / (n_iter + 1):.3f} Lr {optimizer.param_groups[0]['lr']:.2e}")
        dt = time.time() - t0
        epoch_times.append(dt)
        lg.info(f"Epoch {epoch}/{epochs} done: loss {loss_sum / max(n_seen, 1):.3f} acc {acc_sum / max(n_iter + 1, 1):.3f} "
                f"lr {optimizer.param_groups[0]['lr']:.2e} | {dt:.0f}s ({n_seen / max(dt, 1e-9):.0f} img/s)")
        if epoch in (start_epoch, start_epoch + 5):
            # the first epoch includes cuDNN/Triton warm-up (~2x slower), so the estimate is refined 5 epochs later
            remaining = np.mean(epoch_times[-3:]) * (epochs - epoch) / 3600
            left = MAX_SESSION_HOURS - (time.time() - T_NOTEBOOK_START) / 3600
            lg.info(f"[estimate] ~{remaining:.1f} h of training left for {name} at this speed, {left:.1f} h left in this "
                    f"session ({'fits' if remaining < left else 'needs resume across sessions'})"
                    + (f" | first epoch includes warm-up; refined at epoch {start_epoch + 5}" if epoch == start_epoch else ""))
        if EVAL_EVERY and epoch % EVAL_EVERY == 0 and epoch != epochs:
            r = evaluate(model, val_loader, num_query); r["epoch"] = epoch; history.append(r)
            lg.info(f"[eval] epoch {epoch}: mAP {r['mAP']:.1%} R1 {r['R1']:.1%}")
        if epoch % CKPT_EVERY == 0 or epoch == epochs:
            save_state(epoch)
        elapsed_h = (time.time() - T_NOTEBOOK_START) / 3600
        if epoch != epochs and elapsed_h + 1.5 * np.mean(epoch_times) / 3600 + 0.25 > MAX_SESSION_HOURS:
            save_state(epoch)
            lg.info(f"[time] {elapsed_h:.1f} h used: stopping {name} at epoch {epoch}. Attach this output and re-run to resume.")
            return None

    r = evaluate(model, val_loader, num_query); r["epoch"] = epochs; history.append(r)
    lg.info(f"[final] {name}: mAP {r['mAP']:.1%} | R1 {r['R1']:.1%} | R5 {r['R5']:.1%} | R10 {r['R10']:.1%}")
    vmamba_mod.ENABLE_TOKEN_ADAPTATION = False
    r_noadapt = evaluate(model, val_loader, num_query)
    vmamba_mod.ENABLE_TOKEN_ADAPTATION = True
    lg.info(f"[diagnostic] same weights, token pruning/merging switched OFF at test: mAP {r_noadapt['mAP']:.1%} R1 {r_noadapt['R1']:.1%}")
    eff = measure_efficiency(model, cfg.INPUT.SIZE_TEST)
    lg.info(f"[efficiency] tokens/forward -{eff['token_reduction_%']:.1f}% | throughput on {eff['on']['img_per_s']:.0f} "
            f"vs off {eff['off']['img_per_s']:.0f} img/s (x{eff['speedup_x']:.2f})")
    torch.save(model.state_dict(), osp.join(out_dir, "final_model.pth"))
    result = {"dataset": name, "model_size": MODEL_SIZE, "pretrain": model.pretrain_report, "epochs": epochs,
              "run_mode": RUN_MODE, "final": r, "token_adapt_off_at_test": r_noadapt, "efficiency": eff,
              "history": history}
    json.dump(result, open(final_path, "w"), indent=1)
    if osp.isfile(_state_path(name)):
        os.remove(_state_path(name))            # finished: the resumable state is no longer needed
    return result

## ▶ Run
Each dataset is an independent training run, as in the paper. Finished datasets are skipped on re-runs
(`final_results.json`), and unfinished ones resume from `last_state.pth`.

In [ ]:
RESULTS = {}
for name in DATASETS_TO_RUN:
    if name not in PREPARED:
        print(f"[skip] {name}: data not available")
        continue
    r = train_and_eval(name)
    if r is None:
        print("[stop] session time budget reached; re-run with this notebook's output attached to continue.")
        break
    RESULTS[name] = r
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## Results
`PAPER_REPORTED` is left empty on purpose: fill in TE-VMamba's numbers from the paper's tables (the PDF could not
be read while preparing this notebook), using the **tiny** row if the paper reports one.

In [ ]:
import matplotlib.pyplot as plt

PAPER_REPORTED = {   # (mAP %, Rank-1 %) for TE-VMamba-Tiny from the paper; None = not filled in yet
    "market1501": None, "dukemtmc": None, "occ_duke": None, "occ_reid": None,
}
SHORT = {"market1501": "Market-1501", "dukemtmc": "DukeMTMC", "occ_duke": "Occluded-Duke", "occ_reid": "Occluded-REID"}

rows = []
for name in DATASETS_TO_RUN:
    p = osp.join(OUT_DIR, name, "final_results.json")
    if osp.isfile(p):
        rows.append(json.load(open(p)))
if not rows:
    print("no finished runs yet")
else:
    print(f"{'dataset':15s}{'mAP':>7s}{'R1':>7s}{'R5':>7s}{'R10':>7s} | {'paper mAP/R1':>13s} | "
          f"{'no-adapt mAP':>12s} | {'tokens':>7s} {'speed':>7s}")
    for r in rows:
        f, na, e = r["final"], r["token_adapt_off_at_test"], r["efficiency"]
        paper = PAPER_REPORTED.get(r["dataset"])
        ps = f"{paper[0]:.1f}/{paper[1]:.1f}" if paper else "-"
        print(f"{SHORT[r['dataset']]:15s}{100 * f['mAP']:7.1f}{100 * f['R1']:7.1f}{100 * f['R5']:7.1f}{100 * f['R10']:7.1f} | "
              f"{ps:>13s} | {100 * na['mAP']:12.1f} | {-e['token_reduction_%']:6.1f}% x{e['speedup_x']:.2f}")
    print(f"\nsettings: MODEL_SIZE={rows[0]['model_size']}, epochs={[r['epochs'] for r in rows]}, "
          f"init: {rows[0]['pretrain']}, RUN_MODE={rows[0]['run_mode']}")
    print("columns: 'no-adapt mAP' = same trained weights with pruning/merging switched off at test; "
          "'tokens' = change in tokens processed per forward; 'speed' = throughput ratio adapt-on / adapt-off")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    names = [SHORT[r["dataset"]] for r in rows]
    x = np.arange(len(rows)); w = 0.38
    axes[0].bar(x - w / 2 - 0.01, [100 * r["final"]["mAP"] for r in rows], w, color="#2a78d6", label="mAP")
    axes[0].bar(x + w / 2 + 0.01, [100 * r["final"]["R1"] for r in rows], w, color="#eb6834", label="Rank-1")
    for xi, r in zip(x, rows):
        axes[0].text(xi - w / 2, 100 * r["final"]["mAP"] + 1, f"{100 * r['final']['mAP']:.1f}", ha="center", fontsize=8, color="#52514e")
        axes[0].text(xi + w / 2, 100 * r["final"]["R1"] + 1, f"{100 * r['final']['R1']:.1f}", ha="center", fontsize=8, color="#52514e")
    axes[0].set_xticks(x, names); axes[0].set_ylim(0, 105); axes[0].set_ylabel("%")
    axes[0].set_title("TE-VMamba reproduction: accuracy", loc="left", fontsize=11)
    axes[0].legend(frameon=False); axes[0].grid(axis="y", color="#e6e5e0"); axes[0].set_axisbelow(True)
    e = rows[0]["efficiency"]
    axes[1].plot(e["off"]["tokens_per_block"], color="#52514e", lw=2, label="pruning/merging off")
    axes[1].plot(e["on"]["tokens_per_block"], color="#2a78d6", lw=2, label="pruning/merging on")
    axes[1].set_xlabel("VSS block index"); axes[1].set_ylabel("tokens entering block"); axes[1].set_yscale("log")
    axes[1].set_title(f"Tokens per block ({names[0]})", loc="left", fontsize=11)
    axes[1].legend(frameon=False); axes[1].grid(color="#e6e5e0")
    for ax in axes:
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    plt.tight_layout(); fig.savefig(osp.join(OUT_DIR, "results.png"), dpi=150); plt.show()

if RUN_MODE == "quick":
    print("\n(quick mode: 2 short epochs + subsampled evaluation, so these numbers are NOT results)")

## Outputs (`/kaggle/working/TEVMamba_outputs/`)
* `<dataset>/final_results.json`: final metrics, the no-adaptation diagnostic, the efficiency measurement and the
  evaluation history.
* `<dataset>/train_log.txt` and `<dataset>/final_model.pth`.
* `<dataset>/last_state.pth`: only while a run is unfinished; needed to resume.
* `kernel_cache/`: the compiled CUDA kernel, reused when this output is attached to a later session.
* `results.png`.